# TP3 · Pasos 1 y 2 — Depurar y auditar el catálogo

**NLP — UNR / FCEIA · Unidad 3**
Agustín Aiup, Facundo Ferreira da Camara, Franco Paoloni, Julián Silva.

- **Paso 1** — dejar el catálogo ampliado de la cátedra en condiciones de ser auditado.
- **Paso 2** — construir los detectores de contaminación. Por ahora están los
  apartados **2.1** (expresiones regulares), **2.2** (normalizar antes de buscar),
  **2.3** (idioma por oración), **2.4** (NER), **2.5** (clasificador de oraciones),
  **2.6** (libros copiados) y **2.7** (combinación y entrega). El paso 2 está completo.

---

## Paso 1 — Cargar y revisar los libros

Lo que pide el enunciado:

1. Cargar `catalogo_tp3.jsonl` y conservar `slug`, `title`, `authors` y `synopsis`.
2. Eliminar registros sin sinopsis y duplicados por identificador.
3. Normalizar los espacios — y **solo** los espacios.
4. Revisar los duplicados exactos de sinopsis antes de borrarlos, y anotar cuántos se eliminaron.
5. Buscar versiones casi iguales de una obra con una medida de semejanza de cadenas,
   comparando solo libros del mismo autor, con un criterio que **combine título y sinopsis**.
6. Mostrar cinco registros y cuántos libros quedaron y cuántos se eliminaron por cada motivo.

> **Qué no se hace acá.** No se limpian caracteres raros ni se tocan los enlaces: los
> caracteres invisibles, los homoglifos y las URLs disimuladas son la evidencia que el
> paso 2 necesita. Normalizar de más borraría la contaminación que hay que detectar.

## Preparación del entorno

In [1]:
# --- Dependencias ---
# rapidfuzz (paso 1): Levenshtein y Jaro-Winkler; es el motor que usa thefuzz.
# langdetect (2.3): deteccion de idioma.
# spacy (2.4): NER. sentence-transformers (2.5): el modelo de oraciones del TP2,
# la mas pesada de todas porque arrastra torch.
#
# Si las cuatro ya estan, el try corta y la celda no hace nada: asi reejecutar el
# notebook en la misma sesion de Colab no paga el viaje a pip.
try:
    import langdetect, rapidfuzz, sentence_transformers, spacy
except ImportError:
    %pip install -q rapidfuzz langdetect spacy sentence-transformers

C:\Users\Juli\AppData\Local\Temp\claude\C--Users-Juli-OneDrive-Escritorio-NLP-Aiup-Paoloni-Silva\5e86646a-7cef-4748-9015-b1f40e0d049b\scratchpad\libs\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Imports y configuracion ---
import json
import re
import sys
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from langdetect import detect_langs, DetectorFactory, LangDetectException
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GroupKFold, RepeatedStratifiedKFold,
                                     StratifiedKFold, StratifiedShuffleSplit,
                                     cross_val_predict, cross_val_score, train_test_split)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline

# Los dos imports pesados quedan en su apartado y no aca: spacy en el 2.4 y
# sentence_transformers en el 2.5. Asi el paso 1 corre sin cargar torch ni el
# modelo de 550 MB de spaCy, que juntos tardan mas que todo el paso 1.

# La consola de Windows es cp1252: un titulo con un caracter fuera de ese juego
# aborta la corrida entera si no se reemplaza al imprimir.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(errors="replace")

ARCHIVO_CATALOGO = "catalogo_tp3.jsonl"    # catalogo ampliado de la catedra
ARCHIVO_MUESTRA = "muestra_etiquetada.csv" # 500 libros con la etiqueta contaminado
ARCHIVO_SALIDA = "catalogo_paso1.csv"      # salida del paso 1
ARCHIVO_PARES = "paso1_pares.csv"          # evidencia de las decisiones, para el informe

UMBRAL_TITULO = 0.85      # Jaro-Winkler minimo para que dos titulos sean comparables
UMBRAL_SINOPSIS = 0.55    # solo desempata el caso ambiguo; se justifica en la seccion 1.7

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 170)

In [3]:
# --- Ubicacion de los datos ---
def ubicar_carpeta_datos(archivo):
    """Busca el catalogo en las rutas probables.

    El notebook vive en src/TP_3/ y los datos en data/TP_3/, pero en Colab suele
    quedar todo junto en el directorio de trabajo. Probamos las dos formas en vez de
    fijar una ruta que funcione en un entorno y falle en el otro.

    Todas las candidatas son relativas al directorio de trabajo, y devolvemos la
    ruta relativa sin resolver: asi lo que el notebook imprime y queda guardado en
    las salidas no lleva el nombre de usuario de quien lo corrio. En Colab el
    directorio de trabajo es /content, de modo que Path(".") ya lo cubre y la
    carpeta de Drive se alcanza como drive/MyDrive/...
    """
    candidatas = [
        Path("../../data/TP_3"),           # repo, notebook en src/TP_3/
        Path("data/TP_3"),                 # repo, notebook en la raiz
        Path("."),                         # Colab con los archivos subidos a mano
        Path("drive/MyDrive/data/TP_3"),   # Colab con Drive montado
    ]
    for carpeta in candidatas:
        if (carpeta / archivo).exists():
            return carpeta
    raise FileNotFoundError(
        f"No encontre {archivo}. Rutas probadas: "
        + ", ".join(str(c) for c in candidatas)
    )


CARPETA_DATOS = ubicar_carpeta_datos(ARCHIVO_CATALOGO)
print("datos en:", CARPETA_DATOS)

datos en: ..\..\data\TP_3


## 1.1 Carga del catálogo

In [4]:
# --- 1.1 Carga ---
crudo = pd.read_json(CARPETA_DATOS / ARCHIVO_CATALOGO, lines=True)
print("registros en el archivo:", len(crudo))
print("columnas:", list(crudo.columns))
crudo.head(3)

registros en el archivo: 8747
columnas: ['source_url', 'slug', 'title', 'authors', 'genres', 'seed_genre', 'series', 'series_number', 'cover_url', 'synopsis', 'synopsis_chars', 'scraped_at']


,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
0,https://ww3.lectulandia.com/book/de-la-caverna-al-cosmos/,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20Carbonell/De%20la%20caverna%20al%20cosmos...,La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos-y-como-hemos-llegado-hasta-aqui/,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Reich/Quienes%20somos%20y%20como%20hemos%2...,La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importa...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo, Espiritualidad]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20Matos%20Moctezuma/Teotihuacan%20(2)/big.jpg,Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de ...,1070,2026-10-17 23:12:01+00:00


In [5]:
# Conservamos los campos que pide el enunciado. 'series' y 'series_number' se suman
# porque ayudan a interpretar los casi-duplicados de la seccion 1.6.
COLUMNAS = ["slug", "title", "authors", "genres", "series", "series_number", "synopsis"]
libros = crudo[COLUMNAS].copy()

bajas = {}  # motivo -> cantidad eliminada, para el resumen final del paso

## 1.2 Registros sin sinopsis y duplicados por identificador

La sinopsis es la única entrada de los modelos de los pasos 5 a 8, así que un libro sin
sinopsis no sirve para nada. El `slug` es el identificador del libro y el que la cátedra
va a usar para corregir `sospechosos.csv`, de modo que no puede repetirse.

In [6]:
# --- 1.2 Sin sinopsis y slug repetido ---
sin_sinopsis = (
    libros["synopsis"].isna() | (libros["synopsis"].astype(str).str.strip() == "")
)
bajas["sin sinopsis"] = int(sin_sinopsis.sum())
libros = libros[~sin_sinopsis].copy()

antes = len(libros)
libros = libros.drop_duplicates(subset="slug", keep="first").copy()
bajas["slug repetido"] = antes - len(libros)

print(f"sin sinopsis:  {bajas['sin sinopsis']}")
print(f"slug repetido: {bajas['slug repetido']}")
print(f"quedan:        {len(libros)}")

sin sinopsis:  1
slug repetido: 0
quedan:        8746


## 1.3 Normalizar los espacios

Al descargar las páginas, las palabras en cursiva quedaron partidas por saltos de línea en
medio de una frase. Colapsamos cada secuencia de blancos a un solo espacio.

**Y nada más.** Los caracteres invisibles, los homoglifos y los signos reemplazados por
palabras son justamente lo que el paso 2 tiene que detectar: limpiarlos acá sería borrar la
contaminación antes de medirla.

In [7]:
# --- 1.3 Normalizar espacios ---
libros["sinopsis"] = (
    libros["synopsis"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
)

cambiaron = libros["sinopsis"] != libros["synopsis"].astype(str)
print(f"sinopsis que cambiaron al colapsar blancos: {int(cambiaron.sum())} de {len(libros)}")

# Un caso concreto, para ver el salto de linea partiendo una frase.
con_salto = libros[libros["synopsis"].astype(str).str.contains("\n", regex=False)]
ejemplo = con_salto.iloc[0]
print("\nANTES:  ", repr(ejemplo["synopsis"][:150]))
print("DESPUES:", repr(ejemplo["sinopsis"][:150]))

sinopsis que cambiaron al colapsar blancos: 5255 de 8746

ANTES:   'La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos.\nEudald Carbonell nos ayuda a visu'
DESPUES: 'La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos. Eudald Carbonell nos ayuda a visu'


## 1.4 Herramientas para comparar títulos

Antes de buscar duplicados hace falta decidir **cuándo dos títulos nombran la misma obra**.
El enunciado avisa que la semejanza de cadenas mide la forma y no el significado:
*Conversaciones con Dios I* y *II* se parecen muchísimo y son libros distintos.

Separamos cada título en dos cosas:

- **Discriminantes**: los números y numerales romanos. Son lo que distingue un volumen de
  otro (*Epigramas I* vs *Epigramas II*).
- **Nombre de la obra**: el título sin paréntesis, sin numerales y sin marcas de edición o
  traducción. Así *Romeo y Julieta (trad. Neruda)* y *Romeo y Julieta* colapsan al mismo
  nombre, porque son la misma obra en dos ediciones.

In [8]:
# --- 1.4 Herramientas de comparacion de titulos ---
ROMANOS = {
    "i": 1, "ii": 2, "iii": 3, "iv": 4, "v": 5, "vi": 6, "vii": 7, "viii": 8,
    "ix": 9, "x": 10, "xi": 11, "xii": 12, "xiii": 13, "xiv": 14, "xv": 15,
}

# Palabras que anuncian un volumen: no aportan al nombre de la obra, pero el numero
# que las acompania si distingue una entrega de otra.
PALABRAS_VOLUMEN = {"vol", "volumen", "tomo", "parte", "numero", "num", "libro", "serie"}

# Marcas de edicion o traduccion: no distinguen la obra, solo la edicion.
PALABRAS_EDICION = {
    "ed", "edicion", "edition", "trad", "traduccion", "tr", "ilustrado", "ilustrada",
    "bilingue", "anotada", "anotado", "reedicion", "reeditado", "color", "version",
    "completa", "completo", "definitiva", "ampliada", "revisada", "comentada",
    "integral", "clasicos", "medida", "original",
}

PARENTESIS = re.compile(r"[\(\[\{][^\)\]\}]*[\)\]\}]")


def sin_acentos(texto):
    """Minusculas sin diacriticos, para que 'Sabiduria' y 'Sabiduría' coincidan."""
    descompuesto = unicodedata.normalize("NFKD", str(texto).lower())
    return "".join(c for c in descompuesto if not unicodedata.combining(c))


def normalizar_titulo(titulo):
    """Titulo comparable: sin acentos, sin puntuacion y con los espacios colapsados."""
    limpio = re.sub(r"[^a-z0-9 ]", " ", sin_acentos(titulo))
    return re.sub(r"\s+", " ", limpio).strip()


def discriminantes(titulo):
    """Numeros y numerales romanos del titulo.

    Es el campo que distingue un volumen de otro, y lo miramos antes que la sinopsis
    porque en este catalogo muchas series comparten una sola sinopsis: si la sinopsis
    decidiera, fusionariamos volumenes legitimos de una misma obra.
    """
    valores = set()
    for token in normalizar_titulo(titulo).split():
        if token.isdigit():
            valores.add(int(token))
        elif token in ROMANOS:
            valores.add(ROMANOS[token])
    return valores


def nombre_obra(titulo):
    """Titulo sin parentesis, numerales ni marcas de edicion: la obra desnuda."""
    texto = PARENTESIS.sub(" ", sin_acentos(titulo))
    tokens = [
        t for t in normalizar_titulo(texto).split()
        if not t.isdigit() and t not in ROMANOS
        and t not in PALABRAS_VOLUMEN and t not in PALABRAS_EDICION
    ]
    return " ".join(tokens)


def relacion_titulos(titulo_a, titulo_b):
    """Como se relacionan dos titulos: 'igual', 'subtitulo' o 'distinta'."""
    a, b = nombre_obra(titulo_a), nombre_obra(titulo_b)
    if not a or not b:
        return "distinta"
    if a == b:
        return "igual"
    corto, largo = (a, b) if len(a) <= len(b) else (b, a)
    if largo.startswith(corto + " "):
        return "subtitulo"
    return "distinta"

In [9]:
# Chequeo de las funciones sobre casos reales del catalogo.
ejemplos = [
    ("Romeo y Julieta", "Romeo y Julieta (trad. Neruda)"),
    ("Conversaciones Con Dios I", "Conversaciones Con Dios II"),
    ("El ladron mago", "El ladron mago: Perdido"),
    ("Tratados hipocraticos VII", "Tratados hipocraticos VIII"),
    ("Freakonomics (edicion ampliada)", "Freakonomics"),
]
for a, b in ejemplos:
    print(f"{a!r}\n  vs {b!r}")
    print(f"  obra:           {nombre_obra(a)!r} | {nombre_obra(b)!r}")
    print(f"  discriminantes: {discriminantes(a) or set()} | {discriminantes(b) or set()}")
    print(f"  relacion:       {relacion_titulos(a, b)}\n")

'Romeo y Julieta'
  vs 'Romeo y Julieta (trad. Neruda)'
  obra:           'romeo y julieta' | 'romeo y julieta'
  discriminantes: set() | set()
  relacion:       igual

'Conversaciones Con Dios I'
  vs 'Conversaciones Con Dios II'
  obra:           'conversaciones con dios' | 'conversaciones con dios'
  discriminantes: {1} | {2}
  relacion:       igual

'El ladron mago'
  vs 'El ladron mago: Perdido'
  obra:           'el ladron mago' | 'el ladron mago perdido'
  discriminantes: set() | set()
  relacion:       subtitulo

'Tratados hipocraticos VII'
  vs 'Tratados hipocraticos VIII'
  obra:           'tratados hipocraticos' | 'tratados hipocraticos'
  discriminantes: {7} | {8}
  relacion:       igual

'Freakonomics (edicion ampliada)'
  vs 'Freakonomics'
  obra:           'freakonomics' | 'freakonomics'
  discriminantes: set() | set()
  relacion:       igual



## 1.5 Duplicados exactos de sinopsis

El enunciado pide revisarlos antes de borrarlos, porque hay cursos y series completas que
repiten la misma sinopsis en todos sus volúmenes. Vale la pena mirar **qué** son esos grupos
antes de tocarlos.

In [10]:
# --- 1.5 Duplicados exactos de sinopsis ---
repetidas = libros[libros["sinopsis"].duplicated(keep=False)]
grupos = list(repetidas.groupby("sinopsis"))
print(f"registros con sinopsis repetida: {len(repetidas)} en {len(grupos)} grupos")

# Cuantos de esos grupos son volumenes de una misma obra: mismo nombre de obra y
# titulos distintos, o sea que lo unico que los diferencia es el numeral.
volumenes = 0
for _, grupo in grupos:
    obras = {nombre_obra(t) for t in grupo["title"]}
    titulos = {normalizar_titulo(t) for t in grupo["title"]}
    if len(obras) == 1 and len(titulos) > 1:
        volumenes += 1

print(f"  de esos grupos, {volumenes} son volumenes distintos de una misma obra")
print("  (el sitio les repite la sinopsis; no es un error de extraccion)")

print("\nprimeros 6 grupos:")
for _, grupo in grupos[:6]:
    autores = {a[0] if isinstance(a, list) and a else "?" for a in grupo["authors"]}
    print(f"\n  n={len(grupo)} | {autores}")
    for t in grupo["title"]:
        print(f"    - {t}")

registros con sinopsis repetida: 131 en 33 grupos
  de esos grupos, 19 son volumenes distintos de una misma obra
  (el sitio les repite la sinopsis; no es un error de extraccion)

primeros 6 grupos:

  n=2 | {'César Martín'}
    - Prácticas sexuales alternativas (I)
    - Prácticas sexuales alternativas (II)

  n=2 | {'Michael Ende'}
    - La Historia Interminable – Color
    - La Historia Interminable

  n=2 | {'Patrick R. Reid'}
    - La historia de Colditz
    - Últimos días en Colditz

  n=2 | {'Flavia de la Fuente'}
    - Los años irreverentes. Escritos completos en El Amante/Cine Vol.2
    - Los años irreverentes. Escritos completos en El Amante/Cine Vol.1

  n=2 | {'Edgar Landívar'}
    - Comunicaciones unificadas con Elastix
    - Comunicaciones unificadas con Elastix (Volumen 2)

  n=58 | {'Varios Autores'}
    - Alemán – Unidad 40
    - Alemán – Unidad 15
    - Alemán – Unidad 13
    - Alemán – Unidad 14
    - Alemán – Unidad 34
    - Alemán – Unidad 33
    - Alemán – Unidad 

Varios de esos grupos son **volúmenes distintos** de una misma obra, no registros repetidos.
Conservar uno por sinopsis, como pide el enunciado, elimina volúmenes legítimos; lo dejamos
anotado porque es una limitación del criterio y no un acierto.

In [11]:
# El enunciado pide conservar un libro por sinopsis para esta practica.
antes = len(libros)
libros = libros.drop_duplicates(subset="sinopsis", keep="first").copy()
bajas["sinopsis repetida"] = antes - len(libros)
print(f"bajas por sinopsis repetida: {bajas['sinopsis repetida']} | quedan: {len(libros)}")

bajas por sinopsis repetida: 98 | quedan: 8648


## 1.6 Versiones casi iguales de una obra

### Bloqueo por autor

Comparar todos los títulos contra todos sobre ~8600 libros son unos 37 millones de pares.
El enunciado pide comparar solo libros del mismo autor, lo que baja el número en tres
órdenes de magnitud.

In [12]:
# --- 1.6 Candidatos: bloqueo por autor ---
libros["titulo_norm"] = libros["title"].map(normalizar_titulo)
libros["autor_principal"] = [
    a[0] if isinstance(a, list) and a else "" for a in libros["authors"]
]

tamanos = libros["autor_principal"].value_counts()
pares_con_bloqueo = int((tamanos * (tamanos - 1) // 2).sum())
pares_sin_bloqueo = len(libros) * (len(libros) - 1) // 2

print(f"autores distintos:              {len(tamanos)}")
print(f"autores con mas de un libro:    {int((tamanos > 1).sum())}")
print(f"pares con bloqueo por autor:    {pares_con_bloqueo:,}")
print(f"pares sin bloqueo:              {pares_sin_bloqueo:,}")
print(f"reduccion:                      {pares_sin_bloqueo / pares_con_bloqueo:.0f}x")

autores distintos:              5234
autores con mas de un libro:    1364
pares con bloqueo por autor:    18,109
pares sin bloqueo:              37,389,628
reduccion:                      2065x


### El criterio

El orden de las reglas es lo importante. **El título decide primero**, porque es el campo
confiable; la sinopsis entra solo donde el título es ambiguo.

1. **Discriminantes distintos** → volúmenes o entregas distintas. **Separar.**
2. **Mismo nombre de obra** (difieren solo en marcas de edición o traducción) → **unir.**
3. **Uno es el otro más un subtítulo** → ambiguo, lo desempata la sinopsis.
4. **Cualquier otro caso** → obras distintas con título parecido. **Separar.**

La sinopsis **no** se usa para confirmar los casos 1 y 2, y esto es lo que más cuesta ver.
Hay dos razones, las dos medidas sobre este catálogo:

- Muchas series comparten **una sola sinopsis**: *Vidas mágicas e Inquisición I* y *II*
  tienen sinopsis idéntica y son volúmenes distintos. Un criterio que uniera por sinopsis
  parecida los fusionaría.
- Dos ediciones de una misma obra pueden tener sinopsis **sin nada en común**, porque cada
  editorial escribe su propia contratapa. *Romeo y Julieta* y *Romeo y Julieta (trad.
  Neruda)* son la misma obra con similitud de sinopsis cercana a 0.43. Un criterio que
  exigiera sinopsis parecida para unir las separaría.

Es el mismo fenómeno que encontramos en el TP1 con *El Pantano del Diablo* y *La charca del
diablo*: la misma novela de George Sand en dos traducciones, con vocabulario distinto.

In [13]:
# --- 1.6 Clasificacion de cada par ---
def similitud_sinopsis(a, b):
    """fuzz.ratio sobre las sinopsis completas, llevado al rango [0, 1].

    Usamos ratio y no token_set_ratio: el segundo ignora el orden y las repeticiones,
    y da 0.99 para libros distintos que comparten una sinopsis de serie.
    """
    return fuzz.ratio(a, b) / 100


def clasificar_par(a, b):
    """Decide si dos registros son la misma obra. El titulo manda sobre la sinopsis."""
    if discriminantes(a["title"]) != discriminantes(b["title"]):
        return "separar: volumen o entrega distinta"

    relacion = relacion_titulos(a["title"], b["title"])
    if relacion == "igual":
        return "unir: variante de edicion"
    if relacion == "distinta":
        return "separar: obra distinta"

    # Caso ambiguo: un titulo es el otro mas un subtitulo. Puede ser la misma obra con
    # el subtitulo explicito o una entrega distinta de una serie. Decide la sinopsis.
    if similitud_sinopsis(a["sinopsis"], b["sinopsis"]) >= UMBRAL_SINOPSIS:
        return "unir: subtitulo con sinopsis coincidente"
    return "separar: subtitulo con sinopsis distinta"


registros = []
for autor, grupo in libros.groupby("autor_principal"):
    if not autor or len(grupo) < 2:
        continue
    filas = grupo.to_dict("records")
    for i in range(len(filas)):
        for j in range(i + 1, len(filas)):
            a, b = filas[i], filas[j]
            jw = JaroWinkler.similarity(a["titulo_norm"], b["titulo_norm"])
            if jw < UMBRAL_TITULO:
                continue
            registros.append({
                "autor": autor,
                "titulo_a": a["title"],
                "titulo_b": b["title"],
                "slug_a": a["slug"],
                "slug_b": b["slug"],
                "jw_titulo": round(jw, 3),
                "sim_sinopsis": round(similitud_sinopsis(a["sinopsis"], b["sinopsis"]), 2),
                "decision": clasificar_par(a, b),
            })

pares = pd.DataFrame(registros)
print(f"pares candidatos (mismo autor, Jaro-Winkler >= {UMBRAL_TITULO}): {len(pares)}\n")
print(pares["decision"].value_counts().to_string())

pares candidatos (mismo autor, Jaro-Winkler >= 0.85): 550

decision
separar: obra distinta                      248
separar: volumen o entrega distinta         223
unir: variante de edicion                    58
separar: subtitulo con sinopsis distinta     15
unir: subtitulo con sinopsis coincidente      6


### Dos pares que unimos y dos que separamos

Los elegimos para que se vea por qué el título manda sobre la sinopsis: en el primero y en
el tercero, mirar la sinopsis daría la decisión equivocada.

In [14]:
# --- 1.6 Los casos que pide el enunciado ---
ELEGIDOS = [
    ("UNIR", "unir: variante de edicion", True,
     "Misma obra en dos ediciones: la sinopsis NO se parece y aun asi hay que unir."),
    ("UNIR", "unir: subtitulo con sinopsis coincidente", False,
     "Titulo ambiguo (subtitulo); aca la sinopsis confirma que es la misma obra."),
    ("SEPARAR", "separar: volumen o entrega distinta", False,
     "Sinopsis identica y son volumenes distintos: el numeral manda sobre la sinopsis."),
    ("SEPARAR", "separar: subtitulo con sinopsis distinta", True,
     "Un titulo contiene al otro y la sinopsis no coincide: son libros distintos,"
     " no dos ediciones de uno solo."),
]

for etiqueta, decision, ascendente, porque in ELEGIDOS:
    seleccion = pares[pares["decision"] == decision].sort_values(
        "sim_sinopsis", ascending=ascendente
    )
    if seleccion.empty:
        print(f"### {etiqueta} - {decision}: sin casos\n")
        continue
    f = seleccion.iloc[0]
    print(f"### {etiqueta} - {decision}")
    print(f"  A: {f['titulo_a']}")
    print(f"  B: {f['titulo_b']}")
    print(f"  autor: {f['autor']}")
    print(f"  Jaro-Winkler del titulo: {f['jw_titulo']} | similitud de sinopsis: {f['sim_sinopsis']}")
    print(f"  por que: {porque}\n")

### UNIR - unir: variante de edicion
  A: El juguete rabioso (edición crítica)
  B: El juguete rabioso
  autor: Roberto Arlt
  Jaro-Winkler del titulo: 0.906 | similitud de sinopsis: 0.23
  por que: Misma obra en dos ediciones: la sinopsis NO se parece y aun asi hay que unir.

### UNIR - unir: subtitulo con sinopsis coincidente
  A: Introducción a la ciencia. Una guía para todos (o casi)
  B: Introducción a la ciencia
  autor: John Gribbin
  Jaro-Winkler del titulo: 0.896 | similitud de sinopsis: 1.0
  por que: Titulo ambiguo (subtitulo); aca la sinopsis confirma que es la misma obra.

### SEPARAR - separar: volumen o entrega distinta
  A: Cartas eruditas, y curiosas II
  B: Cartas eruditas, y curiosas V
  autor: Benito Jerónimo Feijoo
  Jaro-Winkler del titulo: 0.979 | similitud de sinopsis: 1.0
  por que: Sinopsis identica y son volumenes distintos: el numeral manda sobre la sinopsis.

### SEPARAR - separar: subtitulo con sinopsis distinta
  A: La fuga de Atalanta: alquimia y emblemá

## 1.7 Por qué `UMBRAL_SINOPSIS = 0.55`

El umbral interviene en un solo caso: cuando un título es el otro más un subtítulo. Si entre
los pares que une y los que separa hay una **brecha** de valores vacía, el umbral no está al
filo y cualquier número de ese intervalo da la misma partición. Eso es lo que hay que mostrar
para justificarlo, en lugar de afirmar que 0.55 "funciona bien".

In [15]:
# --- 1.7 Justificacion del umbral ---
ambiguos = pares[pares["decision"].str.contains("subtitulo")]
unidos = ambiguos[ambiguos["decision"].str.startswith("unir")]["sim_sinopsis"]
separados = ambiguos[ambiguos["decision"].str.startswith("separar")]["sim_sinopsis"]

print(f"pares ambiguos (subtitulo): {len(ambiguos)}")
print(f"  unidos    (n={len(unidos)}): {sorted(unidos.tolist())}")
print(f"  separados (n={len(separados)}): {sorted(separados.tolist())}")

if len(unidos) and len(separados):
    techo, piso = separados.max(), unidos.min()
    print(f"\nmayor similitud entre los separados: {techo:.2f}")
    print(f"menor similitud entre los unidos:    {piso:.2f}")
    if techo < piso:
        print(f"brecha vacia entre {techo:.2f} y {piso:.2f}:")
        print(f"  cualquier umbral en ese intervalo da la misma particion.")
        print(f"  UMBRAL_SINOPSIS = {UMBRAL_SINOPSIS} cae dentro.")
    else:
        print("no hay brecha: el umbral separa casos vecinos y conviene revisarlo a mano.")

pares ambiguos (subtitulo): 21
  unidos    (n=6): [0.61, 0.78, 0.94, 0.97, 1.0, 1.0]
  separados (n=15): [0.28, 0.34, 0.36, 0.36, 0.38, 0.41, 0.42, 0.43, 0.44, 0.44, 0.44, 0.44, 0.45, 0.45, 0.45]

mayor similitud entre los separados: 0.45
menor similitud entre los unidos:    0.61
brecha vacia entre 0.45 y 0.61:
  cualquier umbral en ese intervalo da la misma particion.
  UMBRAL_SINOPSIS = 0.55 cae dentro.


## 1.8 Consolidar los grupos y conservar una versión de cada obra

Las uniones se encadenan: si A y B son la misma obra y B y C también, los tres van al mismo
grupo. Se resuelve con *union-find* antes de contar las bajas, porque contar pares en lugar
de grupos daría un número inflado.

De cada grupo conservamos **la sinopsis más larga**. En los pasos 5 a 8 la sinopsis es la
única entrada del modelo, así que la versión más informativa es la que conviene quedarse.

In [16]:
# --- 1.8 Consolidar los casi-duplicados ---
padre = {}


def raiz(x):
    """Representante del grupo de x, con compresion de camino."""
    padre.setdefault(x, x)
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x


def unir_grupos(x, y):
    rx, ry = raiz(x), raiz(y)
    if rx != ry:
        padre[ry] = rx


for _, f in pares[pares["decision"].str.startswith("unir")].iterrows():
    unir_grupos(f["slug_a"], f["slug_b"])

libros["grupo_obra"] = [raiz(s) if s in padre else s for s in libros["slug"]]

tamano = libros["grupo_obra"].value_counts()
multiples = tamano[tamano > 1]
print(f"grupos de casi-duplicados: {len(multiples)}")
print(f"tamanos de grupo: {sorted(Counter(multiples.tolist()).items())}")

antes = len(libros)
libros["largo_sinopsis"] = libros["sinopsis"].str.len()
libros = (
    libros.sort_values("largo_sinopsis", ascending=False)
          .drop_duplicates(subset="grupo_obra", keep="first")
          .sort_index()
          .copy()
)
bajas["casi-duplicado"] = antes - len(libros)
print(f"\nbajas por casi-duplicado: {bajas['casi-duplicado']} | quedan: {len(libros)}")

grupos de casi-duplicados: 50
tamanos de grupo: [(2, 43), (3, 7)]



bajas por casi-duplicado: 57 | quedan: 8591

## 1.9 Resumen del paso 1

In [17]:
# --- 1.9 Resumen de bajas ---
resumen = pd.DataFrame(
    [{"motivo": motivo, "registros": cantidad} for motivo, cantidad in bajas.items()]
)
resumen.loc[len(resumen)] = {"motivo": "TOTAL eliminados", "registros": sum(bajas.values())}

print(f"registros en el archivo original: {len(crudo)}\n")
print(resumen.to_string(index=False))
print(f"\nlibros disponibles para el paso 2: {len(libros)}")

# El conteo tiene que cerrar: si no cierra, alguna baja se conto dos veces.
assert len(crudo) - sum(bajas.values()) == len(libros), "el conteo de bajas no cierra"
print("verificado: original - eliminados = disponibles")

registros en el archivo original: 8747

           motivo  registros
     sin sinopsis          1
    slug repetido          0
sinopsis repetida         98
   casi-duplicado         57
 TOTAL eliminados        156

libros disponibles para el paso 2: 8591
verificado: original - eliminados = disponibles


In [18]:
# Cinco registros del catalogo depurado.
muestra = libros[["slug", "title", "autor_principal", "sinopsis"]].head(5).copy()
muestra["sinopsis"] = muestra["sinopsis"].str.slice(0, 100) + "..."
muestra

,slug,title,autor_principal,sinopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,Eudald Carbonell,La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,David Reich,La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importa...
2,teotihuacan,Teotihuacan,Eduardo Matos Moctezuma,Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de ...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,M.ª Eugenia Aubet,Entre los siglos VIII y VI a. C. los fenicios establecieron el primer sistema comercia...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,Emiliano Bruner,Las raíces evolutivas del sufrimiento humano Una nueva comprensión de la salud mental ...


## Guardado para el paso 2

`authors` y `genres` se guardan como JSON, igual que en el TP1, para poder recuperarlos como
listas de Python y no como el `repr` de una lista.

También guardamos la tabla de pares: es la evidencia de las decisiones del paso 1 y de ahí
salen los ejemplos para el informe.

In [19]:
# --- Guardado ---
salida = libros[
    ["slug", "title", "authors", "genres", "series", "series_number", "sinopsis"]
].copy()
salida["authors"] = salida["authors"].map(lambda v: json.dumps(v, ensure_ascii=False))
salida["genres"] = salida["genres"].map(lambda v: json.dumps(v, ensure_ascii=False))

ruta_salida = CARPETA_DATOS / ARCHIVO_SALIDA
salida.to_csv(ruta_salida, index=False, encoding="utf-8")
print(f"guardado: {ruta_salida}  ({len(salida)} libros)")

ruta_pares = CARPETA_DATOS / ARCHIVO_PARES
pares.to_csv(ruta_pares, index=False, encoding="utf-8")
print(f"guardado: {ruta_pares}  ({len(pares)} pares evaluados)")

guardado: ..\..\data\TP_3\catalogo_paso1.csv  (8591 libros)
guardado: ..\..\data\TP_3\paso1_pares.csv  (550 pares evaluados)


## Cierre del paso 1

- El catálogo depurado está en `catalogo_paso1.csv`, con las sinopsis normalizadas **solo**
  en los espacios: la evidencia de contaminación sigue intacta.
- `paso1_pares.csv` guarda las 550 comparaciones con su decisión y los dos puntajes.

**Limitación a declarar en el informe**: conservar un libro por sinopsis, como pide el
enunciado, elimina volúmenes legítimos de obras que el sitio describe con un solo texto.
Quedó medido en la sección 1.5.

---

# Paso 2 — Auditar el catálogo

El catálogo tiene registros contaminados de varios tipos: algunos con enlaces evidentes,
otros que los disimulan, otros que solo agregan texto promocional y otros que son libros
inventados. **Ningún método detecta todos**, así que se construyen varios detectores, se
mide cada uno y se combinan.

Esta sección cubre **2.1** y **2.2**. Faltan 2.3 (idioma), 2.4 (NER), 2.5 (clasificador de
oraciones), 2.6 (libros copiados) y 2.7 (combinación y entrega).

## 2.0 La muestra etiquetada y su división

La división es **por libro, no por oración**. Si dos oraciones del mismo libro quedan una en
desarrollo y otra en prueba, el detector ya vio parte de ese libro y la medición sale
optimista.

Los detectores se diseñan y calibran mirando **solo desarrollo**. La parte de prueba se usa
una vez, al final, para informar el número.

In [20]:
# --- 2.0 Carga y division de la muestra ---
# keep_default_na=False: sin esto, un fragmento vacio se lee como NaN y despues
# rompe las operaciones de texto.
muestra = pd.read_csv(CARPETA_DATOS / ARCHIVO_MUESTRA, keep_default_na=False)
print(f"muestra: {len(muestra)} libros")
print(muestra["contaminado"].value_counts().rename("libros").to_string())

desarrollo, prueba = train_test_split(
    muestra, test_size=0.3, random_state=42, stratify=muestra["contaminado"]
)
print(f"\ndesarrollo: {len(desarrollo)} | prueba: {len(prueba)}")
print(f"contaminados -> desarrollo: {int(desarrollo['contaminado'].sum())}"
      f" | prueba: {int(prueba['contaminado'].sum())}")

muestra: 500 libros
contaminado
0    400
1    100



desarrollo: 350 | prueba: 150
contaminados -> desarrollo: 70 | prueba: 30


Dos advertencias del enunciado que cambian cómo se leen los números de acá en adelante:

- La muestra está **enriquecida al 20 %** de contaminados, bastante más que el catálogo
  completo. Con la misma tasa de aciertos y de falsos positivos, si la contaminación es más
  rara la **precisión baja**: lo que midamos acá es optimista respecto de lo que devuelva la
  cátedra.
- La muestra **no incluye todas las variantes**. En el catálogo hay mensajes que nuestros
  detectores nunca vieron.

In [21]:
# --- 2.0 Funcion de medicion ---
resultados = []  # cada detector deja su fila aca, para la tabla del 2.7


def medir(nombre, marcado, datos, registrar=True, mostrar_fp=0):
    """Precision y recall de la clase contaminado.

    Informamos tambien VP y FP en crudo: con 30 contaminados en prueba, un solo
    libro mueve el recall 3 puntos y los porcentajes solos enganian.
    """
    y = datos["contaminado"]
    marcado = pd.Series(marcado, index=datos.index).fillna(False).astype(bool)
    fila = {
        "detector": nombre,
        "precision": round(precision_score(y, marcado, zero_division=0), 2),
        "recall": round(recall_score(y, marcado), 2),
        "marca": int(marcado.sum()),
        "VP": int((marcado & (y == 1)).sum()),
        "FP": int((marcado & (y == 0)).sum()),
    }
    if registrar:
        resultados.append(fila)
    print(f"  {nombre:44s} P={fila['precision']:.2f} R={fila['recall']:.2f}"
          f"  VP={fila['VP']:3d} FP={fila['FP']:3d} marca={fila['marca']:3d}")
    if mostrar_fp:
        for s in datos[marcado & (y == 0)]["synopsis"].head(mostrar_fp):
            print(f"      FP: {s[:140]!r}")
    return fila

### Qué tipo de contaminación hay

Antes de escribir una regex conviene saber contra qué se la escribe. Clasificamos los
fragmentos insertados de **desarrollo** según el rastro que dejan.

In [22]:
# --- 2.0 Caracterizacion de los fragmentos de desarrollo ---
ZERO_WIDTH = {"\u200b", "\u200c", "\u200d", "\ufeff", "\u2060", "\u00ad", "\u180e"}


def tiene_invisible(texto):
    """Caracteres de ancho cero o de formato, que no se ven al leer."""
    return any(c in ZERO_WIDTH or unicodedata.category(c) == "Cf" for c in texto)


def tiene_homoglifo(texto):
    """Letra de otro alfabeto dentro de una palabra latina (cirilica, griega)."""
    for palabra in re.findall(r"\S+", texto):
        latinas = any("a" <= c.lower() <= "z" for c in palabra)
        ajenas = any(
            c.isalpha() and not ("a" <= c.lower() <= "z")
            and "LATIN" not in unicodedata.name(c, "")
            for c in palabra
        )
        if latinas and ajenas:
            return True
    return False


RASGOS = {
    "URL explicita (http/www)": lambda f: bool(re.search(r"https?://|www\.", f, re.I)),
    "dominio sin protocolo": lambda f: bool(
        re.search(r"\b[\w\-]+\.(invalid|example|test|com|net|org)\b", f, re.I)
    ) and not re.search(r"https?://|www\.", f, re.I),
    "correo": lambda f: bool(re.search(r"[\w.\-]+@[\w.\-]+\.\w+", f)),
    "telefono": lambda f: bool(re.search(r"\+\d[\d\s\-().]{7,}\d", f)),
    "caracter invisible": tiene_invisible,
    "homoglifo de otro alfabeto": tiene_homoglifo,
    "letras espaciadas": lambda f: bool(re.search(r"\b(?:[a-zA-Z][ ]){2,}[a-zA-Z]\b", f)),
    "signo escrito con palabras": lambda f: bool(
        re.search(r"\b(punto|barra|arroba|guion)\b", f, re.I)
    ),
    "punto disfrazado [.]": lambda f: bool(re.search(r"[\[(]\s*\.\s*[\])]", f)),
}

frags = desarrollo[desarrollo["contaminado"] == 1]["fragmento"]
conteo = {nombre: int(sum(prueba_f(f) for f in frags)) for nombre, prueba_f in RASGOS.items()}

print(f"rasgos en los {len(frags)} fragmentos contaminados de desarrollo:\n")
for nombre, n in sorted(conteo.items(), key=lambda kv: -kv[1]):
    print(f"  {n:3d} ({n / len(frags):5.1%})  {nombre}")

sin_rastro = [f for f in frags if not any(p(f) for p in RASGOS.values())]
print(f"\n  {len(sin_rastro):3d} ({len(sin_rastro) / len(frags):5.1%})  SIN ningun rastro de enlace")
print("\nLos que no dejan rastro son texto promocional puro: ninguna regex los alcanza,")
print("solo el clasificador de oraciones del 2.5.")
for f in sin_rastro[:3]:
    print(f"   * {f[:110]!r}")

rasgos en los 70 fragmentos contaminados de desarrollo:

   18 (25.7%)  dominio sin protocolo
   13 (18.6%)  URL explicita (http/www)
    7 (10.0%)  caracter invisible
    6 ( 8.6%)  letras espaciadas
    5 ( 7.1%)  punto disfrazado [.]
    4 ( 5.7%)  signo escrito con palabras
    3 ( 4.3%)  correo
    3 ( 4.3%)  homoglifo de otro alfabeto
    1 ( 1.4%)  telefono

   28 (40.0%)  SIN ningun rastro de enlace

Los que no dejan rastro son texto promocional puro: ninguna regex los alcanza,
solo el clasificador de oraciones del 2.5.
   * 'El libro más esperado del año llega por fin en edición digital completa. Leé este y miles de libros más sin pu'
   * 'Conseguí este título y más de 50.000 libros en la app LetraTotal: el primer mes es sin cargo.'
   * 'Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.'


## 2.1 Expresiones regulares

Cuatro patrones: protocolo explícito, correo, teléfono y **dominio desnudo** (sin `http://`
ni `www.`), que es el rastro más frecuente.

Un detalle del dominio desnudo que vale la pena: el TLD se exige **en minúscula y sin
`IGNORECASE`**. Si no, `«de la obra.Es un libro»` matchearía con el TLD `.es`, y una frase
partida sin espacio se convertiría en un falso positivo.

In [23]:
# --- 2.1 Expresiones regulares ---
# TLD reservados (no resuelven en la vida real) y un conjunto corto de reales.
# Dejamos afuera .es, .ar, .co y .me a proposito: son terminaciones de palabras
# frecuentes en castellano y generan falsos positivos.
TLD_RESERVADOS = "invalid|example|test|localhost"
TLD_REALES = "com|net|org|io|info|biz|xyz|online|site|club|shop|store|app|edu|gov"

RE_PROTOCOLO = re.compile(r"https?://\S+|\bwww\.[a-z0-9\-]+\.[a-z]{2,}", re.IGNORECASE)
RE_CORREO = re.compile(r"[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}", re.IGNORECASE)
RE_TELEFONO = re.compile(
    r"\+\d[\d\s\-().]{7,}\d"                   # internacional: +54 9 11 5217-8319
    r"|\b\d{2,4}[\s\-]\d{3,4}[\s\-]\d{3,4}\b"  # local con separadores
)
# Sin IGNORECASE: el TLD tiene que venir en minuscula (ver la nota de arriba).
RE_DOMINIO = re.compile(
    r"\b[A-Za-z0-9][A-Za-z0-9\-]{1,}\.(?:" + TLD_RESERVADOS + "|" + TLD_REALES + r")\b"
)


def detector_regex(texto):
    """True si el texto contiene un enlace, un correo o un telefono."""
    return bool(
        RE_PROTOCOLO.search(texto)
        or RE_CORREO.search(texto)
        or RE_TELEFONO.search(texto)
        or RE_DOMINIO.search(texto)
    )


print("=== cada patron por separado, sobre DESARROLLO ===")
for nombre, patron in [
    ("protocolo (http / www)", RE_PROTOCOLO),
    ("correo", RE_CORREO),
    ("telefono", RE_TELEFONO),
    ("dominio desnudo", RE_DOMINIO),
]:
    medir(nombre, desarrollo["synopsis"].str.contains(patron), desarrollo, registrar=False)

print("\n=== combinados ===")
medir("2.1 regex combinada", desarrollo["synopsis"].map(detector_regex),
      desarrollo, registrar=False, mostrar_fp=2)

=== cada patron por separado, sobre DESARROLLO ===
  protocolo (http / www)                       P=0.93 R=0.19  VP= 13 FP=  1 marca= 14


  correo                                       P=1.00 R=0.04  VP=  3 FP=  0 marca=  3
  telefono                                     P=1.00 R=0.01  VP=  1 FP=  0 marca=  1
  dominio desnudo                              P=0.97 R=0.40  VP= 28 FP=  1 marca= 29

=== combinados ===


  2.1 regex combinada                          P=0.97 R=0.41  VP= 29 FP=  1 marca= 30
      FP: 'Un libro en el que el humor y la ciencia se unen para dar explicación a todo tipo de preguntas curiosas.¡Ey! Gracias por fijarte en mi libro'


{'detector': '2.1 regex combinada',
 'precision': 0.97,
 'recall': 0.41,
 'marca': 30,
 'VP': 29,
 'FP': 1}

### El falso positivo es legítimo

El enunciado avisa que hay sinopsis legítimas que mencionan sitios web reales. En desarrollo
aparece exactamente una, y no es un error de la regex: la sinopsis realmente contiene una
URL, solo que puesta por el autor y no por un spammer.

Esto fija el techo del detector: **una regex de URLs no puede distinguir una URL promocional
de una URL citada**. Para eso hace falta el contexto de la oración, que es lo que aporta el
clasificador del 2.5.

In [24]:
# El unico falso positivo de la regex en desarrollo, con el fragmento que matcheo.
y = desarrollo["contaminado"]
marcados = desarrollo["synopsis"].map(detector_regex)
for s in desarrollo[marcados & (y == 0)]["synopsis"]:
    hallado = RE_PROTOCOLO.search(s) or RE_DOMINIO.search(s)
    inicio = max(0, hallado.start() - 90)
    print(f"matcheo: {hallado.group(0)!r}")
    print(f"contexto: ...{s[inicio:hallado.end() + 70]}...")

matcheo: 'http://www.xkcd.com'
contexto: ...ocimientos científicos y matemáticos. Munroe fue físico en la NASA antes de crear la web
http://www.xkcd.com
, que ha recibido más de un millón de visitas....


## 2.2 Normalizar antes de buscar

Parte de los enlaces están disimulados. Lo que encontramos en la muestra, con los códigos
exactos:

| Truco | Ejemplo | Caracteres |
|---|---|---|
| Carácter invisible dentro del dominio | `capitu​loflash.invalid` | `U+200B`, `U+2060` |
| Homoglifos de otro alfabeto | `lеctorflash.example` | cirílicas `е о а с і` |
| Letras espaciadas + punto entre corchetes | `h t t p s : / / tomoya [.] test` | — |
| Signos escritos con palabras | `TomoMax punto example barra cardamomo` | — |

**`NFKC` no alcanza.** Las homoglifas cirílicas no son variantes de compatibilidad de las
latinas, son letras distintas: `unicodedata.normalize("NFKC", ...)` las deja intactas. Hace
falta un mapeo explícito.

El texto normalizado se usa **solo para detectar**. No reemplaza a la sinopsis: el paso 1
conservó los caracteres raros a propósito.

In [25]:
# --- 2.2 Normalizacion para detectar enlaces ocultos ---
# Homoglifos vistos en la muestra, mas los vecinos obvios del mismo bloque.
HOMOGLIFOS = {
    "\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0441": "c", "\u0456": "i",
    "\u0440": "p", "\u0445": "x", "\u0443": "y", "\u0455": "s", "\u04bb": "h",
    "\u03bf": "o", "\u03b1": "a", "\u03b9": "i", "\u03c1": "p", "\u03c5": "y",
}

# Punto o arroba disfrazados: [.] (.) [dot] [at]
DISFRACES = [
    (r"\s*[\[(\{]\s*\.\s*[\])\}]\s*", "."),
    (r"\s*[\[(\{]\s*(?:dot|punto)\s*[\])\}]\s*", "."),
    (r"\s*[\[(\{]\s*(?:at|arroba)\s*[\])\}]\s*", "@"),
]

# Signos escritos con palabras. El orden importa: 'guion bajo' antes que 'guion'.
PALABRAS_SIGNO = [
    (r"\s+punto\s+", "."), (r"\s+dot\s+", "."),
    (r"\s+barra\s+", "/"), (r"\s+slash\s+", "/"),
    (r"\s+arroba\s+", "@"), (r"\s+guion\s+bajo\s+", "_"), (r"\s+guion\s+", "-"),
]

# Letras sueltas separadas por un espacio: 'h t t p s' -> 'https'. Pedimos tres o
# mas para no pegar iniciales ('J. R. R.') ni siglas de un texto normal.
RE_SUELTAS = re.compile(r"\b(?:[a-zA-Z0-9][ ]){2,}[a-zA-Z0-9]\b")


def normalizar(texto):
    """Deshace la ocultacion de enlaces. Solo para detectar, no para guardar."""
    t = unicodedata.normalize("NFKC", texto)
    # 1. Borrar los invisibles: parten un dominio sin que se note al leerlo.
    t = "".join(c for c in t if c not in ZERO_WIDTH and unicodedata.category(c) != "Cf")
    # 2. Traer las homoglifas al alfabeto latino.
    t = "".join(HOMOGLIFOS.get(c, c) for c in t)
    # 3. Destapar los signos disfrazados entre corchetes.
    for patron, reemplazo in DISFRACES:
        t = re.sub(patron, reemplazo, t, flags=re.IGNORECASE)
    # 4. Juntar las letras espaciadas.
    t = RE_SUELTAS.sub(lambda m: m.group(0).replace(" ", ""), t)
    # 5. Reponer los signos escritos con palabras.
    for patron, reemplazo in PALABRAS_SIGNO:
        t = re.sub(patron, reemplazo, t, flags=re.IGNORECASE)
    # 6. Cerrar los separadores que quedaron sueltos: 'https : / / x . test'.
    t = re.sub(r"\s*:\s*/\s*/\s*", "://", t)
    t = re.sub(r"(?<=[A-Za-z0-9])\s*\.\s*(?=(?:" + TLD_RESERVADOS + "|"
               + TLD_REALES + r")\b)", ".", t)
    return t


def detector_normalizado(texto):
    """La misma regex del 2.1, pero sobre el texto con la ocultacion deshecha."""
    return detector_regex(normalizar(texto))

### Los casos que recupera la normalización

In [26]:
# --- 2.2 Casos concretos ---
CASOS = [
    "Visit\u00e1 capitu\u200bloflash.invalid para la versi\u00f3n completa.",
    "Entr\u00e1 a l\u0435\u0441t\u043erflash.example y le\u00ed sin l\u00edmites.",
    "Link en nuestro sitio: h t t p s : / / tomoya [.] test",
    "Descargalo gratis en TomoMax punto example barra cardamomo",
]

for caso in CASOS:
    normalizado = normalizar(caso)
    antes, despues = detector_regex(caso), detector_regex(normalizado)
    print(f"  antes:    {caso!r}")
    print(f"  despues:  {normalizado!r}")
    print(f"  detecta:  crudo={antes}  normalizado={despues}"
          f"   {'<== recuperado' if despues and not antes else ''}\n")

  antes:    'Visitá capitu\u200bloflash.invalid para la versión completa.'
  despues:  'Visitá capituloflash.invalid para la versión completa.'
  detecta:  crudo=True  normalizado=True   

  antes:    'Entrá a lесtоrflash.example y leí sin límites.'
  despues:  'Entrá a lectorflash.example y leí sin límites.'
  detecta:  crudo=False  normalizado=True   <== recuperado

  antes:    'Link en nuestro sitio: h t t p s : / / tomoya [.] test'
  despues:  'Link en nuestro sitio: https://tomoya.test'
  detecta:  crudo=False  normalizado=True   <== recuperado

  antes:    'Descargalo gratis en TomoMax punto example barra cardamomo'
  despues:  'Descargalo gratis en TomoMax.example/cardamomo'
  detecta:  crudo=False  normalizado=True   <== recuperado



In [27]:
# --- 2.2 Medicion sobre DESARROLLO ---
marca_cruda = desarrollo["synopsis"].map(detector_regex)
marca_norm = desarrollo["synopsis"].map(detector_normalizado)

print("=== DESARROLLO ===")
medir("2.1 regex sobre texto crudo", marca_cruda, desarrollo, registrar=False)
medir("2.2 regex sobre texto normalizado", marca_norm, desarrollo, registrar=False)

recuperados = desarrollo[marca_norm & ~marca_cruda & (desarrollo["contaminado"] == 1)]
print(f"\ncontaminados que SOLO ve la version normalizada: {len(recuperados)}")
for f in recuperados["fragmento"].head(5):
    print(f"   * {f[:110]!r}")

=== DESARROLLO ===
  2.1 regex sobre texto crudo                  P=0.97 R=0.41  VP= 29 FP=  1 marca= 30
  2.2 regex sobre texto normalizado            P=0.98 R=0.60  VP= 42 FP=  1 marca= 43

contaminados que SOLO ve la version normalizada: 13
   * 'Entrá a librоexpress.invalid y leé sin límites.'
   * 'Link en nuestro sitio: h t t p s : / / epubhub [.] example'
   * 'Link en nuestro sitio: h t t p s : / / lectorplus [.] invalid'
   * 'Link en nuestro sitio: h t t p s : / / tomoya [.] test'
   * 'Descargalo gratis en TomoClub punto test barra blasfemia'


### Un carácter invisible, por sí solo, no es evidencia

El enunciado lo advierte y conviene medirlo en lugar de creerlo: marcar un libro solo porque
su sinopsis contiene un carácter invisible da una precisión muy pobre. Los invisibles
aparecen también en sinopsis legítimas, probablemente arrastrados del HTML del sitio.

Por eso el `U+200B` no es el detector: el detector es **el dominio que aparece una vez que se
lo quita**.

In [28]:
# --- 2.2 El invisible solo, como detector ---
print("=== DESARROLLO ===")
medir("solo 'contiene caracter invisible'", desarrollo["synopsis"].map(tiene_invisible),
      desarrollo, registrar=False, mostrar_fp=2)

limpias_con_invisible = desarrollo[
    (desarrollo["contaminado"] == 0) & desarrollo["synopsis"].map(tiene_invisible)
]
print(f"\nsinopsis LIMPIAS con algun invisible: {len(limpias_con_invisible)}"
      f" de {int((desarrollo['contaminado'] == 0).sum())}")

=== DESARROLLO ===


  solo 'contiene caracter invisible'           P=0.64 R=0.10  VP=  7 FP=  4 marca= 11
      FP: 'La presente edición de\r\nHispania\r\n, otra de sus obras fundamentales es, ciertamente, oportuna, pues los ejemplares de la única edición españ'
      FP: 'Nadie en el vecindario, ni siquiera su propio marido, pudo entender que la señora Ranelagh se preocupara tan vivamente por la muerte de Anni'



sinopsis LIMPIAS con algun invisible: 4 de 280


## 2.3 Detectar el idioma de cada oración

Parte de los mensajes están en otro idioma dentro de una sinopsis en castellano. El enunciado
pide empezar por la sinopsis completa y observar qué pasa con los registros contaminados.

In [29]:
# --- 2.3 Idioma de la sinopsis completa ---
# langdetect muestrea internamente al elegir los n-gramas que mira. Fijamos la
# semilla por el mismo motivo que random_state=42: que la corrida sea repetible.
DetectorFactory.seed = 0


def idioma_de(texto):
    """(idioma mas probable, probabilidad). (None, 0.0) si no puede decidir."""
    try:
        candidatos = detect_langs(texto)
        return (candidatos[0].lang, candidatos[0].prob) if candidatos else (None, 0.0)
    except LangDetectException:
        return None, 0.0


for etiqueta, grupo in [("limpias", desarrollo[desarrollo["contaminado"] == 0]),
                        ("contaminadas", desarrollo[desarrollo["contaminado"] == 1])]:
    idiomas = Counter(idioma_de(s)[0] for s in grupo["synopsis"])
    print(f"  sinopsis {etiqueta:14s} {idiomas.most_common(5)}")

  sinopsis limpias        [('es', 280)]


  sinopsis contaminadas   [('es', 70)]


**La sinopsis completa da `es` en los 350 casos**, contaminados incluidos. El detector a nivel
documento tiene recall 0: el mensaje insertado es una oración entre siete y queda diluido.

Hay que bajar a nivel de oración. Usamos el partidor de la guía del enunciado.

In [30]:
# --- 2.3 Partidor de oraciones (el de la guia del enunciado) ---
def oraciones(texto):
    """Corta en oraciones por signo terminal o salto de linea.

    El minimo de 15 caracteres saca los restos de puntuacion que no son oraciones.
    """
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]


# Idioma del fragmento insertado, que es donde deberia verse el otro idioma.
frags = desarrollo[desarrollo["contaminado"] == 1]["fragmento"]
idiomas_frag = Counter(idioma_de(f)[0] for f in frags)
print(f"idioma del fragmento insertado (n={len(frags)}): {idiomas_frag.most_common()}")
ajenos = sum(n for lang, n in idiomas_frag.items() if lang not in (None, "es"))
print(f"  fragmentos en otro idioma: {ajenos} de {len(frags)}")

idioma del fragmento insertado (n=70): [('es', 58), ('en', 8), ('pt', 2), ('fr', 2)]
  fragmentos en otro idioma: 12 de 70


### Los tres errores que hay que neutralizar

Aplicar `detect_langs` oración por oración sin cuidado produce muchos falsos positivos. Medido
sobre las sinopsis **limpias** de desarrollo, hay decenas de oraciones castellanas que
langdetect lee como otro idioma. Mirando los casos aparecen tres causas distintas:

1. **Lenguas cercanas.** `«Es ésta la segunda entrega del ciclo que se inicia con Redwall.»`
   sale como catalán con probabilidad 1.00. El castellano, el catalán y el gallego comparten
   demasiados n-gramas. Tratamos `ca` y `gl` como equivalentes a `es`.
2. **Mayúsculas.** Una oración en versales sale como portugués con probabilidad 1.00. Las
   mayúsculas distorsionan los n-gramas, así que **minusculizamos antes de detectar**.
3. **Oraciones cortas, citas y nombres propios**, que es lo que el enunciado advierte. Se
   neutralizan con un mínimo de palabras y descartando citas y oraciones que el partidor
   cortó por la mitad.

Conviene notar que `NFKC` y las mayúsculas son problemas de *representación*, y las lenguas
cercanas un problema de *modelo*. Los tres hacen falta: cada uno solo deja falsos positivos.

In [31]:
# --- 2.3 El detector ---
# Lenguas que langdetect no separa del castellano de forma confiable.
LENGUAS_CERCANAS = {"es", "ca", "gl"}
MIN_PALABRAS = 8      # calibrado abajo; las fallas se concentran en oraciones cortas
UMBRAL_IDIOMA = 0.90  # calibrado abajo; hay meseta entre 0.90 y 0.999

COMILLAS = "\u00ab\u00bb\u201c\u201d\u2018\u2019\"'"
RE_ENTRECOMILLADO = re.compile(r"[\u00ab\u201c\"][^\u00bb\u201d\"]*[\u00bb\u201d\"]")


def es_cita(oracion):
    """Oracion mayormente entre comillas. El enunciado avisa que dan error."""
    texto = oracion.strip()
    if texto[:1] in COMILLAS:
        return True
    dentro = sum(len(m) for m in RE_ENTRECOMILLADO.findall(texto))
    return dentro > 0.4 * len(texto)


def esta_truncada(oracion):
    """El partidor la corto: no termina en signo terminal."""
    return not oracion.rstrip().endswith((".", "!", "?", "\u2026"))


_cache_idioma = {}


def idioma_normalizado(oracion):
    """Idioma de la oracion en minusculas, con cache.

    El cache importa: langdetect es lo mas lento del paso 2 y el barrido de
    calibracion vuelve a pasar por las mismas oraciones muchas veces.
    """
    if oracion not in _cache_idioma:
        _cache_idioma[oracion] = idioma_de(oracion.lower())
    return _cache_idioma[oracion]


def oracion_en_otro_idioma(oracion, min_palabras=MIN_PALABRAS, umbral=UMBRAL_IDIOMA):
    """(idioma, prob) si la oracion esta en otro idioma con confianza; None si no."""
    if len(oracion.split()) < min_palabras:
        return None
    if es_cita(oracion) or esta_truncada(oracion):
        return None
    lang, prob = idioma_normalizado(oracion)
    if lang and lang not in LENGUAS_CERCANAS and prob >= umbral:
        return lang, prob
    return None


def detector_idioma(texto, min_palabras=MIN_PALABRAS, umbral=UMBRAL_IDIOMA):
    """True si alguna oracion de la sinopsis esta en otro idioma."""
    return any(
        oracion_en_otro_idioma(o, min_palabras, umbral) for o in oraciones(texto)
    )

### Calibración de los dos valores sobre desarrollo

In [32]:
# --- 2.3 Barrido de min_palabras y umbral, SOLO sobre desarrollo ---
print(f"{'min_pal':>8} {'umbral':>7} {'P':>6} {'R':>6} {'VP':>4} {'FP':>4}")
print("-" * 42)
for min_pal in [5, 6, 8, 10, 12]:
    for umbral in [0.90, 0.95, 0.99]:
        marcado = desarrollo["synopsis"].map(
            lambda s, mp=min_pal, um=umbral: detector_idioma(s, mp, um)
        )
        fila = medir(f"min_pal={min_pal} umbral={umbral}", marcado, desarrollo,
                     registrar=False)
        print(f"{min_pal:8d} {umbral:7.2f} {fila['precision']:6.2f}"
              f" {fila['recall']:6.2f} {fila['VP']:4d} {fila['FP']:4d}")

 min_pal  umbral      P      R   VP   FP
------------------------------------------


  min_pal=5 umbral=0.9                         P=0.85 R=0.16  VP= 11 FP=  2 marca= 13
       5    0.90   0.85   0.16   11    2
  min_pal=5 umbral=0.95                        P=0.85 R=0.16  VP= 11 FP=  2 marca= 13
       5    0.95   0.85   0.16   11    2


  min_pal=5 umbral=0.99                        P=0.85 R=0.16  VP= 11 FP=  2 marca= 13
       5    0.99   0.85   0.16   11    2


  min_pal=6 umbral=0.9                         P=1.00 R=0.16  VP= 11 FP=  0 marca= 11
       6    0.90   1.00   0.16   11    0
  min_pal=6 umbral=0.95                        P=1.00 R=0.16  VP= 11 FP=  0 marca= 11
       6    0.95   1.00   0.16   11    0


  min_pal=6 umbral=0.99                        P=1.00 R=0.16  VP= 11 FP=  0 marca= 11
       6    0.99   1.00   0.16   11    0


  min_pal=8 umbral=0.9                         P=1.00 R=0.10  VP=  7 FP=  0 marca=  7
       8    0.90   1.00   0.10    7    0
  min_pal=8 umbral=0.95                        P=1.00 R=0.10  VP=  7 FP=  0 marca=  7
       8    0.95   1.00   0.10    7    0


  min_pal=8 umbral=0.99                        P=1.00 R=0.10  VP=  7 FP=  0 marca=  7
       8    0.99   1.00   0.10    7    0
  min_pal=10 umbral=0.9                        P=1.00 R=0.10  VP=  7 FP=  0 marca=  7
      10    0.90   1.00   0.10    7    0


  min_pal=10 umbral=0.95                       P=1.00 R=0.10  VP=  7 FP=  0 marca=  7
      10    0.95   1.00   0.10    7    0
  min_pal=10 umbral=0.99                       P=1.00 R=0.10  VP=  7 FP=  0 marca=  7
      10    0.99   1.00   0.10    7    0


  min_pal=12 umbral=0.9                        P=1.00 R=0.03  VP=  2 FP=  0 marca=  2
      12    0.90   1.00   0.03    2    0
  min_pal=12 umbral=0.95                       P=1.00 R=0.03  VP=  2 FP=  0 marca=  2
      12    0.95   1.00   0.03    2    0


  min_pal=12 umbral=0.99                       P=1.00 R=0.03  VP=  2 FP=  0 marca=  2
      12    0.99   1.00   0.03    2    0


**Qué dice el barrido.** Con `min_palabras = 8` y umbral `0.90` la precisión en desarrollo es
1.00 sin falsos positivos. Los valores no están al filo: `min_palabras` 8 y 10 dan el mismo
resultado, y el umbral da lo mismo de 0.90 a 0.99, porque langdetect devuelve probabilidades
muy cerca de 1 cuando acierta. El parámetro que de verdad manda es el mínimo de palabras.

Bajar a `min_palabras = 6` sube el recall en desarrollo, pero las fallas del detector son
justamente oraciones cortas con nombres propios extranjeros, así que preferimos el valor
conservador: cada falso positivo es un libro que el recomendador nunca podrá sugerir.

In [33]:
# --- 2.3 Que oraciones marca en desarrollo ---
print("oraciones marcadas en desarrollo (etiqueta real del libro):\n")
for s, cont in zip(desarrollo["synopsis"], desarrollo["contaminado"]):
    for o in oraciones(s):
        hallado = oracion_en_otro_idioma(o)
        if hallado:
            lang, prob = hallado
            print(f"  [{'CONTAMINADO' if cont else 'LIMPIO     '}] [{lang} {prob:.2f}]"
                  f" {o[:88]!r}")
            break

oraciones marcadas en desarrollo (etiqueta real del libro):

  [CONTAMINADO] [en 1.00] 'Download this ebook for FREE, no sign-up required, at biblioya.test today.'
  [CONTAMINADO] [pt 1.00] 'Baixe este livro grátis agora mesmo, sem cadastro, em librohub.test.'
  [CONTAMINADO] [en 1.00] 'Download this ebook for FREE, no sign-up required, at relatoflash.test today.'
  [CONTAMINADO] [en 1.00] 'Download this ebook for FREE, no sign-up required, at lectorya.test today.'
  [CONTAMINADO] [en 1.00] 'Download this ebook for FREE, no sign-up required, at capitulototal.invalid today.'
  [CONTAMINADO] [pt 1.00] 'Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!'
  [CONTAMINADO] [en 1.00] 'Get unlimited access to thousands of bestsellers with our limited time offer!'


## Medición final sobre la parte de prueba

Hasta acá todo se decidió mirando desarrollo. Ahora se mide sobre los 150 libros reservados,
que es el número que va al informe.

In [34]:
# --- Medicion final: PRUEBA ---
print(f"=== PRUEBA (n={len(prueba)}, contaminados={int(prueba['contaminado'].sum())}) ===")
medir("linea de base: regex minima del enunciado",
      prueba["synopsis"].str.contains(re.compile(r"https?://|www\.", re.IGNORECASE)), prueba)
medir("2.1 regex propia (texto crudo)", prueba["synopsis"].map(detector_regex), prueba)
marca_22 = prueba["synopsis"].map(detector_normalizado)
medir("2.2 regex + normalizacion", marca_22, prueba)
marca_23 = prueba["synopsis"].map(detector_idioma)
medir("2.3 idioma por oracion", marca_23, prueba)
medir("2.2 OR 2.3 (combinacion parcial)", marca_22 | marca_23, prueba)

print()
print(pd.DataFrame(resultados).to_string(index=False))

nuevos = prueba[marca_23 & ~marca_22 & (prueba["contaminado"] == 1)]
print(f"\naporte neto del 2.3 sobre el 2.2: {len(nuevos)} libros")
for f in nuevos["fragmento"]:
    print(f"   * {f[:95]!r}")

=== PRUEBA (n=150, contaminados=30) ===
  linea de base: regex minima del enunciado    P=1.00 R=0.13  VP=  4 FP=  0 marca=  4


  2.1 regex propia (texto crudo)               P=1.00 R=0.40  VP= 12 FP=  0 marca= 12


  2.2 regex + normalizacion                    P=1.00 R=0.63  VP= 19 FP=  0 marca= 19


  2.3 idioma por oracion                       P=0.89 R=0.27  VP=  8 FP=  1 marca=  9
  2.2 OR 2.3 (combinacion parcial)             P=0.96 R=0.80  VP= 24 FP=  1 marca= 25

                                 detector  precision  recall  marca  VP  FP
linea de base: regex minima del enunciado       1.00    0.13      4   4   0
           2.1 regex propia (texto crudo)       1.00    0.40     12  12   0
                2.2 regex + normalizacion       1.00    0.63     19  19   0
                   2.3 idioma por oracion       0.89    0.27      9   8   1
         2.2 OR 2.3 (combinacion parcial)       0.96    0.80     25  24   1

aporte neto del 2.3 sobre el 2.2: 5 libros
   * 'Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!'
   * 'Get unlimited access to thousands of bestsellers with our limited time offer!'
   * 'Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!'
   * 'Leia milhares de livros sem pagar nada com a nossa oferta por temp

### La calibración no transfirió, y eso es el hallazgo

En desarrollo el 2.3 da precisión **1.00** sin falsos positivos. En prueba baja a **0.89**, con
un falso positivo. No es mala suerte: es la naturaleza del detector.

Sus errores sobre texto castellano son **independientes de la contaminación**. En desarrollo
las oraciones mal leídas cayeron sobre libros que además estaban contaminados, así que
contaron como aciertos; en prueba cayeron sobre libros limpios. El único falso positivo de
prueba lo muestra bien: una oración castellana que enumera títulos de libros en inglés, o sea
la lista de nombres propios que el enunciado anticipa.

**Qué declarar en el informe.** Con 30 contaminados en prueba, un libro mueve la precisión
once puntos; hablar de 0.89 como si fuera un valor estable sería engañoso. Lo honesto es decir
que la precisión del 2.3 está en el orden de 0.9 con mucha incertidumbre, y que su aporte real
no es el número propio sino los libros que suma a los otros detectores.

**Advertencia metodológica**, que conviene declarar antes de que la pregunten: durante el
desarrollo de este detector la parte de prueba se consultó más de una vez, al descubrir que la
calibración no transfería. Las decisiones se tomaron sobre desarrollo, pero mirar prueba
varias veces debilita la independencia de la medición final.

### Lo que aporta el 2.3

Como detector solo es débil: recall bajo y precisión inestable. Su valor es **incremental**.
Los libros que suma sobre el 2.2 son promoción pura en portugués o en inglés, sin ningún
enlace, del grupo que ninguna expresión regular puede alcanzar.

Es decir: el 2.3 no compite con el 2.1 y el 2.2, los complementa en la zona donde ellos son
ciegos. La combinación `2.2 OR 2.3` sobre prueba llega a un recall bastante mayor que
cualquiera de los dos por separado, y ese es el número que importa.

## 2.4 Reconocer entidades

El enunciado pide aplicar NER a las oraciones de los fragmentos contaminados y a una cantidad
similar de oraciones limpias, y responder dos cosas: **qué entidades aparecen en el texto
insertado que no aparecen en una sinopsis normal**, y si **sirven para detectar o solo para
explicar** lo que encontraron otros detectores.

Primero hay que construir el corpus de oraciones etiquetadas. **El 2.5 usa este mismo
corpus**, así que se arma una sola vez acá.

In [35]:
# --- 2.4 Corpus de oraciones etiquetadas (lo reusa el 2.5) ---
def armar_corpus_oraciones(datos):
    """Oraciones con etiqueta: 1 si vienen del fragmento insertado, 0 si no.

    Las negativas salen SOLO de las sinopsis limpias. Si tomaramos las oraciones
    no-fragmento de un libro contaminado, cualquier desalineacion del fragmento
    metria una oracion positiva entre las negativas.
    """
    filas = []
    for slug, sinopsis, contaminado, fragmento in zip(
        datos["slug"], datos["synopsis"], datos["contaminado"], datos["fragmento"]
    ):
        if contaminado == 1:
            filas.extend({"slug": slug, "oracion": o, "y": 1} for o in oraciones(fragmento))
        else:
            filas.extend({"slug": slug, "oracion": o, "y": 0} for o in oraciones(sinopsis))
    return pd.DataFrame(filas)


corpus = armar_corpus_oraciones(desarrollo)
n_pos = int((corpus["y"] == 1).sum())
n_neg = int((corpus["y"] == 0).sum())
print(f"corpus de desarrollo: {len(corpus)} oraciones")
print(f"  positivas (fragmento):     {n_pos}")
print(f"  negativas (sinopsis limpia): {n_neg}")
print(f"  desbalance: 1 positiva cada {n_neg / n_pos:.1f} negativas")
print(f"  piso de azar: {n_pos / len(corpus):.3f}")
print(f"\n  oraciones por fragmento: "
      f"{dict(sorted(Counter(corpus[corpus.y == 1].groupby('slug').size()).items()))}")

corpus de desarrollo: 2095 oraciones
  positivas (fragmento):     88
  negativas (sinopsis limpia): 2007
  desbalance: 1 positiva cada 22.8 negativas
  piso de azar: 0.042

  oraciones por fragmento: {1: 53, 2: 16, 3: 1}


Ese desbalance de 1 a 23 es exactamente el motivo del `class_weight="balanced"` que el
enunciado fija: sin él, predecir siempre 0 acierta el 96 % de las veces.

### Un confundido que conviene medir antes de interpretar nada

Las oraciones contaminadas son **mucho más cortas** que las limpias. Antes de atribuirle mérito
a cualquier detector hay que ver cuánto explica el largo por sí solo.

In [36]:
# --- 2.4 El largo de la oracion, como detector por si solo ---
corpus["palabras"] = corpus["oracion"].str.split().str.len()

print("largo en palabras por clase:")
for etiqueta in (1, 0):
    s = corpus[corpus["y"] == etiqueta]["palabras"]
    print(f"  y={etiqueta}: mediana={s.median():.0f} media={s.mean():.1f}"
          f" min={s.min()} max={s.max()}")

print("\nel largo solo, como detector:")
for umbral in [8, 12, 16, 18, 20]:
    marcado = corpus["palabras"] <= umbral
    medir(f"palabras <= {umbral}", marcado, corpus.rename(columns={"y": "contaminado"}),
          registrar=False)

largo en palabras por clase:
  y=1: mediana=8 media=10.1 min=4 max=18
  y=0: mediana=20 media=22.7 min=1 max=147

el largo solo, como detector:
  palabras <= 8                                P=0.11 R=0.55  VP= 48 FP=397 marca=445
  palabras <= 12                               P=0.09 R=0.66  VP= 58 FP=617 marca=675


  palabras <= 16                               P=0.08 R=0.82  VP= 72 FP=829 marca=901
  palabras <= 18                               P=0.08 R=1.00  VP= 88 FP=955 marca=1043


  palabras <= 20                               P=0.08 R=1.00  VP= 88 FP=1054 marca=1142


**Las 88 oraciones contaminadas tienen 18 palabras o menos; las limpias llegan a 147.** El
largo es señal real, pero como detector es inútil: su mejor precisión es 0.11 contra un piso de
azar de 0.042. Sirve como aviso, no como criterio.

Lo importante es que **no invalida** lo que venga después: si el largo decidiera todo, la
precisión de un detector basado en él sería alta, y no lo es.

In [37]:
# --- 2.4 Modelo de NER ---
# El modelo va aparte de la libreria: pesa unos 550 MB y en Colab hay que bajarlo
# en cada sesion nueva. Conviene lanzar esta celda al principio y seguir
# trabajando mientras baja.
import spacy

if not spacy.util.is_package("es_core_news_lg"):
    spacy.cli.download("es_core_news_lg")

nlp = spacy.load("es_core_news_lg")
print("modelo:", nlp.meta["name"], nlp.meta["version"])
print("etiquetas de NER disponibles:", nlp.get_pipe("ner").labels)

modelo: core_news_lg 3.8.0
etiquetas de NER disponibles: ('LOC', 'MISC', 'ORG', 'PER')


`es_core_news_lg` solo distingue **cuatro** tipos: `LOC`, `MISC`, `ORG` y `PER`. Ninguno
corresponde a lo que querríamos buscar en el spam (un sitio web, una marca, un precio). Esa
limitación del conjunto de etiquetas es la razón por la que el enunciado ofrece GLiNER como
alternativa: con GLiNER se piden etiquetas propias.

In [38]:
# --- 2.4 Entidades en cada clase ---
# El enunciado pide "una cantidad similar de oraciones limpias": tomamos tantas
# negativas como positivas para que los conteos sean comparables y no los domine
# el desbalance 1:23.
positivas = corpus[corpus["y"] == 1]["oracion"].tolist()
negativas = corpus[corpus["y"] == 0]["oracion"].sample(
    n=len(positivas), random_state=42
).tolist()
print(f"comparando {len(positivas)} contaminadas vs {len(negativas)} limpias\n")


def analizar_entidades(lista_oraciones):
    """(conteo por etiqueta, lista de entidades por oracion)."""
    conteo = Counter()
    por_oracion = []
    for doc in nlp.pipe(lista_oraciones, batch_size=64):
        conteo.update(e.label_ for e in doc.ents)
        por_oracion.append([(e.text, e.label_) for e in doc.ents])
    return conteo, por_oracion


conteo_pos, ents_pos = analizar_entidades(positivas)
conteo_neg, ents_neg = analizar_entidades(negativas)

print(f"{'etiqueta':10s} {'contaminadas':>13s} {'limpias':>9s}")
for et in sorted(set(conteo_pos) | set(conteo_neg)):
    print(f"{et:10s} {conteo_pos.get(et, 0):13d} {conteo_neg.get(et, 0):9d}")

con_pos = sum(1 for e in ents_pos if e)
con_neg = sum(1 for e in ents_neg if e)
print(f"\noraciones con al menos una entidad:")
print(f"  contaminadas: {con_pos}/{len(positivas)} ({con_pos / len(positivas):.0%})")
print(f"  limpias:      {con_neg}/{len(negativas)} ({con_neg / len(negativas):.0%})")

comparando 88 contaminadas vs 88 limpias



etiqueta    contaminadas   limpias
LOC                   15        23
MISC                  37        14
ORG                    2         7
PER                   16        30

oraciones con al menos una entidad:
  contaminadas: 51/88 (58%)
  limpias:      42/88 (48%)


In [39]:
# --- 2.4 Que reconoce como entidad en cada clase ---
for etiqueta, ents in [("CONTAMINADAS", ents_pos), ("LIMPIAS", ents_neg)]:
    textos = Counter()
    for entidades in ents:
        for texto, tipo in entidades:
            textos[(texto, tipo)] += 1
    print(f"\n=== {etiqueta}: entidades mas frecuentes ===")
    for (texto, tipo) in [k for k, _ in textos.most_common(14)]:
        print(f"  {textos[(texto, tipo)]:3d}x [{tipo:4s}] {texto[:70]!r}")


=== CONTAMINADAS: entidades mas frecuentes ===
    7x [PER ] 'compartila'
    6x [MISC] 'PDF'
    5x [PER ] 'Ganá'
    4x [MISC] 'Download this ebook for FREE'
    4x [MISC] 'sign-up required'
    4x [MISC] 'EPUB sin registro'
    4x [MISC] 'Descargalo'
    2x [LOC ] 'Oferta'
    2x [MISC] 'EPUB'
    2x [MISC] 'LibroLibre'
    2x [MISC] 'EpubMax'
    1x [LOC ] 'Visitá pagin\u2060azona.invalid'
    1x [LOC ] 'Baixe'
    1x [LOC ] 'grátis agora mesmo'

=== LIMPIAS: entidades mas frecuentes ===
    4x [LOC ] 'Europa'
    2x [LOC ] 'Ellis'
    2x [LOC ] 'la India'
    2x [PER ] 'Hitler'
    2x [PER ] 'Platero'
    1x [PER ] 'Dispenza'
    1x [PER ] 'Goethe'
    1x [MISC] 'Tiempos de barro'
    1x [LOC ] 'adarga antigua'
    1x [LOC ] 'rocín flaco'
    1x [PER ] 'John Mitchell'
    1x [PER ] 'Luis-Eduardo Aute'
    1x [MISC] 'Tablas'
    1x [MISC] 'los Elementos'


### Qué aparece en el texto insertado que no aparece en una sinopsis normal

La diferencia es clara pero no es la que uno esperaría. En las oraciones limpias las entidades
son **personas, lugares y obras reales**: Europa, Goethe, Hitler, Mendéleiev. En las
contaminadas aparecen dos clases de cosas:

1. **Nombres de marca inventados** de los sitios de spam, etiquetados casi siempre como `MISC`
   o `ORG`: LibroLibre, EpubMax, TomoClub, LectorPlus, LetraTotal. Estos sí son entidades
   genuinas, y son lo más parecido a una señal útil.
2. **Errores de etiquetado sobre verbos en imperativo.** `compartila` y `Gané` salen como
   `PER`, `Descargalo` como `MISC`, `Baixe` como `LOC`. Son formas del voseo y del portugués
   que el modelo no reconoce como verbos, así que las toma por nombres propios.

`MISC` es el tipo más discriminativo: 37 apariciones en las contaminadas contra 14 en las
limpias, sobre muestras del mismo tamaño. Pero buena parte de ese 37 son los errores del punto
2, no entidades reales.

In [40]:
# --- 2.4 NER como detector: sirve o no? ---
todas = corpus["oracion"].tolist()
_, ents_todas = analizar_entidades(todas)
corpus_m = corpus.rename(columns={"y": "contaminado"})

print(f"piso de azar: {n_pos / len(corpus):.3f}\n")
medir("tiene alguna entidad", [bool(e) for e in ents_todas], corpus_m, registrar=False)
medir("NO tiene ninguna entidad", [not e for e in ents_todas], corpus_m, registrar=False)
for et in ["MISC", "ORG", "LOC", "PER"]:
    medir(f"tiene entidad {et}",
          [any(t == et for _, t in e) for e in ents_todas], corpus_m, registrar=False)

piso de azar: 0.042

  tiene alguna entidad                         P=0.05 R=0.58  VP= 51 FP=1045 marca=1096
  NO tiene ninguna entidad                     P=0.04 R=0.42  VP= 37 FP=962 marca=999
  tiene entidad MISC                           P=0.07 R=0.27  VP= 24 FP=313 marca=337
  tiene entidad ORG                            P=0.02 R=0.02  VP=  2 FP= 91 marca= 93
  tiene entidad LOC                            P=0.04 R=0.16  VP= 14 FP=303 marca=317
  tiene entidad PER                            P=0.02 R=0.17  VP= 15 FP=664 marca=679


### ¿Sirven para detectar o solo para explicar?

**Solo para explicar.** Ninguna variante pasa de una precisión de 0.07 contra un piso de azar
de 0.042: el mejor detector basado en NER es apenas 1,7 veces mejor que marcar oraciones al
azar. Como criterio para sacar libros del catálogo es inservible.

Por eso el 2.4 **no entra en la tabla de detectores** del 2.7. Incluirlo arruinaría la
precisión de la combinación sin aportar recall.

Dónde sí sirve: para **describir** la contaminación que los otros detectores encontraron. Poder
decir "los fragmentos insertados nombran marcas comerciales que no existen en el catálogo" es
una caracterización útil para el informe y para la defensa.

Hay un matiz que conviene declarar: usar NER como explicación tiene su propia trampa, porque
buena parte de las "entidades" del texto insertado son **errores del modelo** sobre imperativos
del voseo. Explicar el spam señalando esas entidades sería explicarlo por una confusión del
etiquetador, no por una propiedad del texto.

**Lo que quedó sin probar.** Las cuatro etiquetas de `es_core_news_lg` no incluyen nada
parecido a "sitio web", "marca comercial" o "precio", que es lo que haría falta. GLiNER permite
pedir etiquetas propias y es plausible que funcione mejor, pero requiere instalar torch (más de
2 GB) y no lo medimos. Queda como la extensión natural de este apartado, sin afirmar de
antemano cuánto mejoraría.

## 2.5 Clasificar oraciones

El 2.2 llegó a recall 0.63 y ahí se detuvo. Lo que falta es el **texto promocional sin
enlace**, que no deja ninguna cadena reconocible: una regex no tiene a qué agarrarse. El
enunciado pide aprenderlo de ejemplos con dos representaciones —TF-IDF y embeddings— y la
misma regresión logística, para después comparar cuál conviene.

El corpus de oraciones ya está armado en el 2.4: lo reusamos tal cual.

La configuración de la regresión logística es **fija**, la del paso 5:
`C=1`, `max_iter=1000`, `class_weight="balanced"`. No hay búsqueda de hiperparámetros, y el
`class_weight` no es un ajuste para mejorar resultados: con 88 positivas entre 2095 oraciones,
sin él el modelo acierta el 96 % prediciendo siempre 0.

In [41]:
# --- 2.5 Configuracion fija de los dos clasificadores ---
# La del paso 5 del enunciado. Se mantiene en los dos modelos y en todo el TP.
CFG_LR = dict(C=1, max_iter=1000, class_weight="balanced")

X_ora = corpus["oracion"].tolist()
y_ora = corpus["y"].values
g_ora = corpus["slug"].values          # a que libro pertenece cada oracion

print(f"oraciones de entrenamiento: {len(X_ora)}  positivas: {int(y_ora.sum())}")
print(f"libros que las aportan:     {len(set(g_ora))}")

oraciones de entrenamiento: 2095  positivas: 88
libros que las aportan:     350


### ¿Por qué oraciones y no la sinopsis completa?

Es la primera pregunta del apartado y la contestamos midiendo, no argumentando: entrenamos el
**mismo** TF-IDF con regresión logística de las dos formas y los comparamos sobre los mismos
libros de prueba.

In [42]:
# --- 2.5 El experimento que justifica trabajar por oracion ---
# Un documento por libro: la sinopsis entera.
pipe_doc = make_pipeline(TfidfVectorizer(), LogisticRegression(**CFG_LR))
prob_doc = cross_val_predict(
    pipe_doc, desarrollo["synopsis"].tolist(), desarrollo["contaminado"].values,
    cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]


def umbral_para_precision(y_true, prob, objetivo=0.90):
    """Umbral mas bajo que alcanza la precision objetivo.

    El mas bajo y no cualquiera: a igual precision, cuanto mas bajo el umbral,
    mas recall. Es lo que pide el enunciado al comparar los dos modelos.
    """
    p, _, u = precision_recall_curve(y_true, prob)
    posibles = np.where(p[:-1] >= objetivo)[0]
    return float(u[posibles[0]]) if len(posibles) else None


umbral_doc = umbral_para_precision(desarrollo["contaminado"].values, prob_doc)
pipe_doc.fit(desarrollo["synopsis"].tolist(), desarrollo["contaminado"].values)
prob_doc_prueba = pipe_doc.predict_proba(prueba["synopsis"].tolist())[:, 1]
es_cont = (prueba["contaminado"] == 1).values

print(f"umbral elegido en desarrollo: {umbral_doc:.3f}")
print(f"probabilidad en prueba, contaminados: media={prob_doc_prueba[es_cont].mean():.3f}"
      f" max={prob_doc_prueba[es_cont].max():.3f}")
print(f"                        limpios:      media={prob_doc_prueba[~es_cont].mean():.3f}"
      f" max={prob_doc_prueba[~es_cont].max():.3f}")
print(f"libros marcados en prueba: {int((prob_doc_prueba >= umbral_doc).sum())}")

# Cuanto del documento es el fragmento: la razon del fracaso
cont_pru = prueba[prueba["contaminado"] == 1]
frac = np.array([len(f) / len(s) for f, s in zip(cont_pru["fragmento"], cont_pru["synopsis"])])
print(f"\nel fragmento es el {frac.mean():.1%} de los caracteres de la sinopsis"
      f" (mediana {np.median(frac):.1%}, maximo {frac.max():.1%})")

umbral elegido en desarrollo: 0.562
probabilidad en prueba, contaminados: media=0.471 max=0.546
                        limpios:      media=0.404 max=0.500
libros marcados en prueba: 0

el fragmento es el 7.4% de los caracteres de la sinopsis (mediana 6.9%, maximo 17.4%)


**El modelo de sinopsis completa no marca ni un libro.** Su probabilidad sobre prueba va de
0.40 de media en los limpios a 0.47 en los contaminados, con máximos de 0.500 y 0.546: las dos
distribuciones se superponen casi del todo y ninguna cruza el umbral que la curva PR eligió en
desarrollo.

No es mala suerte del umbral. Incluso eligiéndolo *sobre prueba* —algo que no se puede hacer—
el mejor punto da precisión 0.59, y para llegar a precisión 0.90 el recall cae a 0.23.

La causa es aritmética: **el fragmento es el 7 % del texto**. TF-IDF normaliza el vector del
documento, así que siete oraciones de tema literario reparten el peso y diluyen la única
promocional. Al clasificar oración por oración, esa oración es el documento entero.

Es el mismo fenómeno que ya habíamos medido en el 2.3, donde la sinopsis completa daba `es` en
los 350 casos. Dos detectores distintos, la misma lección: **el nivel de análisis tiene que
coincidir con el tamaño de la señal.**

In [43]:
# --- 2.5 Modelo A: TF-IDF ---
# TfidfVectorizer con su configuracion por defecto, como pide el enunciado. El
# fit_transform ve solo las oraciones de desarrollo: el vocabulario y los pesos
# idf se aprenden ahi y nunca con prueba.
pipe_tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(**CFG_LR))

# Probabilidades fuera de muestra para elegir el umbral. Los pliegues se agrupan
# por LIBRO: dos oraciones del mismo fragmento son casi identicas, y si caen en
# pliegues distintos el modelo ya vio la respuesta y el umbral sale optimista.
prob_tfidf_des = cross_val_predict(
    pipe_tfidf, X_ora, y_ora, groups=g_ora, cv=GroupKFold(n_splits=5),
    method="predict_proba")[:, 1]

UMBRAL_TFIDF = umbral_para_precision(y_ora, prob_tfidf_des, 0.90)
pipe_tfidf.fit(X_ora, y_ora)
matriz_tfidf = pipe_tfidf.named_steps["tfidfvectorizer"].transform(X_ora)
print(f"matriz TF-IDF: {matriz_tfidf.shape[0]} oraciones x {matriz_tfidf.shape[1]} terminos")
print(f"umbral para precision 0.90: {UMBRAL_TFIDF:.3f}")

matriz TF-IDF: 2095 oraciones x 10058 terminos
umbral para precision 0.90: 0.484


In [44]:
# --- 2.5 Modelo B: embeddings ---
# El modelo de oraciones del TP2. Tiene que ser multilingue porque las sinopsis
# estan en castellano, y le pasamos el texto natural: sin minusculizar ni sacar
# acentos, que es la informacion que el mecanismo de atencion aprovecha.
from sentence_transformers import SentenceTransformer

MODELO_EMB = "distiluse-base-multilingual-cased-v1"   # el del TP2
modelo_emb = SentenceTransformer(MODELO_EMB)
print(f"{MODELO_EMB}: {modelo_emb.get_sentence_embedding_dimension()} dimensiones,"
      f" max_seq_length={modelo_emb.max_seq_length}")

_cache_emb = {}


def vectorizar(textos):
    """Embeddings con cache en memoria.

    El 2.7 vuelve a pedir los vectores de las mismas oraciones al aplicar el
    detector al catalogo completo; codificar dos veces es el costo mas alto
    del notebook.
    """
    faltan = [t for t in dict.fromkeys(textos) if t not in _cache_emb]
    if faltan:
        nuevos = modelo_emb.encode(faltan, batch_size=64, normalize_embeddings=True,
                                   show_progress_bar=False)
        _cache_emb.update(zip(faltan, nuevos))
    return np.vstack([_cache_emb[t] for t in textos])


Xe_ora = vectorizar(X_ora)
print(f"matriz de embeddings: {Xe_ora.shape[0]} oraciones x {Xe_ora.shape[1]} dimensiones")

# Truncamiento: el TP2 lo verifico sobre sinopsis completas, donde pesaba. Sobre
# oraciones casi no hay, que es otra ventaja de bajar el nivel de analisis.
largos = [len(modelo_emb.tokenizer.encode(t, truncation=False)) for t in X_ora]
truncadas = sum(1 for n in largos if n > modelo_emb.max_seq_length)
print(f"oraciones que superan {modelo_emb.max_seq_length} tokens: {truncadas}"
      f" ({truncadas/len(X_ora):.1%}); mediana {int(np.median(largos))} tokens")

lr_emb = LogisticRegression(**CFG_LR)
prob_emb_des = cross_val_predict(
    lr_emb, Xe_ora, y_ora, groups=g_ora, cv=GroupKFold(n_splits=5),
    method="predict_proba")[:, 1]
UMBRAL_EMB = umbral_para_precision(y_ora, prob_emb_des, 0.90)
lr_emb.fit(Xe_ora, y_ora)
print(f"umbral para precision 0.90: {UMBRAL_EMB:.3f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:  15%|█▌        | 15/100 [00:00<00:00, 100.64it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 413.00it/s]

C:\Users\Juli\AppData\Local\Temp\ipykernel_4260\3942265855.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"{MODELO_EMB}: {modelo_emb.get_sentence_embedding_dimension()} dimensiones,"


distiluse-base-multilingual-cased-v1: 512 dimensiones, max_seq_length=128


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (226 > 128). Running this sequence through the model will result in indexing errors


matriz de embeddings: 2095 oraciones x 512 dimensiones


oraciones que superan 128 tokens: 11 (0.5%); mediana 30 tokens
umbral para precision 0.90: 0.515


### Por qué no sirve el umbral 0.5

El enunciado lo advierte y acá se ve: las dos probabilidades no son comparables.

In [45]:
# --- 2.5 El 0.5 no es el mismo punto en los dos modelos ---
print(f"{'modelo':12s} {'umbral':>7s} {'P':>5s} {'R':>5s} {'marcadas':>9s}")
for nombre, prob, u in [("TF-IDF", prob_tfidf_des, UMBRAL_TFIDF),
                        ("embeddings", prob_emb_des, UMBRAL_EMB)]:
    for etiqueta, umb in [("elegido", u), ("0.5 fijo", 0.5)]:
        m = prob >= umb
        print(f"{nombre:12s} {umb:7.3f} {precision_score(y_ora, m, zero_division=0):5.2f}"
              f" {recall_score(y_ora, m):5.2f} {int(m.sum()):9d}   <- {etiqueta}")

modelo        umbral     P     R  marcadas
TF-IDF         0.484  0.91  0.90        87   <- elegido
TF-IDF         0.500  0.94  0.90        84   <- 0.5 fijo
embeddings     0.515  0.91  0.99        96   <- elegido
embeddings     0.500  0.90  0.99        97   <- 0.5 fijo


Con el umbral elegido los dos quedan en precisión **0.91** sobre desarrollo, que es la
condición que pide el enunciado para que la comparación sea justa: **a igual precisión**. Ahí
los recalls ya se separan —0.90 contra 0.99— y recién entonces la diferencia significa algo.

El 0.5 por defecto no es el mismo punto de operación en los dos modelos, así que compararlos
ahí mide la calibración de las probabilidades y no la calidad de la representación.

In [46]:
# --- 2.5 Aplicar los dos detectores a prueba ---
# La regla del enunciado: una sinopsis es sospechosa si ALGUNA de sus oraciones
# supera el umbral.
def detector_oraciones(texto, cual):
    ora = oraciones(texto)
    if not ora:
        return 0.0
    if cual == "tfidf":
        return float(pipe_tfidf.predict_proba(ora)[:, 1].max())
    return float(lr_emb.predict_proba(vectorizar(ora))[:, 1].max())


prob_tfidf_libro = prueba["synopsis"].map(lambda s: detector_oraciones(s, "tfidf"))
prob_emb_libro = prueba["synopsis"].map(lambda s: detector_oraciones(s, "emb"))
marca_tfidf = prob_tfidf_libro >= UMBRAL_TFIDF
marca_emb = prob_emb_libro >= UMBRAL_EMB

print("=== prueba, a igual precision en desarrollo ===")
medir("2.5 TF-IDF", marca_tfidf, prueba)
medir("2.5 embeddings", marca_emb, prueba)
medir("2.5 TF-IDF OR embeddings", marca_tfidf | marca_emb, prueba, registrar=False)

es_cont_pru = prueba["contaminado"] == 1
print(f"\ncontaminados que encuentran los dos: {int((marca_tfidf & marca_emb & es_cont_pru).sum())}")
print(f"  solo TF-IDF:     {int((marca_tfidf & ~marca_emb & es_cont_pru).sum())}")
print(f"  solo embeddings: {int((marca_emb & ~marca_tfidf & es_cont_pru).sum())}")

=== prueba, a igual precision en desarrollo ===
  2.5 TF-IDF                                   P=1.00 R=0.93  VP= 28 FP=  0 marca= 28
  2.5 embeddings                               P=0.90 R=0.90  VP= 27 FP=  3 marca= 30
  2.5 TF-IDF OR embeddings                     P=0.91 R=0.97  VP= 29 FP=  3 marca= 32

contaminados que encuentran los dos: 26
  solo TF-IDF:     2
  solo embeddings: 1


**Sobre prueba gana TF-IDF**: precisión 1.00 y recall 0.93, contra 0.90 y 0.90 de los
embeddings. Si la decisión se tomara acá, elegiríamos TF-IDF.

Vale la pena mirar los dos que TF-IDF no encuentra, porque los dos fallan por el mismo motivo
y no es el que uno esperaría.

In [47]:
# --- 2.5 Los contaminados que el clasificador no marca ---
for _, f in prueba[es_cont_pru & ~marca_tfidf].iterrows():
    print(f"\n  {f['slug']}  (prob {prob_tfidf_libro[f.name]:.2f})")
    print(f"    fragmento insertado: {f['fragmento'][:90]!r}")
    # Como quedo ese fragmento al partir la sinopsis en oraciones
    absorbida = next((o for o in oraciones(f["synopsis"]) if f["fragmento"][:30] in o), None)
    if absorbida:
        print(f"    quedo dentro de una oracion de {len(absorbida.split())} palabras:")
        print(f"      {absorbida[:150]!r}")


  como-el-rio-que-fluye  (prob 0.44)
    fragmento insertado: 'Pedí tu copia escribiendo a ventas@tomozona.test'
    quedo dentro de una oracion de 40 palabras:
      'Pedí tu copia escribiendo a ventas@tomozona.test Apreciaciones estéticas, observaciones sobre la naturaleza humana, la creación literaria, la historia'

  un-grito-desesperado  (prob 0.31)
    fragmento insertado: 'Link en nuestro sitio: h t t p s : / / lectortotal [.] example'
    quedo dentro de una oracion de 43 palabras:
      'Link en nuestro sitio: h t t p s : / / lectortotal [.] example Cuando una familia muestra síntomas de violencia y destrucción, no es posible seguir fi'


Los dos fragmentos **quedaron pegados al principio de la sinopsis sin punto que los separe**,
así que el partidor no los aísla: terminan dentro de una oración de cuarenta palabras. Es la
dilución del documento completo otra vez, en miniatura.

Pasa en 17 de las 88 oraciones positivas de desarrollo: existen en el entrenamiento como
oración suelta, pero al aplicar el detector aparecen embebidas en una oración larga. Es una
limitación del partidor, no del clasificador, y conviene declararla.

Los dos, además, los marca la regex: uno es un correo visible (2.1) y el otro una URL con
letras espaciadas y signos entre corchetes (2.2). **Los detectores son complementarios**, que
es justamente la apuesta del 2.7.

### La medición que invierte la conclusión

Queda la pregunta más importante del apartado, y la que decide qué usamos en el 2.7: *en el
catálogo hay mensajes que no están en la muestra; ¿qué representación generaliza mejor a una
frase nueva con el mismo sentido?*

Prueba **no puede contestarla**, y conviene ver por qué antes de confiar en el número de
arriba. Los 100 fragmentos de la muestra no son 100 frases distintas: son unas pocas plantillas
con la marca cambiada. Las agrupamos por su armazón de palabras.

In [48]:
# --- 2.5 Tipo de contaminacion de cada fragmento ---
# La muestra no trae el tipo, asi que lo derivamos con los mismos rasgos que ya
# caracterizamos en el 2.1 y 2.2. Es una etiqueta NUESTRA, no de la catedra: sirve
# para ver donde gana cada representacion, no para medir contra la clave oculta.
sinopsis_del_catalogo = {}
for slug, s in zip(crudo["slug"], crudo["synopsis"]):
    sinopsis_del_catalogo.setdefault(re.sub(r"\s+", " ", str(s)).strip(), []).append(slug)


def tipo_de_contaminacion(slug, sinopsis, fragmento):
    """Un tipo por fragmento, en orden de prioridad.

    El orden importa: un fragmento puede tener a la vez un enlace visible y
    caracteres ocultos, y queremos contarlo una sola vez, en la categoria que
    describe que necesita el detector para encontrarlo.
    """
    limpia = re.sub(r"\s+", " ", sinopsis.replace(fragmento, " ")).strip()
    if any(otro != slug for otro in sinopsis_del_catalogo.get(limpia, [])):
        return "copia de sinopsis"
    if RE_PROTOCOLO.search(fragmento) or RE_CORREO.search(fragmento) \
            or RE_TELEFONO.search(fragmento) or RE_DOMINIO.search(fragmento):
        return "enlace visible"
    if tiene_invisible(fragmento) or tiene_homoglifo(fragmento):
        return "enlace con caracteres ocultos"
    if RE_SUELTAS.search(fragmento) or any(
            re.search(patron, fragmento, re.IGNORECASE) for patron, _ in PALABRAS_SIGNO):
        return "enlace disfrazado con texto"
    return "promocional sin enlace"


muestra_toda = pd.concat([desarrollo, prueba]).copy()
muestra_toda["tipo"] = [
    tipo_de_contaminacion(sl, si, fr) if c == 1 else ""
    for sl, si, fr, c in zip(muestra_toda["slug"], muestra_toda["synopsis"],
                             muestra_toda["fragmento"], muestra_toda["contaminado"])
]
print(muestra_toda.loc[muestra_toda["contaminado"] == 1, "tipo"]
      .value_counts().rename("fragmentos").to_string())

tipo
promocional sin enlace           37
enlace visible                   33
enlace disfrazado con texto      13
copia de sinopsis                11
enlace con caracteres ocultos     6


In [49]:
# --- 2.5 Cuantas frases distintas hay en realidad ---
def esqueleto(fragmento):
    """El fragmento sin lo que cambia entre copias: marca, enlace, numeros, signos.

    Lo que queda son las palabras de la plantilla. Dos fragmentos con el mismo
    esqueleto son la misma frase promocional con otra marca encima.
    """
    t = unicodedata.normalize("NFKD", fragmento.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"https?://\S+|www\.\S+|\S+@\S+", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    return " ".join(t.split())


contaminados = muestra_toda[muestra_toda["contaminado"] == 1].copy()

# Dos esqueletos son la misma plantilla si comparten mas del 70% de las palabras.
plantilla_de, representantes = {}, []
for i, e in enumerate(contaminados["fragmento"].map(esqueleto)):
    for j, rep in enumerate(representantes):
        if fuzz.token_sort_ratio(e, rep) >= 70:
            plantilla_de[i] = j
            break
    else:
        plantilla_de[i] = len(representantes)
        representantes.append(e)
contaminados["plantilla"] = [plantilla_de[i] for i in range(len(contaminados))]

print(f"fragmentos contaminados: {len(contaminados)}")
print(f"plantillas distintas:    {contaminados['plantilla'].nunique()}")

pl_des = set(contaminados[contaminados["slug"].isin(desarrollo["slug"])]["plantilla"])
pl_pru = set(contaminados[contaminados["slug"].isin(prueba["slug"])]["plantilla"])
en_prueba = contaminados[contaminados["slug"].isin(prueba["slug"])]
nuevos = en_prueba[en_prueba["plantilla"].isin(pl_pru - pl_des)]
print(f"\nplantillas que aparecen en prueba:        {len(pl_pru)}")
print(f"  de esas, NO vistas en desarrollo:       {len(pl_pru - pl_des)}")
print(f"  contaminados de prueba con frase nueva: {len(nuevos)} de {len(en_prueba)}")

fragmentos contaminados: 100
plantillas distintas:    30

plantillas que aparecen en prueba:        21
  de esas, NO vistas en desarrollo:       1
  contaminados de prueba con frase nueva: 1 de 30


Ahí está el problema: **29 de los 30 contaminados de prueba usan una plantilla que el modelo ya
vio entrenando.** Prueba mide memorización, no generalización. El recall 0.93 de TF-IDF es real,
pero no dice nada sobre lo que va a pasar con los mensajes del catálogo que no están en la
muestra —que es lo que corrige la cátedra con su clave oculta.

Para medir lo que falta cambiamos el criterio de los pliegues: en vez de agrupar por libro,
**agrupamos por plantilla**. Así todas las oraciones de una misma frase caen en el mismo
pliegue y, cuando el modelo las evalúa, nunca la vio. Las dos representaciones se comparan
otra vez a igual precisión.

In [50]:
# --- 2.5 Generaliza a una frase que nunca vio? ---
corpus_todo = armar_corpus_oraciones(muestra_toda)
plantilla_por_slug = contaminados.set_index("slug")["plantilla"]

# Grupo de cada oracion: la plantilla si es positiva, el libro si es negativa.
grupo_plantilla = np.array([
    f"plantilla-{plantilla_por_slug[s]}" if yy == 1 else f"libro-{s}"
    for s, yy in zip(corpus_todo["slug"], corpus_todo["y"])
])

X_todo = corpus_todo["oracion"].tolist()
y_todo = corpus_todo["y"].values
Xe_todo = vectorizar(X_todo)
etiqueta_libro = muestra_toda.set_index("slug")["contaminado"]
tipos_contaminacion = sorted(muestra_toda.loc[muestra_toda["contaminado"] == 1, "tipo"].unique())

filas = []
for criterio, grupos in [("plantilla ya vista", corpus_todo["slug"].values),
                         ("frase nueva", grupo_plantilla)]:
    for nombre, estimador, datos in [
            ("TF-IDF", make_pipeline(TfidfVectorizer(), LogisticRegression(**CFG_LR)), X_todo),
            ("embeddings", LogisticRegression(**CFG_LR), Xe_todo)]:
        prob = cross_val_predict(estimador, datos, y_todo, groups=grupos,
                                 cv=GroupKFold(n_splits=5), method="predict_proba")[:, 1]
        umbral = umbral_para_precision(y_todo, prob, 0.90)
        # Nivel libro: alguna oracion supera el umbral
        maximo = pd.Series(prob, index=corpus_todo["slug"]).groupby(level=0).max()
        y_libro = etiqueta_libro.loc[maximo.index].values
        marcado = (maximo >= umbral).values
        fila = {
            "pliegues": criterio, "modelo": nombre, "umbral": round(umbral, 3),
            "precision": round(precision_score(y_libro, marcado, zero_division=0), 2),
            "recall": round(recall_score(y_libro, marcado), 2),
            "VP": int((marcado & (y_libro == 1)).sum()),
            "FP": int((marcado & (y_libro == 0)).sum()),
        }
        # Recall por tipo, que es donde se ve la diferencia
        for tipo in tipos_contaminacion:
            slugs = muestra_toda.loc[(muestra_toda["contaminado"] == 1)
                                     & (muestra_toda["tipo"] == tipo), "slug"]
            idx = maximo.index.get_indexer(slugs)
            fila[tipo] = f"{int(marcado[idx].sum())}/{len(slugs)}"
        filas.append(fila)

generalizacion = pd.DataFrame(filas)
print(generalizacion[["pliegues", "modelo", "umbral", "precision", "recall", "VP", "FP"]]
      .to_string(index=False))

          pliegues     modelo  umbral  precision  recall  VP  FP
plantilla ya vista     TF-IDF   0.332       0.89    1.00 100  12
plantilla ya vista embeddings   0.507       0.89    1.00 100  12
       frase nueva     TF-IDF   0.387       0.91    0.67  67   7
       frase nueva embeddings   0.485       0.90    0.89  89  10


In [51]:
# --- 2.5 Donde se abre la diferencia ---
print("recall por tipo de contaminacion (aciertos / total):\n")
print(generalizacion.set_index(["pliegues", "modelo"])[tipos_contaminacion].to_string())

recall por tipo de contaminacion (aciertos / total):

                              copia de sinopsis enlace con caracteres ocultos enlace disfrazado con texto enlace visible promocional sin enlace
pliegues           modelo                                                                                                                      
plantilla ya vista TF-IDF                 11/11                           6/6                       13/13          33/33                  37/37
                   embeddings             11/11                           6/6                       13/13          33/33                  37/37
frase nueva        TF-IDF                 11/11                           6/6                       10/13          31/33                   9/37
                   embeddings             11/11                           6/6                       10/13          33/33                  29/37


**Con la plantilla ya vista empatan en recall 1.00.** Con una frase nueva, TF-IDF cae a
**0.67** y los embeddings quedan en **0.89**: veintidós puntos de diferencia, a igual precisión.

Y la diferencia está concentrada en un solo tipo:

| tipo de contaminación | n | TF-IDF | embeddings |
|---|---|---|---|
| copia de sinopsis | 11 | 11 | 11 |
| enlace con caracteres ocultos | 6 | 6 | 6 |
| enlace disfrazado con texto | 13 | 10 | 10 |
| enlace visible | 33 | 31 | 33 |
| **promocional sin enlace** | **37** | **9** | **29** |

Es exactamente lo que predice la teoría de la unidad, y conviene decirlo con sus palabras: la
tabla comparativa de la sección 3 anota que TF-IDF «no reconoce sinónimos a menos que aparezcan
explícitamente» y «no captura contexto o significado semántico», mientras los embeddings
«pueden reconocer y relacionar palabras semánticamente similares».

Acá se ve por qué eso importa en la práctica:

- Donde hay un **enlace**, la frase nueva comparte cadenas literales con las vistas —`gratis`,
  `descargá`, `example`, `.test`— y TF-IDF las encuentra igual. Pierde solo 2 de 33.
- Donde **no hay enlace**, la frase nueva no comparte casi ninguna palabra con las de
  entrenamiento: cambia la marca, el verbo y la promesa. TF-IDF se queda en 9 de 37, porque lo
  único que aprendió fueron términos literales. Los embeddings llegan a 29, porque *«conseguí
  este título en la app X»* y *«leé miles de libros sin publicidad»* caen cerca en el espacio
  vectorial aunque no compartan vocabulario.

**Esto invierte la conclusión de prueba**, y es el resultado más importante del apartado: el
conjunto de prueba dice que conviene TF-IDF, y se equivoca, porque mide un escenario que no es
el del catálogo.

### Respuestas del 2.5

**¿Por qué conviene clasificar oraciones y no la sinopsis completa?** Porque el fragmento es
el 7 % del texto. Medido: el clasificador de sinopsis completa no marca ningún libro de
prueba, y aun eligiéndole el umbral sobre prueba no pasa de precisión 0.59. Por oración, 1.00
de precisión con 0.93 de recall.

**¿Qué representación detecta mejor los libros contaminados de prueba?** TF-IDF: precisión
1.00 y recall 0.93 contra 0.90 y 0.90. Pero prueba no es el escenario del catálogo, y con
frase nueva se da vuelta: 0.67 contra 0.89.

**¿En qué tipos de contaminación gana cada una?** Empatan en todo lo que tiene un enlace, por
disimulado que esté, y en las copias de sinopsis. La única diferencia real está en el
promocional sin enlace: 9 de 37 contra 29 de 37. Es, además, el tipo más numeroso.

**¿Cuál debería generalizar mejor a frases nuevas, y cuál detecta mejor cadenas literales?**
Los embeddings generalizan y TF-IDF reconoce literales, y las dos mitades de la medición lo
confirman por separado: TF-IDF no pierde casi nada donde hay una URL, y pierde tres cuartos
donde no hay ninguna. Para el 2.7 el criterio es que **la combinación conserva las dos
ventajas**: sobre prueba la unión de los dos da recall 0.97 contra 0.93 del mejor solo.

**Una advertencia para el informe.** La precisión que informamos es optimista por construcción:
la muestra está enriquecida al 20 % de contaminados y el catálogo real tiene una proporción
mucho menor. Con la misma tasa de falsos positivos por libro limpio, la precisión sobre el
catálogo completo va a ser más baja que acá.

## 2.6 Encontrar libros copiados

Hay libros que no existen: copian la sinopsis de otro libro del catálogo y le agregan texto.
El paso 1 no los pudo tocar, porque solo elimina **sinopsis idénticas** y estas no lo son: el
texto agregado las vuelve distintas.

El enunciado pide buscar, para cada libro, su vecino más parecido por **similitud del coseno**
con `NearestNeighbors`, y revisar los pares con similitud alta y **autores distintos**.

In [52]:
# --- 2.6 Vecino mas cercano por coseno ---
# TF-IDF y no embeddings: la copia reusa el texto del original casi palabra por
# palabra, asi que la senial es lexica. Un espacio semantico pondria cerca dos
# sinopsis que hablan del mismo tema sin compartir una linea, que es justo lo que
# NO queremos marcar aca.
tfidf_catalogo = TfidfVectorizer()
X_catalogo = tfidf_catalogo.fit_transform(libros["sinopsis"])
print(f"matriz TF-IDF del catalogo depurado: {X_catalogo.shape}")

# n_neighbors=2 porque el vecino mas cercano de cada libro es el libro mismo.
vecinos = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X_catalogo)
distancia, indice_vecino = vecinos.kneighbors(X_catalogo)

similitud_vecino = 1 - distancia[:, 1]
posicion_vecino = indice_vecino[:, 1]
print(f"similitud con el vecino: mediana={np.median(similitud_vecino):.3f}"
      f" | percentil 99={np.quantile(similitud_vecino, 0.99):.3f}"
      f" | maxima={similitud_vecino.max():.3f}")
print(f"libros con vecino por encima de 0.85: {int((similitud_vecino >= 0.85).sum())}")

matriz TF-IDF del catalogo depurado: (8591, 68548)


similitud con el vecino: mediana=0.214 | percentil 99=0.962 | maxima=1.000
libros con vecino por encima de 0.85: 224


La mediana es **0.214**: dos libros cualesquiera del catálogo no se parecen. Pero el percentil
99 salta a **0.962**, así que hay una cola de pares que comparten casi todo el texto. Esa cola
es lo que hay que mirar.

In [53]:
# --- 2.6 Armar la tabla de pares ---
autores_por_libro = libros["authors"].tolist()
slugs = libros["slug"].values
largos = libros["sinopsis"].str.len().values

vecindad = pd.DataFrame({
    "slug": slugs,
    "titulo": libros["title"].values,
    "slug_vecino": slugs[posicion_vecino],
    "titulo_vecino": libros["title"].values[posicion_vecino],
    "similitud": similitud_vecino.round(3),
    "largo": largos,
    "largo_vecino": largos[posicion_vecino],
    # Comparten algun autor? Las series legitimas comparten texto, pero en
    # general tienen el mismo autor; es el filtro que pide el enunciado.
    "mismo_autor": [bool(set(autores_por_libro[i]) & set(autores_por_libro[j]))
                    for i, j in enumerate(posicion_vecino)],
})

alto = vecindad["similitud"] >= 0.85
print(f"pares con similitud >= 0.85:        {int(alto.sum())}")
print(f"  de ellos, con autor compartido:   {int((alto & vecindad['mismo_autor']).sum())}")
print(f"  de ellos, con autores distintos:  {int((alto & ~vecindad['mismo_autor']).sum())}")

pares con similitud >= 0.85:        224
  de ellos, con autor compartido:   99
  de ellos, con autores distintos:  125


### Las 11 copias que podemos verificar

Antes de elegir un umbral conviene tener con qué comparar. En el 2.5 etiquetamos como *copia
de sinopsis* a los contaminados cuya sinopsis, **sacándole el fragmento**, coincide exacta con
la de otro libro del catálogo. Esas 11 son copias confirmadas, no sospechas: sabemos cuál es
la copia y cuál el original.

In [54]:
# --- 2.6 Las copias confirmadas y su original ---
copias_confirmadas = {}
for slug, sin, frag in zip(muestra_toda["slug"], muestra_toda["synopsis"],
                           muestra_toda["fragmento"]):
    if not frag:
        continue
    sin_fragmento = re.sub(r"\s+", " ", sin.replace(frag, " ")).strip()
    duenos = [o for o in sinopsis_del_catalogo.get(sin_fragmento, []) if o != slug]
    if duenos:
        copias_confirmadas[slug] = duenos[0]

print(f"copias confirmadas: {len(copias_confirmadas)}")

# El vecino que encontro el coseno es el original correcto?
vecino_de = dict(zip(vecindad["slug"], vecindad["slug_vecino"]))
aciertos = sum(1 for copia, original in copias_confirmadas.items()
               if vecino_de.get(copia) == original)
print(f"  su vecino mas cercano es el original correcto: {aciertos} de {len(copias_confirmadas)}")

en_copias = vecindad["slug"].isin(copias_confirmadas)
print(f"\nsimilitud de esas {int(en_copias.sum())} copias con su original:"
      f" minima={vecindad.loc[en_copias, 'similitud'].min():.3f}"
      f" mediana={vecindad.loc[en_copias, 'similitud'].median():.3f}")
print(f"la copia es mas larga que el original:"
      f" {int((vecindad.loc[en_copias, 'largo'] > vecindad.loc[en_copias, 'largo_vecino']).sum())}"
      f" de {int(en_copias.sum())}")

copias confirmadas: 11
  su vecino mas cercano es el original correcto: 11 de 11

similitud de esas 11 copias con su original: minima=0.859 mediana=0.955
la copia es mas larga que el original: 11 de 11


### El problema que no es obvio: el detector marca los dos lados

El coseno es simétrico. Si la copia aparece junto a su original, **el original también aparece
junto a la copia**, con la misma similitud y los mismos autores distintos. Y el original es un
libro legítimo que no hay que sacar del catálogo.

Es la pregunta del enunciado —*¿cuál de los dos es la copia?*— y acá se vuelve una necesidad
práctica, no una curiosidad: sin resolverla, por cada copia que eliminamos nos llevamos un
libro bueno.

La respuesta sale de cómo se fabricó la copia: es el original **más** texto agregado. Entonces
la copia es la más larga de las dos, y además las palabras del original tienen que reaparecer
casi todas dentro de ella.

In [55]:
# --- 2.6 Jaccard y contencion ---
def conjunto_palabras(texto):
    return set(re.findall(r"\w+", texto.lower()))


def jaccard(a, b):
    """Similitud de Jaccard sobre conjuntos de palabras, como sugiere el enunciado."""
    pa, pb = conjunto_palabras(a), conjunto_palabras(b)
    return len(pa & pb) / len(pa | pb) if (pa | pb) else 0.0


def contencion(texto_largo, texto_corto):
    """Que proporcion de las palabras del corto esta dentro del largo.

    Es asimetrica, y por eso sirve donde Jaccard no alcanza: si la copia es el
    original mas texto agregado, casi todas las palabras del original reaparecen
    en la copia, aunque el texto agregado le baje el Jaccard.
    """
    pl, pc = conjunto_palabras(texto_largo), conjunto_palabras(texto_corto)
    return len(pl & pc) / len(pc) if pc else 0.0


sinopsis_por_slug = dict(zip(libros["slug"], libros["sinopsis"]))

# Solo los candidatos: autor distinto y el lado mas largo del par.
candidatos = vecindad[~vecindad["mismo_autor"]
                      & (vecindad["largo"] > vecindad["largo_vecino"])].copy()
candidatos["jaccard"] = [round(jaccard(sinopsis_por_slug[a], sinopsis_por_slug[b]), 3)
                         for a, b in zip(candidatos["slug"], candidatos["slug_vecino"])]
candidatos["contencion"] = [round(contencion(sinopsis_por_slug[a], sinopsis_por_slug[b]), 3)
                            for a, b in zip(candidatos["slug"], candidatos["slug_vecino"])]
candidatos["es_copia_confirmada"] = candidatos["slug"].isin(copias_confirmadas)
candidatos["etiqueta"] = candidatos["slug"].map(
    dict(zip(muestra_toda["slug"], muestra_toda["contaminado"])))

confirmadas = candidatos[candidatos["es_copia_confirmada"]]
print("las 11 copias confirmadas, medidas contra su original:")
print(f"  Jaccard:    minimo={confirmadas['jaccard'].min():.3f}"
      f" maximo={confirmadas['jaccard'].max():.3f}")
print(f"  contencion: minimo={confirmadas['contencion'].min():.3f}"
      f" maximo={confirmadas['contencion'].max():.3f}")

las 11 copias confirmadas, medidas contra su original:
  Jaccard:    minimo=0.738 maximo=0.954
  contencion: minimo=1.000 maximo=1.000


**La contención es exactamente 1.000 en las once.** No queda ni una palabra del original
afuera de la copia: el texto se reusó entero. El Jaccard, en cambio, se reparte entre 0.738 y
0.954 según cuánto texto se agregó, así que separa peor. Lo usamos igual porque es lo que
sugiere el enunciado, pero el que decide es la contención.

In [56]:
# --- 2.6 Eleccion del umbral de coseno ---
# Lo barremos sobre los candidatos que tienen etiqueta en la muestra: son los
# unicos donde podemos contar aciertos y falsos positivos.
con_etiqueta = candidatos[candidatos["etiqueta"].notna()]
print(f"candidatos con etiqueta: {len(con_etiqueta)}\n")
print(f"{'coseno':>7} {'marca':>6} {'VP':>4} {'FP':>4} {'copias':>7}")
for umbral in [0.75, 0.78, 0.80, 0.82, 0.85, 0.87, 0.90]:
    sel = con_etiqueta[con_etiqueta["similitud"] >= umbral]
    print(f"{umbral:7.2f} {len(sel):6d} {int((sel['etiqueta'] == 1).sum()):4d}"
          f" {int((sel['etiqueta'] == 0).sum()):4d}"
          f" {int(sel['es_copia_confirmada'].sum()):7d}")

candidatos con etiqueta: 109

 coseno  marca   VP   FP  copias
   0.75     12   11    1      11
   0.78     12   11    1      11
   0.80     11   11    0      11
   0.82     11   11    0      11
   0.85     11   11    0      11
   0.87     10   10    0      10
   0.90      9    9    0       9


Hay **meseta entre 0.80 y 0.85**: los tres valores dan 11 marcados, 11 aciertos, ningún falso
positivo y las 11 copias. Tomamos **0.85**, el extremo conservador de la meseta.

Por debajo de 0.80 entra un falso positivo, y vale mirarlo porque explica para qué sirve la
contención: es un par con coseno 0.789 cuya contención es **0.783**, no 1.000. Comparte buena
parte del vocabulario sin ser una copia literal. La contención lo descarta sola, sin depender
del umbral de coseno.

In [57]:
# --- 2.6 La regla completa ---
UMBRAL_COSENO_COPIA = 0.85   # meseta entre 0.80 y 0.85; tomamos el extremo conservador
UMBRAL_CONTENCION = 0.95     # las copias confirmadas dan 1.00; el falso positivo, 0.78

copias_detectadas = candidatos[
    (candidatos["similitud"] >= UMBRAL_COSENO_COPIA)
    & (candidatos["contencion"] >= UMBRAL_CONTENCION)
].copy()

print(f"libros marcados como copia en el catalogo depurado: {len(copias_detectadas)}")
etiquetados = copias_detectadas[copias_detectadas["etiqueta"].notna()]
print(f"  con etiqueta en la muestra: {len(etiquetados)}"
      f"  -> contaminados {int((etiquetados['etiqueta'] == 1).sum())}"
      f" | limpios {int((etiquetados['etiqueta'] == 0).sum())}")
print(f"  sin etiqueta (resto del catalogo): {int(copias_detectadas['etiqueta'].isna().sum())}")

# Lo que la regla tiene que garantizar: no marcar ningun original.
originales = set(copias_confirmadas.values())
print(f"\noriginales marcados por error: {len(set(copias_detectadas['slug']) & originales)}")

print("\n=== 8 pares detectados, de mayor a menor coseno ===")
print(copias_detectadas.sort_values("similitud", ascending=False)[
    ["titulo", "titulo_vecino", "similitud", "jaccard", "contencion"]
].head(8).to_string(index=False))

libros marcados como copia en el catalogo depurado: 62
  con etiqueta en la muestra: 11  -> contaminados 11 | limpios 0
  sin etiqueta (resto del catalogo): 51

originales marcados por error: 0

=== 8 pares detectados, de mayor a menor coseno ===
                                       titulo                             titulo_vecino  similitud  jaccard  contencion
                      Los hijos de la promesa                         Elvira de Hidalgo      0.988    0.954         1.0
                           La última tormenta La pequeña comunista que no sonreía nunca      0.988    0.944         1.0
                                 Bajo la isla                            Trás-os-Montes      0.985    0.954         1.0
Richard P. Feynman. La física de las palabras                 La física de las palabras      0.984    0.926         1.0
                           Crónica del bosque                       Cotufas en el golfo      0.977    0.922         1.0
                           La últ

In [58]:
# --- 2.6 Evaluacion sobre prueba, para la tabla del 2.7 ---
marca_copia = prueba["slug"].isin(set(copias_detectadas["slug"]))
medir("2.6 copia de sinopsis", marca_copia, prueba)

# El tipo de contaminacion vive en muestra_toda y no en prueba, asi que cruzamos
# por slug y no por indice.
copias_en_prueba = set(muestra_toda.loc[
    (muestra_toda["contaminado"] == 1)
    & (muestra_toda["tipo"] == "copia de sinopsis"), "slug"]) & set(prueba["slug"])
encontradas = copias_en_prueba & set(copias_detectadas["slug"])
print(f"'copia de sinopsis' en prueba: {len(copias_en_prueba)}"
      f" | encontradas: {len(encontradas)}")

  2.6 copia de sinopsis                        P=1.00 R=0.10  VP=  3 FP=  0 marca=  3
'copia de sinopsis' en prueba: 3 | encontradas: 3


### Respuestas del 2.6

**¿Cuál de los dos es la copia?** La más larga, porque la copia es el original más el texto
agregado. Se verifica con la contención: las palabras del original reaparecen **todas** dentro
de la copia (contención 1.000 en las 11 confirmadas), y no al revés. Sin esta regla el
detector marcaría los dos lados del par y eliminaría un libro legítimo por cada copia.

**¿Cómo se separan las copias de las series legítimas?** Con el filtro de autor que pide el
enunciado. De los pares con coseno ≥ 0.85, la mayoría comparte autor: son volúmenes de una
misma obra que el sitio describe con un solo texto, lo mismo que ya habíamos visto en el paso
1. Esos no se tocan.

**Resultado.** Precisión **1.00** y recall **0.10** sobre prueba: encuentra las 3 copias que
hay y ningún falso positivo. El recall bajo no es una falla: este detector ataca **un solo
tipo** de contaminación, que son 3 de los 30 contaminados de prueba. En su tipo acierta 3 de
3, y en desarrollo 8 de 8.

**Lo que aporta al catálogo completo.** Marca **62 libros**, de los cuales solo 11 están en la
muestra etiquetada. Los otros **51** son copias que la muestra no contiene, y son justamente
las que la cátedra corrige con su clave oculta. Es el detector con el mejor precio del paso 2:
cero falsos positivos conocidos y 51 libros que ningún otro apartado alcanza.

**Una salvedad honesta.** Entre esos 62 hay dos clases de pares. Unos son libros inventados
—títulos y autores sintéticos— que copiaron una obra real. Otros son duplicados legítimos del
catálogo que el paso 1 no unió porque su bloqueo por autor no los comparó, como un mismo libro
cargado dos veces con el nombre del autor dentro del título. Los dos casos conviene quitarlos,
pero el motivo no es el mismo, y al escribir `sospechosos.csv` en el 2.7 hay que recordar que
la etiqueta «copia» describe bien a los primeros y solo aproximadamente a los segundos.

## 2.7 Combinar y entregar

Lo que pide el enunciado:

1. Combinar los detectores — un libro es sospechoso si **al menos uno** lo marca.
2. Presentar la tabla de precisión y recall de cada detector y de la combinación, sobre prueba.
3. Aplicar el sistema al catálogo completo y guardar `sospechosos.csv` con `slug` y `motivo`.
4. Quitar esos libros y guardar `catalogo_limpio.csv`.

Y la advertencia que ordena todo el apartado: **marcar todo no sirve**. Cada libro legítimo que
eliminamos es un libro que el recomendador nunca va a poder sugerir, y el paso 3 valora sobre
el catálogo limpio.

In [59]:
# --- 2.7 La tabla de prueba, detector por detector ---
# 'resultados' viene cargandose desde el 2.1: cada detector dejo su fila al medirse
# sobre prueba. El 2.4 no esta porque lo excluimos a proposito (precision 0.07).
tabla = pd.DataFrame(resultados)

# La combinacion que llevamos al catalogo usa el 2.5 de TF-IDF. La celda de abajo
# justifica esa eleccion; la decide la medicion sobre la muestra completa, no esta.
combinada = marca_22 | marca_23 | marca_tfidf | marca_copia
con_embeddings = marca_22 | marca_23 | marca_emb | marca_copia

print("=== cada detector sobre prueba ===")
print(tabla.to_string(index=False))

print("")
print("=== combinaciones ===")
medir("COMBINADA (2.2 + 2.3 + 2.5 TF-IDF + 2.6)", combinada, prueba, registrar=False)
medir("  la misma con embeddings en el 2.5", con_embeddings, prueba, registrar=False)
medir("  con los dos modelos del 2.5", combinada | marca_emb, prueba, registrar=False)

=== cada detector sobre prueba ===
                                 detector  precision  recall  marca  VP  FP
linea de base: regex minima del enunciado       1.00    0.13      4   4   0
           2.1 regex propia (texto crudo)       1.00    0.40     12  12   0
                2.2 regex + normalizacion       1.00    0.63     19  19   0
                   2.3 idioma por oracion       0.89    0.27      9   8   1
         2.2 OR 2.3 (combinacion parcial)       0.96    0.80     25  24   1
                               2.5 TF-IDF       1.00    0.93     28  28   0
                           2.5 embeddings       0.90    0.90     30  27   3
                    2.6 copia de sinopsis       1.00    0.10      3   3   0

=== combinaciones ===
  COMBINADA (2.2 + 2.3 + 2.5 TF-IDF + 2.6)     P=0.97 R=1.00  VP= 30 FP=  1 marca= 31
    la misma con embeddings en el 2.5          P=0.88 R=1.00  VP= 30 FP=  4 marca= 34
    con los dos modelos del 2.5                P=0.88 R=1.00  VP= 30 FP=  4 marca= 34


{'detector': '  con los dos modelos del 2.5',
 'precision': 0.88,
 'recall': 1.0,
 'marca': 34,
 'VP': 30,
 'FP': 4}

In [60]:
# --- 2.7 Que aporta cada detector a la combinacion ---
partes = {"2.2 enlace": marca_22, "2.3 idioma": marca_23,
          "2.5 promocional": marca_tfidf, "2.6 copia": marca_copia}
es_contaminado = prueba["contaminado"] == 1

print("contaminados que encuentra SOLO ese detector:")
for nombre, marca in partes.items():
    otros = pd.Series(False, index=prueba.index)
    for otro_nombre, otra in partes.items():
        if otro_nombre != nombre:
            otros |= pd.Series(otra).reindex(prueba.index).fillna(False).astype(bool)
    solo = pd.Series(marca).reindex(prueba.index).fillna(False).astype(bool) & ~otros
    print(f"  {nombre:18s} {int((solo & es_contaminado).sum())}")

print("")
print("recall de la combinacion por tipo de contaminacion:")
tipo_de = dict(zip(muestra_toda["slug"], muestra_toda["tipo"]))
for tipo in sorted({tipo_de.get(s, "") for s in prueba.loc[es_contaminado, "slug"]}):
    del_tipo = prueba["slug"].map(lambda s: tipo_de.get(s) == tipo) & es_contaminado
    print(f"  {tipo:32s} {int((combinada & del_tipo).sum())}/{int(del_tipo.sum())}")

print("")
print(f"contaminados que no encuentra ninguno: {int((~combinada & es_contaminado).sum())}")
for slug in prueba.loc[~combinada & es_contaminado, "slug"]:
    print(f"  {slug}  ({tipo_de.get(slug, '?')})")

contaminados que encuentra SOLO ese detector:
  2.2 enlace         2
  2.3 idioma         0
  2.5 promocional    5
  2.6 copia          0

recall de la combinacion por tipo de contaminacion:
  copia de sinopsis                3/3
  enlace con caracteres ocultos    3/3
  enlace disfrazado con texto      4/4
  enlace visible                   10/10
  promocional sin enlace           10/10

contaminados que no encuentra ninguno: 0


### Qué dice la tabla

**La combinación encuentra los 30 contaminados de prueba**, y los cinco tipos quedan completos:
10/10 enlace visible, 10/10 promocional sin enlace, 4/4 disfrazado con texto, 3/3 caracteres
ocultos y 3/3 copias. Ningún contaminado se escapa, con un solo falso positivo.

Y la combinación es lo que hace eso posible: **ningún detector solo pasa de recall 0.93**. El
mejor individual es el clasificador de oraciones; el 2.2 aporta 2 contaminados que al
clasificador se le escapan y el clasificador aporta 5 que la regex no ve. Son errores
independientes, y por eso sumarlos con un OR gana recall sin pagar casi nada de precisión.

Un cuidado al leer el aporte único: **el 2.3 y el 2.6 dan 0**, pero eso no los vuelve
inútiles. Sobre 30 contaminados el estadístico es demasiado chico, y en el catálogo completo el
2.3 marca 112 libros y el 2.6 otros 62, con una precisión que medimos en su tipo y es perfecta.
Lo que el 0 dice es que en *estos* 30 libros lo que ellos encuentran también lo encuentra otro.

### Qué modelo del 2.5 llevamos al catálogo

Sobre prueba las dos combinaciones alcanzan recall 1.00, así que prueba no distingue. Y son
solo 30 contaminados: un libro mueve el recall tres puntos. Para decidir usamos **los 500
libros etiquetados**, que es cinco veces más evidencia, y miramos las dos cosas que la cátedra
devuelve: cuántos contaminados encontramos y cuántos libros limpios nos llevamos de más.

In [61]:
# --- 2.7 La medicion que decide: las dos combinaciones sobre los 500 etiquetados ---
# Los detectores se aplican directo a las sinopsis de la muestra, sin pasar por
# el catalogo depurado: asi entran tambien los 3 libros que el paso 1 elimino.
sinopsis_muestra = muestra_toda["synopsis"].tolist()


def promocional(textos, cual):
    """Maxima probabilidad entre las oraciones de cada sinopsis."""
    salida = []
    for texto in textos:
        ora = oraciones(texto)
        if not ora:
            salida.append(0.0)
        elif cual == "tfidf":
            salida.append(float(pipe_tfidf.predict_proba(ora)[:, 1].max()))
        else:
            salida.append(float(lr_emb.predict_proba(vectorizar(ora))[:, 1].max()))
    return np.array(salida)


m_enlace = np.array([detector_normalizado(s) for s in sinopsis_muestra])
m_idioma = np.array([detector_idioma(s) for s in sinopsis_muestra])
m_copia = muestra_toda["slug"].isin(set(copias_detectadas["slug"])).values
m_tfidf = promocional(sinopsis_muestra, "tfidf") >= UMBRAL_TFIDF
m_emb = promocional(sinopsis_muestra, "emb") >= UMBRAL_EMB

y_muestra = muestra_toda["contaminado"].values
base = m_enlace | m_idioma | m_copia

print(f"sobre los {len(muestra_toda)} libros etiquetados"
      f" ({int((y_muestra == 1).sum())} contaminados):")
print(f"{'version del 2.5':18s} {'recall':>14s} {'falsos positivos':>20s} {'marca':>6s}")
for nombre, marca in [("TF-IDF", base | m_tfidf), ("embeddings", base | m_emb)]:
    vp = int((marca & (y_muestra == 1)).sum())
    fp = int((marca & (y_muestra == 0)).sum())
    limpios = int((y_muestra == 0).sum())
    print(f"{nombre:18s} {vp:6d}/{int((y_muestra == 1).sum()):<7d}"
          f" {fp:8d}/{limpios:<11d} {int(marca.sum()):6d}")

sobre los 500 libros etiquetados (100 contaminados):
version del 2.5            recall     falsos positivos  marca
TF-IDF                100/100            4/400            104
embeddings            100/100            8/400            108


**TF-IDF gana en las dos columnas**: encuentra más contaminados y se lleva menos libros
limpios. Parece contradecir al 2.5, donde los embeddings generalizaban mucho mejor a una frase
nueva (0.89 contra 0.67), y no lo contradice: lo pone en escala. Donde el clasificador TF-IDF
falla con una frase que nunca vio, casi siempre hay además un enlace, un idioma ajeno o una
copia que delata al libro, así que **los otros tres detectores cubren ese hueco**. Lo que no
cubre nadie son los falsos positivos de los embeddings, y esos sí se pagan.

Queda una limitación que conviene declarar: la muestra tiene 29 de sus 30 plantillas también
en desarrollo, así que **sobreestima a TF-IDF**. La ventaja medida es real, pero el margen
sobre los embeddings sería menor en un catálogo con más frases nuevas de las que vimos.

### Aplicar el sistema al catálogo completo

Los cuatro detectores corren directo sobre las 8591 sinopsis. Al llevar TF-IDF y no embeddings
se evita además el paso más caro del notebook: ya no hay que codificar las 53 mil oraciones del
catálogo con el modelo de oraciones, que en CPU tardaba media hora y ocupaba 100 MB en disco.

In [62]:
# --- 2.7 Oraciones del catalogo ---
oraciones_por_libro = [oraciones(s) for s in libros["sinopsis"]]
oraciones_catalogo = [o for lista in oraciones_por_libro for o in lista]
distintas = sorted(set(oraciones_catalogo))
print(f"oraciones del catalogo: {len(oraciones_catalogo)} | distintas: {len(distintas)}")

# El clasificador de oraciones se evalua una vez por oracion distinta y despues
# se reparte por libro: hay 1470 oraciones repetidas entre sinopsis.
prob_oracion = pipe_tfidf.predict_proba(distintas)[:, 1]
fila_de = {o: i for i, o in enumerate(distintas)}
prob_max_libro = np.array([
    max((prob_oracion[fila_de[o]] for o in lista), default=0.0)
    for lista in oraciones_por_libro
])
print(f"libros cuya oracion mas sospechosa pasa el umbral {UMBRAL_TFIDF:.3f}:"
      f" {int((prob_max_libro >= UMBRAL_TFIDF).sum())}")

oraciones del catalogo: 53177 | distintas: 51707


libros cuya oracion mas sospechosa pasa el umbral 0.484: 273


In [63]:
# --- 2.7 Los cuatro detectores sobre el catalogo completo ---
catalogo_marcas = pd.DataFrame({
    "slug": libros["slug"].values,
    "enlace": libros["sinopsis"].map(detector_normalizado).values,
    "idioma": libros["sinopsis"].map(detector_idioma).values,
    "promocional": prob_max_libro >= UMBRAL_TFIDF,
    "copia": libros["slug"].isin(set(copias_detectadas["slug"])).values,
})
catalogo_marcas["sospechoso"] = catalogo_marcas[
    ["enlace", "idioma", "promocional", "copia"]].any(axis=1)

print("libros marcados en el catalogo depurado, por detector:")
for columna in ["enlace", "idioma", "promocional", "copia"]:
    n = int(catalogo_marcas[columna].sum())
    print(f"  {columna:12s} {n:5d}  ({n/len(libros):5.1%})")
print(f"  {'TOTAL':12s} {int(catalogo_marcas['sospechoso'].sum()):5d}"
      f"  ({catalogo_marcas['sospechoso'].mean():5.1%})")
print("")
print(f"quedan limpios: {int((~catalogo_marcas['sospechoso']).sum())}")

libros marcados en el catalogo depurado, por detector:
  enlace         200  ( 2.3%)
  idioma         112  ( 1.3%)
  promocional    273  ( 3.2%)
  copia           62  ( 0.7%)
  TOTAL          407  ( 4.7%)

quedan limpios: 8184


In [64]:
# --- 2.7 sospechosos.csv y catalogo_limpio.csv ---
ARCHIVO_SOSPECHOSOS = "sospechosos.csv"
ARCHIVO_LIMPIO = "catalogo_limpio.csv"

# El motivo en palabras. Si lo marcan varios detectores los juntamos con '; ':
# el enunciado pide una columna, y perder esa informacion seria perder el unico
# rastro de por que cada libro salio.
NOMBRE_MOTIVO = {
    "enlace": "enlace o contacto en la sinopsis",
    "idioma": "oracion en otro idioma",
    "promocional": "texto promocional",
    "copia": "sinopsis copiada de otro libro",
}

motivos = [
    "; ".join(NOMBRE_MOTIVO[c] for c in ["enlace", "idioma", "promocional", "copia"]
              if fila[c])
    for _, fila in catalogo_marcas.iterrows()
]
catalogo_marcas["motivo"] = motivos

sospechosos = catalogo_marcas.loc[catalogo_marcas["sospechoso"], ["slug", "motivo"]]
sospechosos.to_csv(CARPETA_DATOS / ARCHIVO_SOSPECHOSOS, index=False, encoding="utf-8")
print(f"guardado: {CARPETA_DATOS / ARCHIVO_SOSPECHOSOS}  ({len(sospechosos)} libros)")

print("\nmotivos mas frecuentes:")
print(sospechosos["motivo"].value_counts().head(8).to_string())

limpios = set(catalogo_marcas.loc[~catalogo_marcas["sospechoso"], "slug"])
catalogo_limpio = libros[libros["slug"].isin(limpios)][
    ["slug", "title", "authors", "genres", "series", "series_number", "sinopsis"]
].copy()
# authors y genres como JSON, igual que en el paso 1: se recuperan con json.loads.
catalogo_limpio["authors"] = catalogo_limpio["authors"].map(
    lambda v: json.dumps(v, ensure_ascii=False))
catalogo_limpio["genres"] = catalogo_limpio["genres"].map(
    lambda v: json.dumps(v, ensure_ascii=False))
catalogo_limpio.to_csv(CARPETA_DATOS / ARCHIVO_LIMPIO, index=False, encoding="utf-8")
print(f"\nguardado: {CARPETA_DATOS / ARCHIVO_LIMPIO}  ({len(catalogo_limpio)} libros)")

# El conteo tiene que cerrar.
assert len(catalogo_limpio) + len(sospechosos) == len(libros), "el conteo no cierra"
print(f"verificado: {len(libros)} - {len(sospechosos)} = {len(catalogo_limpio)}")

guardado: ..\..\data\TP_3\sospechosos.csv  (407 libros)

motivos mas frecuentes:
motivo
enlace o contacto en la sinopsis; texto promocional                                    101
texto promocional                                                                       90
oracion en otro idioma                                                                  83
enlace o contacto en la sinopsis                                                        43
enlace o contacto en la sinopsis; texto promocional; sinopsis copiada de otro libro     37
texto promocional; sinopsis copiada de otro libro                                       17
oracion en otro idioma; texto promocional                                               14
enlace o contacto en la sinopsis; oracion en otro idioma; texto promocional             14



guardado: ..\..\data\TP_3\catalogo_limpio.csv  (8184 libros)
verificado: 8591 - 407 = 8184


In [65]:
# --- 2.7 Control: cuanto nos llevamos de la muestra etiquetada ---
# Es la unica parte del catalogo donde sabemos la verdad, asi que es la mejor
# estimacion de lo que le estamos haciendo al resto.
en_muestra = catalogo_marcas[catalogo_marcas["slug"].isin(set(muestra_toda["slug"]))].copy()
en_muestra["contaminado"] = en_muestra["slug"].map(
    dict(zip(muestra_toda["slug"], muestra_toda["contaminado"])))

cruce = pd.crosstab(en_muestra["contaminado"], en_muestra["sospechoso"],
                    rownames=["contaminado"], colnames=["marcado"])
print("sobre los 497 libros de la muestra que sobrevivieron al paso 1:")
print(cruce.to_string())

limpios_perdidos = int(((en_muestra["contaminado"] == 0) & en_muestra["sospechoso"]).sum())
total_limpios = int((en_muestra["contaminado"] == 0).sum())
print(f"\nlibros limpios que perdemos: {limpios_perdidos} de {total_limpios}"
      f" ({limpios_perdidos/total_limpios:.1%})")
print("Extrapolado al catalogo, es el orden de libros buenos que el recomendador")
print("no va a poder sugerir. Es el costo de la combinacion, y hay que declararlo.")

sobre los 497 libros de la muestra que sobrevivieron al paso 1:
marcado      False  True 
contaminado              
0              397      2
1                0     98

libros limpios que perdemos: 2 de 399 (0.5%)
Extrapolado al catalogo, es el orden de libros buenos que el recomendador
no va a poder sugerir. Es el costo de la combinacion, y hay que declararlo.


## Respuestas del enunciado para 2.1, 2.2 y 2.3

**¿Qué tipo de contaminación detecta bien la regex y cuál no?**

Detecta bien lo que deja una cadena literal reconocible: URLs con protocolo, dominios
desnudos, correos y teléfonos. Con la normalización suma los que la ocultan con caracteres
invisibles, homoglifos, letras espaciadas o signos escritos con palabras.

No detecta nada del texto promocional sin enlace, que es la porción más grande de la
contaminación. Ahí la regex no tiene a qué agarrarse y hace falta aprender de ejemplos (2.5).

**¿Qué recuperó la normalización?** Subió el recall sin tocar la precisión: los mismos
patrones, aplicados a un texto donde la ocultación ya fue deshecha. Es el detector con mejor
relación costo-beneficio del paso 2, porque reutiliza la regex ya escrita.

**El costo de un falso positivo.** Cada libro legítimo que marcamos sale del catálogo y deja
de poder recomendarse. En el paso 3 el lector valora sobre el catálogo limpio, así que un
falso positivo le saca un candidato antes de que llegue a verlo. Por eso preferimos patrones
conservadores —TLD en minúscula, lista corta de dominios reales— antes que ganar recall
marcando de más.

**¿Por qué el idioma por oración y no por sinopsis?** Porque la sinopsis completa da `es` en
los 350 casos de desarrollo: el mensaje insertado es una oración entre siete y queda diluido.
El mismo argumento, medido, vale para el clasificador del 2.5.

**¿Cómo se justificaron el mínimo de palabras y el umbral?** Con el barrido sobre desarrollo,
y mostrando que los valores elegidos están en una meseta y no al filo: `min_palabras` 8 y 10
dan el mismo resultado, y el umbral da lo mismo entre 0.90 y 0.99.

### Respuestas del 2.7

**¿Cuánto del catálogo queda marcado?** **407 libros de 8591, el 4,7 %**, y quedan **8184** en
`catalogo_limpio.csv`. Por detector: texto promocional 273, enlace o contacto 200, otro idioma
112, copia de sinopsis 62. Suman más de 407 porque muchos libros los marca más de uno, y por
eso el `motivo` los junta con `; ` en vez de quedarse con el primero.

**¿Cuánto cuesta en libros buenos?** Es la pregunta que el enunciado pone en el centro, y la
contestamos sobre la única parte del catálogo donde sabemos la verdad. De los 497 libros de la
muestra que sobrevivieron al paso 1, el sistema marca **98 de 98 contaminados** y se lleva
**2 libros limpios de 399: el 0,5 %**. Extrapolado al catálogo son unas cuatro decenas de
libros legítimos que el recomendador no va a poder sugerir. Es un costo real y hay que
declararlo, pero es el precio de no dejar pasar contaminación.

**¿Por qué TF-IDF y no embeddings en el clasificador?** Porque sobre los 500 etiquetados las
dos versiones encuentran los 100 contaminados, y la de embeddings se lleva **8 libros limpios
contra 4**. Sobre prueba la diferencia aparece como precisión 0.97 contra 0.88. Esto no
contradice al 2.5 —donde los embeddings generalizaban mucho mejor a una frase nueva, 0.89
contra 0.67— sino que lo pone en escala: donde el clasificador TF-IDF falla con una frase que
nunca vio, casi siempre hay además un enlace, un idioma ajeno o una copia que delata al libro,
así que **los otros tres detectores cubren ese hueco**. Los falsos positivos de los embeddings,
en cambio, no los cubre nadie.

**Lo que esta medición no puede ver.** La muestra tiene 29 de sus 30 plantillas también en
desarrollo, así que **sobreestima a TF-IDF**: en el catálogo hay mensajes con frases que no
vimos, y ahí el clasificador de embeddings tendría ventaja. No podemos medir cuántos son, y es
la limitación principal de todo el paso 2. Si la cátedra devuelve un recall por tipo bajo en
*promocional sin enlace*, ésta es la causa a mirar primero, y el cambio sería de una línea.

**Dos salvedades más para el informe.** La muestra está enriquecida al 20 % de contaminados,
bastante más que el catálogo, así que la precisión real va a ser más baja que la que medimos.
Y el 2.3 vio la parte de prueba más de una vez durante su calibración, lo que hace optimista su
número en particular.

## Cierre del paso 2

| apartado | qué aporta | estado |
|---|---|---|
| 2.1 Expresiones regulares | enlaces y contactos visibles | hecho |
| 2.2 Normalizar antes de buscar | enlaces disimulados | hecho |
| 2.3 Idioma por oración | mensajes en otro idioma | hecho |
| 2.4 NER | nada: explica, no detecta | hecho y **excluido** |
| 2.5 Clasificador de oraciones | texto promocional sin enlace | hecho |
| 2.6 Libros copiados | sinopsis copiada de otro libro | hecho |
| 2.7 Combinar y entregar | `sospechosos.csv`, `catalogo_limpio.csv` | hecho |

El paso 2 queda cerrado. Las salidas son `sospechosos.csv` (407 libros con su motivo) y
`catalogo_limpio.csv` (8184 libros), que es el catálogo sobre el que trabajan los pasos 3 a 8.

El paso 3 arranca desde acá: se mezcla `catalogo_limpio.csv` una sola vez con
`sample(frac=1, random_state=42)` y el lector de referencia recorre ese orden.

# Paso 3 — Registrar las preferencias de un lector

**Lector de referencia: Julián Silva.** Una sola persona da todas las respuestas, y las da
antes de ver cualquier predicción del modelo: si el lector ya vio lo que el modelo predice,
sus respuestas quedan contaminadas y el paso 6 no mide nada.

Lo que pide el enunciado:

1. Mezclar el catálogo limpio **una sola vez** con `sample(frac=1, random_state=42)` y recorrer
   ese orden desde el principio, sin saltear ni elegir.
2. Leer título y sinopsis de cada libro y responder: **¿me interesaría leer este libro?**
3. Columna `interes`: **1** sí, **0** no, **vacío** si no puede decidir.
4. Seguir hasta reunir **al menos 100 respuestas válidas y 20 de cada clase**.

Un libro sin valorar tiene preferencia desconocida: **no se convierte en 0**. Esos registros se
conservan y viajan en el entregable; el paso 4 trabaja solo con los 0 y los 1.

In [66]:
# --- 3.1 El orden de lectura ---
ARCHIVO_PREFERENCIAS = "preferencias.csv"
LIBROS_A_PREPARAR = 200   # con 100 validas y 20 por clase alcanza; sobra margen

# El orden sale del catalogo limpio con la semilla que fija el enunciado, asi que
# cualquiera del grupo puede reconstruir exactamente esta misma lista.
orden_lectura = catalogo_limpio.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"catalogo limpio mezclado: {len(orden_lectura)} libros")
print("primeros 5 del orden:")
for i, (slug, titulo) in enumerate(zip(orden_lectura["slug"].head(5),
                                       orden_lectura["title"].head(5)), start=1):
    print(f"  {i}. {slug}  -  {titulo}")

catalogo limpio mezclado: 8184 libros
primeros 5 del orden:
  1. cantos-folkloricos-chilenos  -  Cantos folklóricos chilenos
  2. alabardas  -  Alabardas
  3. sexual-personae  -  Sexual Personae
  4. condenados-a-escribir  -  Condenados a escribir
  5. la-habitacion-sellada  -  La habitación sellada


In [67]:
# --- 3.2 La planilla para valorar ---
# Si el archivo ya existe NO se toca: tiene las respuestas del lector, que son
# una hora de trabajo que no se puede regenerar. Solo se crea la primera vez.
ruta_preferencias = CARPETA_DATOS / ARCHIVO_PREFERENCIAS

if ruta_preferencias.exists():
    preferencias = pd.read_csv(ruta_preferencias, keep_default_na=False)
    print(f"preferencias.csv ya existe: {len(preferencias)} filas (no se sobrescribe)")
else:
    # Las columnas son las que nombra el enunciado: slug, title, synopsis, interes.
    preferencias = orden_lectura.head(LIBROS_A_PREPARAR)[["slug", "title", "sinopsis"]].copy()
    preferencias = preferencias.rename(columns={"sinopsis": "synopsis"})
    preferencias["interes"] = ""
    preferencias.to_csv(ruta_preferencias, index=False, encoding="utf-8")
    print(f"creado {ruta_preferencias} con {len(preferencias)} libros para valorar")

# Estado de la valoracion
respuestas = pd.to_numeric(preferencias["interes"], errors="coerce")
validas = int(respuestas.notna().sum())
print("")
print(f"respuestas validas: {validas} de {len(preferencias)}")
print(f"  interes = 1: {int((respuestas == 1).sum())}")
print(f"  interes = 0: {int((respuestas == 0).sum())}")
print(f"  sin valorar: {int(respuestas.isna().sum())}")
if validas >= 100 and (respuestas == 1).sum() >= 20 and (respuestas == 0).sum() >= 20:
    print("alcanza el minimo del enunciado: 100 validas y 20 de cada clase")
else:
    print("todavia no alcanza el minimo del enunciado (100 validas, 20 por clase)")

preferencias.csv ya existe: 200 filas (no se sobrescribe)

respuestas validas: 200 de 200
  interes = 1: 29
  interes = 0: 171
  sin valorar: 0
alcanza el minimo del enunciado: 100 validas y 20 de cada clase


### Cómo completarla

Se abre `data/TP_3/preferencias.csv`, se lee título y sinopsis de cada fila **en el orden en que
están** y se escribe el valor en la columna `interes`. Tres reglas:

- **No reordenar ni ordenar la planilla.** El orden *es* el `sample` del enunciado; si se
  altera, se pierde y deja de ser reproducible.
- **Dejar vacío, no poner 0**, cuando no se puede decidir.
- **Cerrar la planilla antes de volver a correr el notebook.** La carpeta está sincronizada, y
  un archivo abierto en Excel no se puede escribir. La celda de arriba no sobrescribe el
  archivo si ya existe, así que las respuestas no corren riesgo por una corrida.

Un detalle práctico: las sinopsis llegan a **4581 caracteres**, y un CSV crudo en Excel las
muestra desbordadas en una sola línea. La lectura se hizo sobre `preferencias.xlsx`, la misma
tabla con texto ajustado y la columna `interes` validada a 1 / 0 / vacío. **El CSV sigue siendo
la fuente**: el `.xlsx` es solo la interfaz de lectura y se regenera desde el CSV, así que no
agrega un segundo origen de verdad ni entra en el entregable.

Si durante la lectura aparece un libro contaminado que el paso 2 no detectó, el enunciado pide
dejarlo con `interes` vacío y agregarlo a `sospechosos.csv` con motivo **«revisión manual»**.
Vale la pena anotar cuáles fueron: recorrer unos 150 libros ya depurados y contar cuántos
contaminados quedaron es **la única medición independiente del recall del paso 2** que vamos a
tener, y va derecho al informe.

In [68]:
# --- 3.3 Que quedo de la valoracion ---
# Se recarga del disco a proposito: la celda 3.2 no sobrescribe el archivo, asi que
# en una corrida posterior a la lectura 'preferencias' ya trae las respuestas.
preferencias = pd.read_csv(ruta_preferencias, keep_default_na=False)
valor = pd.to_numeric(preferencias["interes"], errors="coerce")

print(f"libros valorados: {int(valor.notna().sum())} de {len(preferencias)}")
print(f"  me interesa (1):    {int((valor == 1).sum())}")
print(f"  no me interesa (0): {int((valor == 0).sum())}")
print(f"  abstenciones:       {int(valor.isna().sum())}")
print(f"tasa de positivos: {valor.mean():.1%}")

# El desbalance es el dato que condiciona los pasos 4 a 6, no un detalle: con esta
# tasa el 20 % de prueba se queda con muy pocos positivos, y cada uno mueve el
# recall de la clase 1 varios puntos. Se calcula aca para poder declararlo.
positivos = int((valor == 1).sum())
en_prueba = round(positivos * 0.2)
print("")
print(f"el 20 % de prueba del paso 4 va a tener ~{en_prueba} positivos: "
      f"cada uno vale {1 / en_prueba:.1%} de recall")

libros valorados: 200 de 200
  me interesa (1):    29
  no me interesa (0): 171
  abstenciones:       0
tasa de positivos: 14.5%

el 20 % de prueba del paso 4 va a tener ~6 positivos: cada uno vale 16.7% de recall


### Qué quedó

**200 libros valorados, sin abstenciones**: 29 con `interes = 1` y 171 con `interes = 0`. Supera
el mínimo del enunciado —100 válidas y 20 de cada clase— y el orden de lectura quedó intacto:
los 200 slugs siguen siendo los primeros 200 de `sample(frac=1, random_state=42)`.

**La tasa de positivos es 14,5 %, y eso condiciona todo lo que viene.** Conviene decirlo acá y
no descubrirlo en el paso 6: al partir 80/20, la parte de prueba se queda con unos **6 libros
de la clase 1**, así que cada acierto o error mueve el recall de esa clase **17 puntos**. Un
recall de 0.67 y uno de 0.83 se diferencian en dos libros. Por eso en el paso 6 el número de
una sola partición no alcanza como evidencia, y hay que acompañarlo con validación cruzada
sobre los 200: es la misma lección del 2.5, donde una partición favorable daba una conclusión
que la medición por plantilla desarmaba.

El desbalance no se corrige tocando las etiquetas: son las preferencias reales del lector y
falsearlas haría que el paso 6 no mida nada. Se corrige en el modelo, y de hecho el
`class_weight="balanced"` que fija el enunciado es exactamente eso.

# Paso 4 — Separar los libros para aprender y evaluar

El enunciado pide:
1. Trabajar solo con los registros que tienen respuesta válida (`interes` igual a 0 o 1).
2. Dividir 80% para entrenamiento y 20% para prueba usando `train_test_split` con `test_size=0.2`, `random_state=42` y `stratify=validas["interes"]`.
3. Guardar en `preferencias.csv` la columna `conjunto` indicando `entrenamiento` o `prueba`.
4. Mostrar la cantidad de libros y respuestas positivas en cada partición.

In [69]:
#  Carga y división estratificada
from sklearn.model_selection import train_test_split

ruta_preferencias = CARPETA_DATOS / ARCHIVO_PREFERENCIAS
preferencias = pd.read_csv(ruta_preferencias, keep_default_na=False)

# Aseguramos formato numérico para interes
preferencias["interes_num"] = pd.to_numeric(preferencias["interes"], errors="coerce")

# Filtramos solo respuestas válidas (0 o 1)
validas = preferencias[preferencias["interes_num"].isin([0, 1])].copy()
validas["interes_num"] = validas["interes_num"].astype(int)

print(f"Total registros en archivo: {len(preferencias)}")
print(f"Respuestas válidas para el split: {len(validas)}")

# División 80/20 con semilla fija y estratificación
train_idx, test_idx = train_test_split(
    validas.index,
    test_size=0.2,
    random_state=42,
    stratify=validas["interes_num"]
)

# Asignamos la columna 'conjunto' en el DataFrame principal
preferencias["conjunto"] = ""
preferencias.loc[train_idx, "conjunto"] = "entrenamiento"
preferencias.loc[test_idx, "conjunto"] = "prueba"

# Guardamos de nuevo preferencias.csv preservando la columna conjunto
preferencias_salida = preferencias.drop(columns=["interes_num"])
preferencias_salida.to_csv(ruta_preferencias, index=False, encoding="utf-8")
print(f"\nArchivo actualizado en: {ruta_preferencias} con columna 'conjunto'")

# DataFrames separados para los pasos siguientes
entrenamiento = preferencias.loc[train_idx].copy()
prueba = preferencias.loc[test_idx].copy()

# Conteo y distribución de clases
print("\n=== Distribución de las particiones ===")
print(f"Entrenamiento: {len(entrenamiento)} libros | Positivos (1): {(entrenamiento['interes_num'] == 1).sum()} ({(entrenamiento['interes_num'] == 1).mean():.1%})")
print(f"Prueba:        {len(prueba)} libros | Positivos (1): {(prueba['interes_num'] == 1).sum()} ({(prueba['interes_num'] == 1).mean():.1%})")

Total registros en archivo: 200
Respuestas válidas para el split: 200

Archivo actualizado en: ..\..\data\TP_3\preferencias.csv con columna 'conjunto'

=== Distribución de las particiones ===
Entrenamiento: 160 libros | Positivos (1): 23 (14.4%)
Prueba:        40 libros | Positivos (1): 6 (15.0%)


# Paso 5 — Entrenar dos clasificadores

La entrada es la **sinopsis** y lo que hay que aprender es **`interes`**. Cambia una sola cosa
entre los dos modelos: **cómo se representa el texto**. La configuración de la regresión
logística la fija el enunciado —`C=1`, `max_iter=1000`, `class_weight="balanced"`— y es la misma
en los dos, sin búsqueda de hiperparámetros en ningún momento del TP.

- **Modelo A — TF-IDF.** Cada sinopsis es un vector de pesos por palabra. Reconoce coincidencias
  literales de vocabulario.
- **Modelo B — embeddings multilingües.** El modelo del TP2,
  `distiluse-base-multilingual-cased-v1`, convierte cada sinopsis en 512 dimensiones que
  codifican significado. Reconoce temas parecidos aunque las palabras no se repitan.

`class_weight="balanced"` importa especialmente acá: con 23 positivos sobre 160 libros, una
regresión logística sin ese ajuste tiende a predecir 0 para todos, porque acertarle a la clase
mayoritaria es lo que más reduce el error. El ajuste le da más peso a equivocarse en la clase
menos frecuente. Es parte de la configuración fija, no una mejora que buscamos nosotros.

In [70]:
# --- 5.1 Los dos conjuntos y la configuracion fija ---
# La configuracion la fija el enunciado para todo el TP y no se toca: no hay
# busqueda de hiperparametros. Se escribe una vez y se usa en los dos modelos,
# para que no haya forma de que difieran en algo que no sea la representacion.
CONFIG_LR = dict(C=1, max_iter=1000, class_weight="balanced")

preferencias = pd.read_csv(ruta_preferencias, keep_default_na=False)
libros_ent = preferencias[preferencias["conjunto"] == "entrenamiento"]
libros_pru = preferencias[preferencias["conjunto"] == "prueba"]
y_ent = pd.to_numeric(libros_ent["interes"]).to_numpy()
y_pru = pd.to_numeric(libros_pru["interes"]).to_numpy()

print(f"entrenamiento: {len(libros_ent)} libros, {y_ent.sum()} positivos")
print(f"prueba:        {len(libros_pru)} libros, {y_pru.sum()} positivos")
print(f"configuracion: LogisticRegression({CONFIG_LR})")

entrenamiento: 160 libros, 23 positivos
prueba:        40 libros, 6 positivos
configuracion: LogisticRegression({'C': 1, 'max_iter': 1000, 'class_weight': 'balanced'})


In [71]:
# --- 5.2 Modelo A: TF-IDF ---
# fit_transform sobre entrenamiento y transform sobre prueba, nunca al reves: el
# vocabulario y los pesos idf tienen que aprenderse solo con los textos de
# entrenamiento. Si se ajustara sobre todo, prueba filtraria informacion al modelo
# y la evaluacion del paso 6 quedaria inflada.
vectorizador = TfidfVectorizer()
tfidf_ent = vectorizador.fit_transform(libros_ent["synopsis"])
tfidf_pru = vectorizador.transform(libros_pru["synopsis"])

modelo_a = LogisticRegression(**CONFIG_LR).fit(tfidf_ent, y_ent)
pred_a = modelo_a.predict(tfidf_pru)

print(f"matriz de entrenamiento: {tfidf_ent.shape}")
print(f"matriz de prueba:        {tfidf_pru.shape}")
print(f"vocabulario aprendido:   {len(vectorizador.vocabulary_)} terminos")
print(f"densidad: {tfidf_ent.nnz / (tfidf_ent.shape[0] * tfidf_ent.shape[1]):.2%} de celdas no nulas")
print(f"predicciones positivas: {int(pred_a.sum())} de {len(pred_a)}")

matriz de entrenamiento: (160, 6845)


matriz de prueba:        (40, 6845)
vocabulario aprendido:   6845 terminos
densidad: 1.44% de celdas no nulas
predicciones positivas: 0 de 40


In [72]:
# --- 5.3 Modelo B: embeddings multilingues ---
# Se reusa 'modelo_emb' del 2.5, que es el del TP2
# (distiluse-base-multilingual-cased-v1): el enunciado pide un modelo multilingue
# porque las sinopsis estan en castellano. El texto va natural, con tildes y
# mayusculas, que es lo que el modelo vio al entrenarse.
emb_ent = modelo_emb.encode(libros_ent["synopsis"].tolist(), show_progress_bar=False)
emb_pru = modelo_emb.encode(libros_pru["synopsis"].tolist(), show_progress_bar=False)

modelo_b = LogisticRegression(**CONFIG_LR).fit(emb_ent, y_ent)
pred_b = modelo_b.predict(emb_pru)

print(f"matriz de entrenamiento: {emb_ent.shape}")
print(f"matriz de prueba:        {emb_pru.shape}")
print(f"predicciones positivas: {int(pred_b.sum())} de {len(pred_b)}")

# Cuanto se pierde por truncamiento. Importa declararlo: el modelo tiene
# max_seq_length 128 y estas sinopsis son mucho mas largas que las oraciones del
# 2.5, asi que buena parte del texto no llega a entrar al vector.
tokens = [len(modelo_emb.tokenizer.encode(s)) for s in preferencias["synopsis"]]
print("")
print(f"tokens por sinopsis: mediana {int(np.median(tokens))}, maximo {max(tokens)}")
print(f"sinopsis truncadas a {modelo_emb.max_seq_length} tokens: "
      f"{sum(1 for n in tokens if n > modelo_emb.max_seq_length)} de {len(tokens)}")

matriz de entrenamiento: (160, 512)
matriz de prueba:        (40, 512)
predicciones positivas: 6 de 40

tokens por sinopsis: mediana 209, maximo 1100
sinopsis truncadas a 128 tokens: 166 de 200


### El tamaño de las matrices

| representación | entrenamiento | prueba | qué es cada columna |
|---|---|---|---|
| A — TF-IDF | 160 × 6845 | 40 × 6845 | un término del vocabulario de entrenamiento |
| B — embeddings | 160 × 512 | 40 × 512 | una dimensión del espacio del modelo |

Las dos matrices de prueba tienen **las mismas columnas** que su matriz de entrenamiento, y eso
no es casualidad: en TF-IDF porque `transform` reusa el vocabulario aprendido, y en embeddings
porque el modelo está fijo y siempre devuelve 512 dimensiones. Si el número de columnas cambiara
entre entrenamiento y prueba, la regresión logística no podría ni recibir los vectores.

**La diferencia de forma es el dato interesante del paso.** TF-IDF usa **6845 columnas para
describir 160 libros**: hay 43 veces más rasgos que ejemplos, y la matriz tiene apenas **1,4 %
de celdas no nulas**. Los embeddings describen el mismo libro en 512 dimensiones densas, todas
con información. Con tan pocos ejemplos, 6845 rasgos dispersos le dan al modelo muchísimas
formas de separar los 160 libros de entrenamiento por coincidencias que no se repiten afuera.

**Lo que el modelo B no ve.** La mediana de las sinopsis es de **209 tokens** y el modelo corta
en **128**: **166 de los 200 libros (83 %) entran truncados**, algunos de 1100 tokens. El vector
representa el primer párrafo y poco más. Es una limitación real de este modelo sobre textos
largos —en el 2.5 no molestaba porque ahí clasificábamos oraciones— y hay que declararla.

In [73]:
# --- 5.4 Conservar las predicciones ---
# El enunciado pide conservarlas: el paso 6 las compara con las respuestas reales
# y el paso 7 las convierte en recomendaciones. Se guardan junto al slug y a la
# etiqueta verdadera para que cada fila se pueda auditar libro por libro.
predicciones = pd.DataFrame({
    "slug": libros_pru["slug"].values,
    "title": libros_pru["title"].values,
    "interes": y_pru,
    "pred_tfidf": pred_a,
    "pred_embeddings": pred_b,
})
print(f"{len(predicciones)} libros de prueba con las dos predicciones")
print(f"los dos modelos coinciden en {(pred_a == pred_b).sum()} de {len(pred_a)} libros")
print("")
print(predicciones.head(8).to_string(index=False, max_colwidth=34))

40 libros de prueba con las dos predicciones
los dos modelos coinciden en 34 de 40 libros

                              slug                              title  interes  pred_tfidf  pred_embeddings
                         alabardas                          Alabardas        0           0                0
renata-toca-el-piano-estudia-in... Renata toca el piano, estudia i...        0           0                0
himnos-a-la-noche-enrique-de-of... Himnos a la noche. Enrique de O...        0           0                0
                 diarios-1984-1989                  Diarios 1984-1989        0           0                0
crimen-y-castigo-trad-rafael-ca... Crimen y castigo (trad. Rafael ...        1           0                0
         confesiones-de-un-vicioso          Confesiones de un vicioso        1           0                0
                historias-de-divan                 Historias de diván        0           0                1
rompecabezas-y-paradojas-cienti... Rompecabez

In [74]:
# --- 5.5 Por que el modelo A no marca nada ---
# Esta celda no la pide el enunciado, pero la afirmacion de abajo tiene que poder
# mostrarse: el TP se defiende oralmente y un numero escrito en prosa que ninguna
# celda imprime no es evidencia. Mide, no cambia nada del modelo.
prob_a = modelo_a.predict_proba(tfidf_pru)[:, 1]
print(f"F1 de la clase 1 en ENTRENAMIENTO: {f1_score(y_ent, modelo_a.predict(tfidf_ent)):.3f}")
print(f"F1 de la clase 1 en PRUEBA:        {f1_score(y_pru, pred_a, zero_division=0):.3f}")
print("")
print(f"probabilidad mas alta en prueba: {prob_a.max():.3f} (hace falta 0.5 para predecir 1)")
print(f"ese libro estaba valorado con: {y_pru[prob_a.argmax()]}")

# Si el problema fuera vocabulario desconocido, la cobertura seria baja. No lo es:
# el modelo lee casi todas las palabras, pero separo entrenamiento por terminos
# que no distinguen interes afuera. Eso es sobreajuste, no falta de vocabulario.
palabras = [re.findall(r"\w+", s.lower()) for s in libros_pru["synopsis"]]
cobertura = [sum(1 for p in ps if p in vectorizador.vocabulary_) / len(ps)
             for ps in palabras if ps]
print(f"palabras de prueba presentes en el vocabulario aprendido: {np.mean(cobertura):.0%}")

F1 de la clase 1 en ENTRENAMIENTO: 1.000
F1 de la clase 1 en PRUEBA:        0.000

probabilidad mas alta en prueba: 0.449 (hace falta 0.5 para predecir 1)
ese libro estaba valorado con: 0
palabras de prueba presentes en el vocabulario aprendido: 76%


### Lo que ya se ve

Las predicciones están guardadas y el paso 6 es el que las evalúa, pero hay un hecho del paso 5
que conviene no tapar: **el modelo A no marca ni un libro como de interés** —0 positivos de 40—
y el modelo B marca 6. El enunciado anticipa exactamente este caso y pide aclararlo.

No es que al modelo A le falte poco. Su probabilidad más alta en prueba es **0.449** (lo imprime la celda 5.5): ningún
libro llega al 0.5, y el que más se acerca es uno que el lector había valorado con 0. Al mismo
tiempo, sobre los libros de entrenamiento **los separa perfecto**. Memorizó el vocabulario de
esos 160 libros en vez de aprender el criterio, que es justo lo que invitan 6845 rasgos para 160
ejemplos. El 76 % de las palabras de prueba sí están en el vocabulario aprendido, así que el
problema no es que lea palabras desconocidas: es que las palabras que usó para separar no son
las que distinguen un libro interesante de otro.

**No tocamos nada para arreglarlo.** La configuración es fija por enunciado y cambiar el modelo
mirando los resultados de prueba invalidaría la evaluación. El resultado se mide en el paso 6 y
se interpreta en el informe.

# Paso 6 — Comprobar las predicciones

Hasta acá los dos modelos predijeron; ahora hay que ver cuánto valen esas predicciones. El
enunciado pide cuatro cosas: el `classification_report` con la matriz de confusión, una tabla
con precisión, recall y F1 **de la clase 1** para cada modelo, una validación cruzada que
muestre cuánto varían esos números, y dos libros comentados uno por uno.

Las tres medidas de la clase 1, dichas como preguntas:

| medida | qué pregunta |
|---|---|
| precisión | de los libros que predijimos de interés, ¿cuántos interesaban? |
| recall | de los libros que interesaban, ¿cuántos detectamos? |
| F1 | ¿qué balance hay entre las dos? |

Con 40 libros de prueba y 6 positivos, **un libro mueve el recall 17 puntos**. Por eso las
proporciones van acompañadas del conteo crudo de aciertos y errores, y por eso la validación
cruzada no es un adorno: es lo que distingue un resultado de una casualidad.

In [75]:
# --- 6.1 Reporte y matriz de confusion ---
# zero_division=0 porque el modelo A no predice ningun positivo: sin esto, la
# precision de la clase 1 seria 0/0 y sklearn avisaria con una advertencia en vez
# de dar un numero. El 0 no significa "no se pudo medir", significa que de los
# libros que predijo de interes (ninguno) no acerto ninguno.
MODELOS = [("A  TF-IDF", pred_a), ("B  embeddings", pred_b)]

for nombre, pred in MODELOS:
    print(f"===== Modelo {nombre} =====")
    print(classification_report(y_pru, pred, zero_division=0, digits=3,
                                target_names=["no interesa (0)", "interesa (1)"]))
    mc = confusion_matrix(y_pru, pred, labels=[0, 1])
    print(pd.DataFrame(mc,
                       index=["real 0", "real 1"],
                       columns=["predijo 0", "predijo 1"]).to_string())
    print("")

===== Modelo A  TF-IDF =====
                 precision    recall  f1-score   support

no interesa (0)      0.850     1.000     0.919        34
   interesa (1)      0.000     0.000     0.000         6

       accuracy                          0.850        40
      macro avg      0.425     0.500     0.459        40
   weighted avg      0.722     0.850     0.781        40

        predijo 0  predijo 1
real 0         34          0
real 1          6          0

===== Modelo B  embeddings =====
                 precision    recall  f1-score   support

no interesa (0)      0.853     0.853     0.853        34
   interesa (1)      0.167     0.167     0.167         6

       accuracy                          0.750        40
      macro avg      0.510     0.510     0.510        40
   weighted avg      0.750     0.750     0.750        40

        predijo 0  predijo 1
real 0         29          5
real 1          5          1



In [76]:
# --- 6.2 La tabla de la clase 1 ---
# Las tres medidas de la clase 1 juntas, mas los aciertos y errores en crudo: con
# 40 libros de prueba y 6 positivos, un solo libro mueve las proporciones varios
# puntos, asi que el conteo es mas informativo que el porcentaje.
filas = []
for nombre, pred in MODELOS:
    vn, fp, fn, vp = confusion_matrix(y_pru, pred, labels=[0, 1]).ravel()
    filas.append({
        "modelo": nombre,
        "precision": round(precision_score(y_pru, pred, zero_division=0), 3),
        "recall": round(recall_score(y_pru, pred, zero_division=0), 3),
        "f1": round(f1_score(y_pru, pred, zero_division=0), 3),
        "marco": int(vp + fp),       # cuantos libros predijo de interes
        "acerto": int(vp),           # de esos, cuantos interesaban
        "se le escaparon": int(fn),  # interesaban y no los marco
        "aciertos totales": int(vn + vp),
    })
tabla_clase1 = pd.DataFrame(filas)
print(tabla_clase1.to_string(index=False))
print("")
print(f"libros de prueba: {len(y_pru)} | interesaban al lector: {int(y_pru.sum())}")
print(f"piso de azar del F1 (marcar los 40 con 1): "
      f"{2 * y_pru.mean() / (1 + y_pru.mean()):.3f}")

       modelo  precision  recall    f1  marco  acerto  se le escaparon  aciertos totales
    A  TF-IDF      0.000   0.000 0.000      0       0                6                34
B  embeddings      0.167   0.167 0.167      6       1                5                30

libros de prueba: 40 | interesaban al lector: 6
piso de azar del F1 (marcar los 40 con 1): 0.261


### Qué dicen las matrices

**Modelo A — TF-IDF.** Acierta **34 de 40**, y aun así es inútil: acertó los 34 libros que no
interesaban y **perdió los 6 que sí**. Su precisión, recall y F1 de la clase 1 son **0,000**.
La exactitud del 85 % que informa el reporte es la trampa clásica del dato desbalanceado:
responder «no» siempre da 85 % acá, porque 34 de 40 libros son «no». No aprendió nada sobre lo
que te gusta; aprendió que la mayoría de los libros no te gustan.

**Modelo B — embeddings.** Marca 6 libros y acierta **1**. Precisión **0,167**, recall **0,167**,
F1 **0,167**. Acierta **30 de 40**, o sea **menos** que el modelo A, y sin embargo es el único de
los dos que sirve para algo: equivocarse intentando es informativo, y no intentar nunca no lo es.
Esta es la razón por la que el enunciado pide mirar la clase 1 y no la exactitud.

**El piso de azar.** Marcar los 40 libros con 1 —sin leer nada— daría recall 1,000, precisión
0,150 y **F1 0,261**. Los dos modelos están por debajo de eso. No es un detalle para esconder en
una nota al pie: con este conjunto de datos, ninguno de los dos clasificadores supera a una regla
que no mira el texto.

In [77]:
# --- 6.3 Cuanto varia el F1: validacion cruzada ---
# Con 6 positivos en prueba, un libro vale 17 puntos de recall. El enunciado pide
# medir esa inestabilidad con RepeatedStratifiedKFold sobre ENTRENAMIENTO: 5
# pliegues repetidos 10 veces son 50 mediciones del mismo modelo.
#
# El modelo A va dentro de un Pipeline a proposito: asi el TfidfVectorizer se
# ajusta con los libros de cada pliegue y no con los 160. Si se vectorizara una
# vez afuera, el vocabulario del pliegue de validacion entraria al de ajuste y el
# numero saldria optimista.
#
# Esto NO se usa para elegir parametros: la configuracion esta fija por enunciado.
cv_repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

f1_a = cross_val_score(make_pipeline(TfidfVectorizer(), LogisticRegression(**CONFIG_LR)),
                       libros_ent["synopsis"], y_ent, cv=cv_repetida, scoring="f1")
f1_b = cross_val_score(LogisticRegression(**CONFIG_LR), emb_ent, y_ent,
                       cv=cv_repetida, scoring="f1")

for nombre, s in [("A  TF-IDF", f1_a), ("B  embeddings", f1_b)]:
    print(f"{nombre:16s} F1 clase 1: {s.mean():.3f} +- {s.std():.3f}  "
          f"(min {s.min():.3f}, max {s.max():.3f}, cero en {int((s == 0).sum())}/{len(s)} pliegues)")

A  TF-IDF        F1 clase 1: 0.000 +- 0.000  (min 0.000, max 0.000, cero en 50/50 pliegues)
B  embeddings    F1 clase 1: 0.132 +- 0.155  (min 0.000, max 0.571, cero en 27/50 pliegues)


### Lo que dice la validación cruzada

| modelo | F1 de la clase 1 (50 pliegues) | pliegues en cero |
|---|---|---|
| A — TF-IDF | **0,000 ± 0,000** | 50 de 50 |
| B — embeddings | **0,132 ± 0,155** | 27 de 50 |

Los dos números dicen cosas distintas y las dos importan.

**El cero del modelo A no es mala suerte de una partición.** En las 50 mediciones no marcó un
solo positivo. Si hubiera sido cuestión del reparto de los libros, alguna de las 50 habría dado
algo distinto de cero. No lo hizo ninguna: con 160 libros y 6845 términos, TF-IDF más regresión
logística simplemente no llega a predecir esta clase.

**El modelo B es inestable, no bueno.** Su desvío (0,155) es más grande que su media (0,132), y
va de 0,000 a 0,571 según qué libros le toquen. Un F1 de 0,167 en prueba cae dentro de ese rango:
no es una medición de su calidad, es una muestra de su ruido. Si la partición hubiera sido otra,
el número habría sido muy distinto —y la conclusión también, si uno se lo creyera—.

In [78]:
# --- 6.4 El orden, no solo el umbral ---
# Esto no lo pide el enunciado, pero el paso 7 recomienda eligiendo los libros con
# mas probabilidad, o sea que depende del ORDEN y no de cuantos cruzan el 0.5. Un
# modelo puede ordenar bien y no marcar ninguno. Si no se mide, el paso 7 arranca
# sin saber si tiene de donde agarrarse.
#
# ROC AUC: probabilidad de que un libro que interesa quede arriba de uno que no.
#   0.5 es ordenar al azar.
# AP (average precision): el piso de azar es la proporcion de positivos, no 0.5.
prob_a_pru = modelo_a.predict_proba(tfidf_pru)[:, 1]
prob_b_pru = modelo_b.predict_proba(emb_pru)[:, 1]

print(f"sobre los {len(y_pru)} libros de prueba (piso del AP: {y_pru.mean():.3f})")
for nombre, prob in [("A  TF-IDF", prob_a_pru), ("B  embeddings", prob_b_pru)]:
    print(f"  {nombre:16s} ROC AUC {roc_auc_score(y_pru, prob):.3f} | "
          f"AP {average_precision_score(y_pru, prob):.3f}")

# Una particion de 40 libros con 6 positivos da numeros muy ruidosos, asi que se
# repite la medicion con los mismos 50 pliegues de arriba antes de afirmar nada.
print("")
print(f"validado con los 50 pliegues sobre entrenamiento (piso del AP: {y_ent.mean():.3f})")
for nombre, est, X in [
    ("A  TF-IDF", make_pipeline(TfidfVectorizer(), LogisticRegression(**CONFIG_LR)),
     libros_ent["synopsis"]),
    ("B  embeddings", LogisticRegression(**CONFIG_LR), emb_ent),
]:
    auc = cross_val_score(est, X, y_ent, cv=cv_repetida, scoring="roc_auc")
    ap = cross_val_score(est, X, y_ent, cv=cv_repetida, scoring="average_precision")
    print(f"  {nombre:16s} ROC AUC {auc.mean():.3f} +- {auc.std():.3f} "
          f"(bajo el azar en {int((auc < 0.5).sum())}/{len(auc)}) | "
          f"AP {ap.mean():.3f} +- {ap.std():.3f} "
          f"(bajo el azar en {int((ap < y_ent.mean()).sum())}/{len(ap)})")
# Cuantos de los primeros del ranking interesaban de verdad. Es la pregunta exacta
# que va a contestar el paso 7, planteada sobre los libros donde sabemos la verdad.
print("")
print(f"aciertos entre los primeros del ranking (de {int(y_pru.sum())} que interesaban)")
for nombre, prob in [("A  TF-IDF", prob_a_pru), ("B  embeddings", prob_b_pru)]:
    orden = np.argsort(-prob)
    top6 = int(y_pru[orden][:6].sum())
    top10 = int(y_pru[orden][:10].sum())
    print(f"  {nombre:16s} primeros 6: {top6} | primeros 10: {top10} "
          f"(al azar daria {6 * y_pru.mean():.1f} y {10 * y_pru.mean():.1f})")

sobre los 40 libros de prueba (piso del AP: 0.150)
  A  TF-IDF        ROC AUC 0.740 | AP 0.308
  B  embeddings    ROC AUC 0.686 | AP 0.282

validado con los 50 pliegues sobre entrenamiento (piso del AP: 0.144)


  A  TF-IDF        ROC AUC 0.599 +- 0.107 (bajo el azar en 9/50) | AP 0.286 +- 0.138 (bajo el azar en 1/50)


  B  embeddings    ROC AUC 0.581 +- 0.105 (bajo el azar en 10/50) | AP 0.287 +- 0.117 (bajo el azar en 2/50)

aciertos entre los primeros del ranking (de 6 que interesaban)
  A  TF-IDF        primeros 6: 2 | primeros 10: 3 (al azar daria 0.9 y 1.5)
  B  embeddings    primeros 6: 1 | primeros 10: 2 (al azar daria 0.9 y 1.5)


### El orden tiene algo que el umbral no muestra

Hay una pregunta que las medidas anteriores no contestan: el 0,5 es un corte arbitrario, y el
paso 7 no va a recomendar los libros que cruzan un umbral, va a **ordenar el catálogo por
probabilidad y tomar los primeros**. Un modelo puede ordenar razonablemente bien y no marcar
ninguno. Vale la pena medirlo antes de llegar al paso 7.

Sobre los 40 de prueba, el modelo A da **ROC AUC 0,740** y **AP 0,308** contra un piso de 0,150.
Leído así, el modelo que no marcó ni un libro resulta ser **el mejor ordenando**, lo cual
reencuadraría toda la conclusión.

**Pero ese número no aguanta la validación.** Repetido sobre los 50 pliegues, el AUC del modelo A
baja a **0,599 ± 0,107** y el del B a **0,581 ± 0,105**: apenas por encima del 0,500 del azar, y
por debajo del azar en 9 y 10 pliegues respectivamente. El 0,740 era una partición afortunada. Es
exactamente el error que el paso 2 ya nos había enseñado a no cometer.

**Lo que sí sobrevive es el AP**: **0,286 ± 0,138** el modelo A y **0,287 ± 0,117** el B, contra
un piso de 0,144. Es el doble del azar, y queda por debajo del piso en apenas 1 y 2 pliegues de
50. Dicho sin tecnicismo: **si se ordena el catálogo con cualquiera de los dos modelos y se toman
los primeros, la proporción de aciertos es alrededor del doble que eligiendo al azar.** Es una
señal débil, pero es consistente, y es mucho más de lo que sugiere un F1 de 0,000.

Los dos modelos **empatan** como ordenadores (0,286 contra 0,287). Esta evidencia no alcanza para
elegir uno, y conviene decirlo en lugar de inventar un ganador.

In [79]:
# --- 6.5 Libro por libro ---
# El enunciado pide mostrar dos libros de prueba y explicar que paso, incluyendo
# un error. Se arma la tabla completa de los errores del modelo B para que los dos
# comentados no sean una eleccion conveniente.
detalle = pd.DataFrame({
    "title": libros_pru["title"].values,
    "real": y_pru,
    "pred_A": pred_a,
    "pred_B": pred_b,
    "prob_B": prob_b_pru.round(3),
})
errores_b = detalle[detalle["real"] != detalle["pred_B"]].copy()
errores_b["tipo"] = np.where(errores_b["real"] == 0, "falso positivo", "falso negativo")
print(f"errores del modelo B: {len(errores_b)} de {len(detalle)}")
print(errores_b.sort_values("prob_B", ascending=False).to_string(index=False, max_colwidth=46))

# Los dos libros a comentar: el unico acierto de B y el error mas revelador.
print("")
for slug in ["renacimiento-el-arte-que-conquisto-el-mundo",
             "la-inteligencia-artificial-explicada-a-los-humanos"]:
    fila = libros_pru[libros_pru["slug"] == slug].iloc[0]
    i = int(np.where(libros_pru["slug"].values == slug)[0][0])
    print("-" * 78)
    print(f"{fila['title']}")
    print(f"  valoracion real: {y_pru[i]} | modelo A: {pred_a[i]} | "
          f"modelo B: {pred_b[i]} (prob {prob_b_pru[i]:.3f})")
    print(f"  sinopsis ({len(fila['synopsis'])} caracteres): {fila['synopsis'][:260]}...")

errores del modelo B: 10 de 40
                                         title  real  pred_A  pred_B  prob_B           tipo
              Recuerdos de la guerra de España     0       0       1   0.593 falso positivo
                                El libro negro     0       0       1   0.577 falso positivo
                            Historias de diván     0       0       1   0.521 falso positivo
     Historias del crepúsculo y lo desconocido     0       0       1   0.519 falso positivo
                         El fútbol, de la mano     0       0       1   0.500 falso positivo
La inteligencia artificial explicada a los ...     1       0       0   0.471 falso negativo
      Crimen y castigo (trad. Rafael Cansinos)     1       0       0   0.431 falso negativo
                    ¿Puede pensar una máquina?     1       0       0   0.428 falso negativo
          Rompecabezas y paradojas científicos     1       0       0   0.424 falso negativo
                     Confesiones de un vicioso   

### Dos libros, uno por uno

**El único acierto: *Renacimiento: El arte que conquistó el mundo*.** El modelo B le dio 0,584 y
lo marcó; el lector lo había valorado con 1. Es historia y arte, dos de los temas que recorren
los 29 positivos, y su sinopsis tiene 1788 caracteres de vocabulario temático bien explícito.
Cuando el tema está representado en el entrenamiento y el texto es largo, el modelo lo ve.

**El error que más dice: *La inteligencia artificial explicada a los humanos*.** Es un falso
negativo: valorado con 1, el modelo le dio **0,471** y se quedó a tres centésimas de marcarlo. Y
no está solo —*¿Puede pensar una máquina?* quedó en 0,428, también valorado con 1—. **Los dos
libros de inteligencia artificial de prueba se le escaparon**, y la IA es uno de los ejes más
claros del criterio del lector.

Ahí está el problema de fondo, y no es del modelo: en los 160 libros de entrenamiento hay **23
positivos** repartidos entre divulgación científica, historia, IA y ensayo. Son unos pocos
ejemplos por tema. El modelo no tiene con qué aprender que «inteligencia artificial» predice
interés, porque casi no vio libros de IA valorados con 1. No es que lea mal: es que no hay
suficiente que leer.

**Un detalle sobre los falsos positivos.** Los 5 del modelo B caen entre 0,500 y 0,593, es decir
**por encima de los 5 falsos negativos**, que van de 0,371 a 0,471. El modelo no está dudando
entre dos grupos parecidos: está ordenando los libros equivocados arriba de los correctos. De los
6 mejor rankeados, solo 1 interesaba.

## Cierre del paso 6

| | modelo A — TF-IDF | modelo B — embeddings |
|---|---|---|
| precisión clase 1 | 0,000 | 0,167 |
| recall clase 1 | 0,000 | 0,167 |
| F1 clase 1 | 0,000 | 0,167 |
| F1 validado (50 pliegues) | 0,000 ± 0,000 | 0,132 ± 0,155 |
| AP validado | 0,286 ± 0,138 | 0,287 ± 0,117 |
| aciertos totales | 34 de 40 | 30 de 40 |

**Tres cosas para el informe, en orden de importancia.**

1. **Como clasificadores binarios al 0,5, los dos fracasan**, y por debajo del piso de azar. El
   enunciado anticipa este caso y pide declararlo, no arreglarlo.
2. **Como ordenadores valen el doble del azar**, de forma consistente en 49 de 50 pliegues. Eso
   es lo que el paso 7 puede usar, y es la diferencia entre «el TP no funcionó» y «funciona
   débilmente, y sabemos cuánto».
3. **La causa es la cantidad de datos, no la representación.** 23 positivos repartidos en cuatro
   o cinco temas no alcanzan, y los dos modelos empatan como ordenadores justamente porque el
   límite no está en cómo se lee el texto sino en cuánto hay para aprender.

**Lo que no hicimos.** No cambiamos el umbral, ni los parámetros, ni las etiquetas. La validación
cruzada se usó para medir variabilidad, nunca para elegir configuración, que es lo que el
enunciado prohíbe expresamente. Las dos únicas celdas que exceden lo pedido —la 5.5 y la 6.4—
miden y no modifican nada.

# Paso 7 — Convertir las predicciones en recomendaciones

Recomendar no es lo mismo que clasificar. Al clasificador le preguntamos «¿sí o no?»; para
recomendar necesitamos **un orden**, porque hay que elegir cinco libros entre cuarenta. Un modelo
puede ordenar razonablemente bien y no marcar a ninguno con 1 —es exactamente lo que pasa con el
modelo A—, así que el umbral deja de importar.

Se arman **tres listas sobre los mismos 40 libros de prueba**:

- **A — TF-IDF con clasificador.** `predict_proba`, ordenar de mayor a menor, los primeros cinco.
- **B — embeddings con clasificador.** Lo mismo con el modelo B.
- **C — similitud al perfil del lector.** Sin entrenar nada: se promedian los embeddings de los
  libros de entrenamiento valorados con 1 y ese vector es el «perfil». Después, similitud del
  coseno entre el perfil y cada libro de prueba. Es el método del TP2.

La tercera lista es la interesante del paso, porque es la competencia honesta: una idea mucho más
simple, sin aprendizaje, contra los dos clasificadores.

**La métrica es precision@5**: de los cinco primeros, cuántos tenían `interes=1`, dividido cinco.
Y hay que leerla contra la referencia que pide el enunciado: la proporción de 1 en todo el
conjunto de prueba, **0,150**, que es lo que daría elegir cinco libros al azar.

In [80]:
# --- 7.1 Las tres listas y precision@5 ---
# classes_ no es un tramite: predict_proba devuelve una columna por clase en el
# orden de classes_. Si uno agarra la columna equivocada ordena al reves, y el
# resultado se ve como un modelo pesimo en vez de como un error de indice.
col_a = list(modelo_a.classes_).index(1)
col_b = list(modelo_b.classes_).index(1)
print(f"classes_ A: {modelo_a.classes_} -> la clase 1 es la columna {col_a}")
print(f"classes_ B: {modelo_b.classes_} -> la clase 1 es la columna {col_b}")

# Lista C: el perfil del lector es el promedio de los embeddings de los libros que
# valoro con 1. Solo los de ENTRENAMIENTO: con los de prueba, el perfil ya sabria
# las respuestas que despues se usan para medirlo.
perfil_lector = emb_ent[y_ent == 1].mean(axis=0)
sim_perfil = cosine_similarity(emb_pru, perfil_lector.reshape(1, -1)).ravel()
print(f"\nperfil armado con {int((y_ent == 1).sum())} libros, "
      f"{perfil_lector.shape[0]} dimensiones")

TOPE = 5
LISTAS = [
    ("A  TF-IDF + clasificador", modelo_a.predict_proba(tfidf_pru)[:, col_a]),
    ("B  embeddings + clasificador", modelo_b.predict_proba(emb_pru)[:, col_b]),
    ("C  similitud al perfil", sim_perfil),
]

azar = y_pru.mean()
print(f"\nreferencia: proporcion de 1 en prueba = {azar:.3f} "
      f"(precision esperada eligiendo al azar)")
precision_en_5 = {}
for nombre, puntaje in LISTAS:
    top = np.argsort(-puntaje)[:TOPE]
    aciertos = int(y_pru[top].sum())
    precision_en_5[nombre] = aciertos / TOPE
    print("")
    print(f"=== {nombre} | precision@{TOPE} = {aciertos}/{TOPE} = {aciertos / TOPE:.2f} ===")
    for puesto, i in enumerate(top, start=1):
        marca = "SI" if y_pru[i] == 1 else "no"
        print(f"  {puesto}. [{marca}] {puntaje[i]:.3f}  {libros_pru['title'].iloc[i][:56]}")

classes_ A: [0 1] -> la clase 1 es la columna 1
classes_ B: [0 1] -> la clase 1 es la columna 1

perfil armado con 23 libros, 512 dimensiones

referencia: proporcion de 1 en prueba = 0.150 (precision esperada eligiendo al azar)

=== A  TF-IDF + clasificador | precision@5 = 2/5 = 0.40 ===
  1. [no] 0.449  Alabardas
  2. [no] 0.446  Recuerdos de la guerra de España
  3. [no] 0.444  Historias del crepúsculo y lo desconocido
  4. [SI] 0.428  Renacimiento: El arte que conquistó el mundo
  5. [SI] 0.422  La inteligencia artificial explicada a los humanos

=== B  embeddings + clasificador | precision@5 = 1/5 = 0.20 ===
  1. [no] 0.593  Recuerdos de la guerra de España
  2. [SI] 0.584  Renacimiento: El arte que conquistó el mundo
  3. [no] 0.577  El libro negro
  4. [no] 0.521  Historias de diván
  5. [no] 0.519  Historias del crepúsculo y lo desconocido

=== C  similitud al perfil | precision@5 = 1/5 = 0.20 ===
  1. [no] 0.557  Historias del crepúsculo y lo desconocido
  2. [no] 0.542  A trav

### Qué dieron las tres listas

| lista | precision@5 | contra el azar (0,150) |
|---|---|---|
| A — TF-IDF + clasificador | **0,40** (2 de 5) | 2,7 veces |
| B — embeddings + clasificador | **0,20** (1 de 5) | 1,3 veces |
| C — similitud al perfil | **0,20** (1 de 5) | 1,3 veces |

Las tres superan al azar, y la lista A lo hace por bastante. Pero **antes de declarar ganador
hay que mirar el tamaño de la diferencia**: con cinco libros, un acierto vale 20 puntos. La
distancia entre 0,40 y 0,20 es **un libro**. No es un margen, es una anécdota.

Dos cosas que se ven mirando las listas y no las métricas:

- **La lista A ordena con probabilidades que nunca llegan a 0,5** —van de 0,449 a 0,422—. Es el
  mismo modelo que en el paso 6 no marcaba nada: recomienda igual, porque para ordenar no hace
  falta cruzar ningún umbral. Sus dos aciertos están en los puestos 4 y 5.
- **Los tres métodos se equivocan con los mismos libros.** *Historias del crepúsculo y lo
  desconocido* aparece en las tres listas y en ninguna interesaba. Cuando tres métodos distintos
  cometen el mismo error, el problema está en los datos y no en el método.

In [81]:
# --- 7.2 Una sola particion no distingue 0.40 de 0.20 ---
# Esto no lo pide el enunciado. Con 5 libros, un acierto vale 20 puntos: la
# diferencia entre las listas de arriba es UN libro. Para saber si el orden entre
# los tres metodos es real, se repite la comparacion completa sobre 50 particiones
# estratificadas distintas de los mismos 200 libros.
#
# Los embeddings se calculan una vez para los 200 porque el modelo esta fijo y
# codificar un texto no mira etiquetas: no hay filtracion. El TfidfVectorizer si
# se reajusta en cada particion, que es donde si la habria.
emb_todos = modelo_emb.encode(preferencias["synopsis"].tolist(), show_progress_bar=False)
y_todos = pd.to_numeric(preferencias["interes"]).to_numpy()
sinopsis_todas = preferencias["synopsis"].to_numpy()

resultados = {"A  TF-IDF": [], "B  embeddings": [], "C  perfil": []}
particiones = StratifiedShuffleSplit(n_splits=50, test_size=0.2, random_state=42)
for i_ent, i_pru in particiones.split(sinopsis_todas, y_todos):
    ye, yp = y_todos[i_ent], y_todos[i_pru]
    v = TfidfVectorizer()
    pa = (LogisticRegression(**CONFIG_LR)
          .fit(v.fit_transform(sinopsis_todas[i_ent]), ye)
          .predict_proba(v.transform(sinopsis_todas[i_pru]))[:, 1])
    pb = (LogisticRegression(**CONFIG_LR).fit(emb_todos[i_ent], ye)
          .predict_proba(emb_todos[i_pru])[:, 1])
    perfil = emb_todos[i_ent][ye == 1].mean(axis=0)
    pc = cosine_similarity(emb_todos[i_pru], perfil.reshape(1, -1)).ravel()
    for nombre, puntaje in [("A  TF-IDF", pa), ("B  embeddings", pb), ("C  perfil", pc)]:
        resultados[nombre].append(yp[np.argsort(-puntaje)[:TOPE]].sum() / TOPE)

azar_todos = y_todos.mean()
print(f"piso de azar sobre los 200 libros: {azar_todos:.3f}")
print("")
for nombre, valores in resultados.items():
    v = np.array(valores)
    print(f"{nombre:16s} precision@5: {v.mean():.3f} +- {v.std():.3f} | "
          f"cero en {int((v == 0).sum())}/50 | bajo el azar en {int((v < azar_todos).sum())}/50")

a, b, c = (np.array(resultados[k]) for k in ["A  TF-IDF", "B  embeddings", "C  perfil"])
print("")
print(f"A gana a B en {int((a > b).sum())}/50 particiones, pierde en {int((a < b).sum())}, "
      f"empata en {int((a == b).sum())}")
print(f"A gana a C en {int((a > c).sum())}/50 particiones, pierde en {int((a < c).sum())}, "
      f"empata en {int((a == c).sum())}")

piso de azar sobre los 200 libros: 0.145

A  TF-IDF        precision@5: 0.196 +- 0.130 | cero en 11/50 | bajo el azar en 11/50
B  embeddings    precision@5: 0.252 +- 0.149 | cero en 6/50 | bajo el azar en 6/50
C  perfil        precision@5: 0.160 +- 0.160 | cero en 21/50 | bajo el azar en 21/50

A gana a B en 8/50 particiones, pierde en 17, empata en 25
A gana a C en 24/50 particiones, pierde en 15, empata en 11


### Repetido 50 veces, el orden se da vuelta

| método | precision@5 (50 particiones) | cero | bajo el azar |
|---|---|---|---|
| A — TF-IDF | 0,196 ± 0,130 | 11 de 50 | 11 de 50 |
| B — embeddings | **0,252 ± 0,149** | 6 de 50 | 6 de 50 |
| C — perfil | 0,160 ± 0,160 | 21 de 50 | 21 de 50 |

**El mejor es el B, no el A.** Y no por poco: en el enfrentamiento directo, **A le gana a B en
8 de 50 particiones, pierde en 17 y empata en 25**. El 0,40 contra 0,20 de la partición que fija
el enunciado era una casualidad favorable al modelo A.

Es la tercera vez en este TP que una sola partición dice lo contrario que la medición repetida
—pasó con las plantillas en el 2.5 y con el AUC en el 6.4—, y las tres veces en la misma
dirección: **el número de una partición exagera**. Lo que hay que informar es el resultado que
pide el enunciado, con `random_state=42`; lo que hay que *creer* es esta tabla.

El modelo B es también el más confiable de los tres: queda por debajo del azar en apenas **6 de
50** particiones, contra 11 del A y 21 del C. Pero su desvío (0,149) sigue siendo más grande que
la distancia que lo separa del azar (0,252 − 0,145 = 0,107): recomienda mejor que elegir al azar,
y en una tanda concreta de cinco libros puede no parecerlo.

In [82]:
# --- 7.3 Por que el perfil no puede funcionar ---
# Promediar 29 embeddings supone que esos libros estan juntos en el espacio: que
# el gusto es UN punto. Si estan tan dispersos como los demas, el promedio no
# apunta a ningun lado. Es una hipotesis que se puede medir en un numero.
def similitudes_internas(X):
    """Similitudes de todos contra todos, sin la diagonal ni los repetidos."""
    S = cosine_similarity(X)
    return S[np.triu_indices_from(S, k=1)]

entre_positivos = similitudes_internas(emb_todos[y_todos == 1])
entre_negativos = similitudes_internas(emb_todos[y_todos == 0])
print(f"similitud promedio entre los {int((y_todos == 1).sum())} positivos: "
      f"{entre_positivos.mean():.3f}")
print(f"similitud promedio entre los {int((y_todos == 0).sum())} negativos: "
      f"{entre_negativos.mean():.3f}")
print(f"diferencia: {entre_positivos.mean() - entre_negativos.mean():+.3f}")

perfil_todos = emb_todos[y_todos == 1].mean(axis=0)
cerca_pos = cosine_similarity(emb_todos[y_todos == 1], perfil_todos.reshape(1, -1)).ravel()
cerca_neg = cosine_similarity(emb_todos[y_todos == 0], perfil_todos.reshape(1, -1)).ravel()
print("")
print(f"similitud al perfil: positivos {cerca_pos.mean():.3f} +- {cerca_pos.std():.3f} | "
      f"negativos {cerca_neg.mean():.3f} +- {cerca_neg.std():.3f}")
print(f"{int((cerca_neg > cerca_pos.mean()).sum())} de {len(cerca_neg)} negativos estan "
      f"mas cerca del perfil que el positivo promedio")

similitud promedio entre los 29 positivos: 0.184
similitud promedio entre los 171 negativos: 0.198
diferencia: -0.014

similitud al perfil: positivos 0.461 +- 0.100 | negativos 0.405 +- 0.079
44 de 171 negativos estan mas cerca del perfil que el positivo promedio


### Por qué el perfil no podía funcionar

El perfil queda en **0,160** contra un azar de **0,145**, y por debajo del azar en 21 de 50
particiones. Es, en la práctica, elegir al azar. Vale la pena entender por qué, porque la razón
no es que el método sea malo: es que **no se corresponde con la forma de estos datos**.

Promediar 29 embeddings supone algo fuerte: que esos 29 libros están **juntos** en el espacio, o
sea que el gusto del lector es *un* punto al que se le puede apuntar. Medido:

- similitud promedio entre los 29 positivos: **0,184**
- similitud promedio entre los 171 negativos: **0,198**

**Los libros que le gustaron están, si algo, más dispersos entre sí que los que no.** No hay una
región «de su gusto»: hay divulgación científica, historia, inteligencia artificial y ensayo, que
en el espacio del modelo no tienen por qué estar cerca. El promedio de vectores que apuntan a
lados distintos no apunta a ninguno: cae en un centro vacío, equidistante de todo.

Se ve en el solapamiento: los positivos están a **0,461 ± 0,100** del perfil y los negativos a
**0,405 ± 0,079**, y **44 de los 171 negativos están más cerca del perfil que el positivo
promedio**. Hay separación, pero mínima frente a la dispersión.

Esto no es una falla de la similitud del TP2, donde funcionó bien. Allá se comparaba **una
consulta** contra los libros, un punto contra otros puntos. Acá se la obliga a resumir un gusto
multitemático en un solo vector, y eso es una suposición que los datos no cumplen.

## La respuesta del paso 7

**¿Los clasificadores recomendaron mejor que la similitud?** **Sí, los dos**, y el fundamento no
es la partición del enunciado sino la medición repetida.

Sobre la partición que fija el enunciado, A da 0,40 y B y C empatan en 0,20: ahí la similitud no
queda peor que el clasificador con embeddings, y si uno se quedara con ese número la respuesta
sería «solo uno de los dos». Repetido 50 veces, **B 0,252 y A 0,196 contra 0,160 del perfil**, y
el perfil cae por debajo del azar en 21 de 50 particiones contra 6 del modelo B. La diferencia
está, aunque sea chica.

**Por qué ganan.** Un clasificador aprende **qué distingue** un libro que interesa de uno que no,
y puede usar rasgos distintos para cada tema. El perfil solo sabe **a qué se parece** el promedio
de lo que gustó, y cuando lo que gustó no se parece entre sí, ese promedio no describe a nadie.
Los números del 7.3 lo muestran: positivos a 0,184 de similitud entre ellos, negativos a 0,198.

**Tres cosas que hay que decir con el resultado, no después.**

1. **Ninguno de los tres sirve para armar una lista corta y confiable.** El mejor acierta 1,3
   libros de cada 5 en promedio. Es mejor que el azar —que daría 0,7— pero está lejos de una
   recomendación en la que uno confiaría.
2. **precision@5 sobre cinco libros es ruidosa por construcción.** Un acierto son 20 puntos, y el
   desvío de los tres métodos (0,13 a 0,16) es del tamaño de las diferencias entre ellos. Por eso
   la tabla de 50 particiones no es un extra: sin ella, cualquier conclusión es sobre un libro.
3. **El techo lo pone la cantidad de datos.** Los tres métodos fallan con los mismos libros, y el
   paso 6 ya había mostrado la causa: 23 positivos repartidos en cuatro o cinco temas dejan un
   puñado de ejemplos por tema.

**Lo que esto significa para el paso 8.** El enunciado pide recomendar del catálogo completo con
el clasificador de embeddings, y esta medición lo respalda: es el mejor de los tres y el más
consistente. Pero las cinco recomendaciones del paso 8 salen de un modelo que acierta alrededor
de 1 de cada 5, así que conviene presentarlas como lo que son y pedirle al lector que las
comente, que es justamente lo que el paso 8 pide.

# Paso 8 — Recomendar del catálogo completo

El último paso sale del conjunto de prueba y va a los **8184 libros del catálogo limpio**, donde
no hay respuestas con las que comparar. Tiene cuatro partes:

1. **Recomendar cinco libros** con el clasificador de embeddings del paso 5, **ya entrenado y sin
   reajustarlo**, excluyendo todo lo que el lector ya valoró y revisando que ninguna
   recomendación sea otra versión de un libro que ya valoró.
2. **Resumir cada sinopsis en dos oraciones** con el método extractivo de la unidad.
3. **Describir el perfil del lector** con spaCy: frases sustantivas y entidades de las sinopsis de
   entrenamiento, separadas por valoración.
4. **El comentario del lector** sobre las recomendaciones y sobre el perfil. Eso no lo puede
   producir el notebook.

El modelo se usa tal como quedó: entrenado con los 160 libros de entrenamiento. Volver a
ajustarlo con los 200 metería los libros de prueba, que son justamente los que midieron su
calidad en los pasos 6 y 7.

**Lo que hay que tener presente al leer las cinco recomendaciones:** el paso 7 midió que este
modelo acierta alrededor de **1 de cada 5** libros que recomienda (precision@5 de 0,252 contra
0,145 del azar). No son cinco aciertos esperados: son cinco apuestas de las que probablemente
una dé.

In [83]:
# --- 8.1 Los candidatos del catalogo ---
# Se excluyen TODOS los libros de preferencias.csv, incluidas las abstenciones:
# un libro que el lector vio y no supo decidir ya no es una recomendacion nueva.
# (Esta vez no hubo abstenciones, pero la regla queda escrita por si se amplia.)
ARCHIVO_EMB_CATALOGO = "emb_catalogo.npy"   # cache: codificar 8000 sinopsis tarda ~9 min

ya_valorados = set(preferencias["slug"])
candidatos = catalogo_limpio[~catalogo_limpio["slug"].isin(ya_valorados)].reset_index(drop=True)
print(f"catalogo limpio: {len(catalogo_limpio)} | ya valorados: {len(ya_valorados)}")
print(f"candidatos: {len(candidatos)}")

# El cache evita repetir el tramo mas lento del TP en cada corrida. Se valida la
# forma antes de usarlo: un cache de otra version del catalogo daria probabilidades
# asignadas a los libros equivocados, que es un error silencioso.
ruta_emb = CARPETA_DATOS / ARCHIVO_EMB_CATALOGO
if ruta_emb.exists() and np.load(ruta_emb).shape[0] == len(candidatos):
    emb_candidatos = np.load(ruta_emb)
    print(f"embeddings leidos del cache: {emb_candidatos.shape}")
else:
    emb_candidatos = modelo_emb.encode(candidatos["sinopsis"].tolist(),
                                       show_progress_bar=False, batch_size=64)
    np.save(ruta_emb, emb_candidatos)
    print(f"codificados y guardados en cache: {emb_candidatos.shape}")

# El modelo del paso 5, tal como quedo entrenado con los 160 de entrenamiento.
# No se reajusta: el enunciado lo pide asi y volver a entrenarlo con los 200
# mezclaria los libros de prueba, que son los que midieron su calidad.
prob_catalogo = modelo_b.predict_proba(emb_candidatos)[:, col_b]
print("")
print(f"probabilidad de interes: min {prob_catalogo.min():.3f}, "
      f"max {prob_catalogo.max():.3f}, mediana {np.median(prob_catalogo):.3f}")
print(f"por encima de 0.5: {int((prob_catalogo >= 0.5).sum())} de {len(prob_catalogo)} "
      f"({(prob_catalogo >= 0.5).mean():.1%})")

catalogo limpio: 8184 | ya valorados: 200
candidatos: 7984
embeddings leidos del cache: (7984, 512)

probabilidad de interes: min 0.168, max 0.748, mediana 0.403
por encima de 0.5: 1264 de 7984 (15.8%)


In [84]:
# --- 8.2 Que ninguna recomendacion sea otra version de un libro ya valorado ---
# Excluir por slug no alcanza: el catalogo trae ediciones, traducciones y
# reempaquetados del mismo libro con slugs distintos.
#
# El umbral de titulo del paso 1 (Jaro-Winkler 0.85) NO sirve solo: ahi era una
# compuerta previa a comparar sinopsis. Usado por si mismo, cualquier prefijo
# compartido es un duplicado ("Cuentos putrefactos" contra "Cuentos dispersos").
# Por eso la regla pide mismo autor Y titulo parecido, o sinopsis muy parecidas.
UMBRAL_TITULO_DUP = 0.85     # Jaro-Winkler, solo si ademas coincide el autor
UMBRAL_SINOPSIS_DUP = 0.55   # el mismo del paso 1
UMBRAL_CONTENCION_DUP = 0.90 # la regla del 2.6: una sinopsis contenida en la otra
A_REVISAR = 40               # cuantos candidatos del ranking se revisan

autor_por_slug = dict(zip(catalogo_limpio["slug"], catalogo_limpio["authors"].fillna("")))
valorados = preferencias.assign(
    authors=preferencias["slug"].map(autor_por_slug).fillna(""))


def conjunto_palabras(texto):
    return set(str(texto).lower().split())


def contencion_pal(a, b):
    """Que proporcion de las palabras de b esta dentro de a."""
    pa, pb = conjunto_palabras(a), conjunto_palabras(b)
    return len(pa & pb) / len(pb) if pb else 0.0


ranking = np.argsort(-prob_catalogo)
descartados, admitidos = [], []
for i in ranking[:A_REVISAR]:
    titulo = str(candidatos["title"].iloc[i])
    sinopsis = str(candidatos["sinopsis"].iloc[i])
    autor = str(candidatos["authors"].iloc[i]).strip().lower()
    hallazgo = None
    for _, v in valorados.iterrows():
        jw = JaroWinkler.normalized_similarity(titulo.lower(), str(v["title"]).lower())
        fr = fuzz.ratio(sinopsis.lower(), str(v["synopsis"]).lower()) / 100
        cont = max(contencion_pal(sinopsis, v["synopsis"]),
                   contencion_pal(v["synopsis"], sinopsis))
        mismo_autor = autor != "" and autor == str(v["authors"]).strip().lower()
        if ((mismo_autor and jw >= UMBRAL_TITULO_DUP)
                or fr >= UMBRAL_SINOPSIS_DUP or cont >= UMBRAL_CONTENCION_DUP):
            hallazgo = {"titulo": titulo, "prob": round(prob_catalogo[i], 3),
                        "parecido_a": v["title"], "interes": v["interes"],
                        "mismo_autor": mismo_autor, "titulo_jw": round(jw, 2),
                        "sinopsis": round(fr, 2), "contencion": round(cont, 2)}
            break
    (descartados if hallazgo else admitidos).append(hallazgo or i)

print(f"revisados los {A_REVISAR} mejores del ranking")
print(f"descartados por ser otra version de un libro ya valorado: {len(descartados)}")
if descartados:
    print(pd.DataFrame(descartados).to_string(index=False, max_colwidth=26))

recomendados = candidatos.iloc[admitidos[:5]].copy()
recomendados["probabilidad"] = prob_catalogo[admitidos[:5]].round(3)
print("")
print("los 5 recomendados:")
print(recomendados[["probabilidad", "title", "authors"]].to_string(index=False, max_colwidth=44))

revisados los 40 mejores del ranking
descartados por ser otra version de un libro ya valorado: 1
          titulo  prob          parecido_a  interes  mismo_autor  titulo_jw  sinopsis  contencion
Mi cocina casera 0.647 Mis mejores recetas        1         True       0.62      0.81        0.82

los 5 recomendados:
 probabilidad                                      title                                      authors
        0.748               Guerras, soldados y máquinas ["F. Xavier Hernàndez Cardona", "Xavier R...
        0.712              Ciencia: lo que hay que saber ["José Manuel Sánchez Ron", "Miguel Artola"]
        0.704                           Geno y su cocina                    ["Genoveva Ruiz Zabalza"]
        0.701 Geohistoria. Otra historia de la humanidad                      ["Christian Grataloup"]
        0.685                      El secreto de la vida                            ["Howard Markel"]


In [85]:
# --- 8.2b Por que la regla no puede ser solo el titulo ---
# Esta celda no decide nada: muestra que habria descartado la primera version del
# control, que usaba el umbral de titulo del paso 1 por si solo. Esta aca porque
# la justificacion de la regla de arriba es precisamente esta lista, y una
# afirmacion que ninguna celda imprime no es evidencia para la defensa oral.
solo_titulo = []
for i in ranking[:A_REVISAR]:
    titulo = str(candidatos["title"].iloc[i])
    for _, v in valorados.iterrows():
        jw = JaroWinkler.normalized_similarity(titulo.lower(), str(v["title"]).lower())
        if jw >= UMBRAL_TITULO_DUP:
            sinopsis = str(candidatos["sinopsis"].iloc[i])
            solo_titulo.append({
                "titulo": titulo, "parecido_a": v["title"], "titulo_jw": round(jw, 2),
                "sinopsis": round(fuzz.ratio(sinopsis.lower(),
                                             str(v["synopsis"]).lower()) / 100, 2),
                "mismo_autor": str(candidatos["authors"].iloc[i]).strip().lower()
                               == str(v["authors"]).strip().lower()})
            break

print(f"la regla de solo titulo (Jaro-Winkler >= {UMBRAL_TITULO_DUP}) descartaria "
      f"{len(solo_titulo)} libros:")
print(pd.DataFrame(solo_titulo).to_string(index=False, max_colwidth=30))
print("")
print("Jaro-Winkler pesa mucho el prefijo compartido, asi que titulos que arrancan")
print("igual dan 0.86-0.90 aunque las sinopsis no se parezcan en nada. En el paso 1")
print("ese umbral era una COMPUERTA previa a comparar sinopsis, no una regla sola.")

la regla de solo titulo (Jaro-Winkler >= 0.85) descartaria 4 libros:
                titulo              parecido_a  titulo_jw  sinopsis  mismo_autor
 El secreto de la vida              El secreto       0.90      0.42        False
              Cuántica                Cuántica       1.00      0.42        False
   Cuentos putrefactos       Cuentos dispersos       0.86      0.33        False
El arte en la historia El arte de la intrusión       0.90      0.41        False

Jaro-Winkler pesa mucho el prefijo compartido, asi que titulos que arrancan
igual dan 0.86-0.90 aunque las sinopsis no se parezcan en nada. En el paso 1
ese umbral era una COMPUERTA previa a comparar sinopsis, no una regla sola.


### El control de duplicados, y por qué no es un trámite

Excluir por `slug` no alcanza. El catálogo trae ediciones, traducciones y reempaquetados del
mismo libro con slugs distintos — el paso 1 ya había encontrado **33 grupos de sinopsis
repetida**, y de esos 14 eran volúmenes legítimos distintos. Acá el riesgo es el inverso:
recomendarle al lector un libro que ya valoró, con otra tapa.

**Primero lo intenté con el umbral de título del paso 1 y salió mal.** Jaro-Winkler ≥ 0,85 sobre
los títulos daba **cuatro descartes y los cuatro eran falsos**: *Cuentos putrefactos* contra
*Cuentos dispersos* (0,86), *El arte en la historia* contra *El arte de la intrusión* (0,90), *El
secreto de la vida* contra *El secreto* (0,90) y el *Cuántica* de la tabla. Y peor todavía: **no
detectaba el único duplicado real**, porque *Mi cocina casera* se parece a *Mis mejores recetas*
apenas 0,62 en el título. Jaro-Winkler pesa mucho el prefijo compartido, y en el paso 1 ese
umbral era una **compuerta previa** a comparar sinopsis, no una regla por sí sola.

**El dato que decide es el autor.** La regla corregida descarta un libro si **coincide el autor y
el título se parece**, o si las **sinopsis** se parecen (`fuzz.ratio` ≥ 0,55 o contención ≥ 0,90,
los umbrales del paso 1 y del 2.6). Con eso, los dos casos límite se resuelven en direcciones
opuestas, y cada uno justifica una mitad de la regla:

| caso | título | autor | sinopsis | decisión |
|---|---|---|---|---|
| *Mi cocina casera* vs *Mis mejores recetas* | 0,62 | **el mismo** | 0,81 / contención 0,82 | **descartado** |
| *Cuántica* (Philip Ball) vs *Cuántica* (Jim Al-Khalili) | **1,00** | distinto | 0,42 | **conservado** |

El primero es un reempaquetado: título distinto, autor igual, casi la misma sinopsis. **Mirando
solo el título se escapaba.** El segundo son dos libros de divulgación sobre física cuántica con
el mismo título y autores distintos: **mirando solo el título se descartaba un libro válido.**
Ninguno de los dos criterios funciona solo.

In [86]:
# --- 8.2 Resumen extractivo de dos oraciones ---
# El metodo de la unidad: partir en oraciones, representarlas con TF-IDF, medir la
# similitud del coseno entre ellas y quedarse con las dos mas CENTRALES, o sea las
# que mas se parecen al resto. La idea de fondo es que una oracion que repite el
# vocabulario de todo el texto lo esta resumiendo.
#
# Extractivo significa que no se escribe nada nuevo: se eligen oraciones del
# original. Por eso se devuelven en su ORDEN ORIGINAL y no por centralidad, que
# es lo unico que las hace leerse como un texto y no como dos frases sueltas.
ORACIONES_RESUMEN = 2
MINIMO_ORACION = 30   # caracteres; por debajo de esto no es una oracion sino un resto


def partir_en_oraciones(texto):
    """Corta por punto, signo de pregunta o exclamacion seguidos de espacio."""
    piezas = re.split(r"(?<=[.!?])\s+", texto.strip())
    return [p.strip() for p in piezas if len(p.strip()) > MINIMO_ORACION]


def resumir(texto, cuantas=ORACIONES_RESUMEN):
    """Las 'cuantas' oraciones mas centrales, en el orden en que aparecen.

    Si el texto tiene esa cantidad o menos, se devuelve completo: no hay nada que
    resumir y forzarlo solo perderia informacion.
    """
    oraciones = partir_en_oraciones(texto)
    if len(oraciones) <= cuantas:
        return oraciones, oraciones, None
    matriz = TfidfVectorizer().fit_transform(oraciones)
    similitudes = cosine_similarity(matriz)
    np.fill_diagonal(similitudes, 0)     # una oracion no se resume a si misma
    centralidad = similitudes.sum(axis=1)
    elegidas = sorted(np.argsort(-centralidad)[:cuantas])
    return [oraciones[i] for i in elegidas], oraciones, centralidad

In [87]:
# --- 8.3 Las cinco recomendaciones, con su resumen ---
# Se imprime el titulo, el resumen de dos oraciones y la cantidad de oraciones del
# original, para que se pueda juzgar cuanto se perdio al resumir.
for n, (_, libro) in enumerate(recomendados.iterrows(), start=1):
    resumen, todas, _ = resumir(libro["sinopsis"])
    print("=" * 78)
    print(f"{n}. {libro['title']}   (probabilidad {libro['probabilidad']})")
    print(f"   autor: {libro['authors']}")
    print(f"   sinopsis original: {len(str(libro['sinopsis']))} caracteres, "
          f"{len(todas)} oraciones")
    if len(todas) <= ORACIONES_RESUMEN:
        print("   sin resumen: el texto ya tiene dos oraciones o menos")
    for oracion in resumen:
        print(f"   > {oracion}")
    print("")

1. Guerras, soldados y máquinas   (probabilidad 0.7480000257492065)
   autor: ["F. Xavier Hernàndez Cardona", "Xavier Rubio Campillo"]
   sinopsis original: 1599 caracteres, 7 oraciones
   > Manual de historia universal de la guerra, la obra Guerras, soldados y máquinas está organizada a partir de criterios cronológicos, recoge las aportaciones de diversas corrientes historiográficas contemporáneas y combina niveles de análisis general con focalizaciones de microhistoria.
   > La obra pretende dar una visión de conjunto a partir de tres ejes principales que se relacionan entre sí: la tecnología y la técnica (armas y tácticas de combate); la cultura militar (organización y doctrina militar) y la determinación de los individuos (decisiones de los mandos y capacidades de los soldados).

2. Ciencia: lo que hay que saber   (probabilidad 0.7120000123977661)
   autor: ["José Manuel Sánchez Ron", "Miguel Artola"]
   sinopsis original: 444 caracteres, 2 oraciones
   sin resumen: el texto ya tie

In [88]:
# --- 8.4 Donde el resumen funciona y donde pierde informacion ---
# Se muestra la centralidad de cada oracion de dos libros: el resumen elige las
# dos de arriba, asi que ver el puntaje de las que quedaron afuera es la unica
# forma de mostrar QUE se perdio y por que.
for slug_o_titulo in ["Guerras, soldados y maquinas", "Geno y su cocina"]:
    fila = recomendados[recomendados["title"].str.startswith(slug_o_titulo[:14])]
    if fila.empty:
        print(f"(no esta entre los recomendados de esta corrida: {slug_o_titulo})")
        continue
    texto = fila.iloc[0]["sinopsis"]
    oraciones = partir_en_oraciones(texto)
    matriz = TfidfVectorizer().fit_transform(oraciones)
    sim = cosine_similarity(matriz)
    np.fill_diagonal(sim, 0)
    centralidad = sim.sum(axis=1)
    elegidas = set(np.argsort(-centralidad)[:ORACIONES_RESUMEN])
    print("=" * 78)
    print(f"{fila.iloc[0]['title']}  ({len(oraciones)} oraciones)")
    for n, oracion in enumerate(oraciones):
        marca = "ELEGIDA" if n in elegidas else "       "
        print(f"  {marca} [{centralidad[n]:.2f}] {oracion[:120]}")
    print("")

Guerras, soldados y máquinas  (7 oraciones)
  ELEGIDA [1.36] Manual de historia universal de la guerra, la obra Guerras, soldados y máquinas está organizada a partir de criterios cr
          [1.06] En tanto que manual que aporta información y reflexión sobre el fenómeno de la guerra a lo largo del tiempo, Guerras, so
  ELEGIDA [1.46] La obra pretende dar una visión de conjunto a partir de tres ejes principales que se relacionan entre sí: la tecnología 
          [1.11] Se trata por tanto de una propuesta holística que considera globalmente el fenómeno de la guerra.
          [1.05] Pretende aportar una visión muy plural que va desde las emociones del soldado a la articulación entre medios técnicos, c
          [0.93] El trabajo se vertebra a partir de un criterio cronológico, los diferentes periodos históricos se van tratando de manera
          [1.27] Los análisis globales se contrastan a su vez con aproximaciones de microhistoria que narran el punto de vista subjetivo 

Geno y su co

### Dónde el resumen funciona y dónde pierde información

**Funciona: *Guerras, soldados y máquinas*** (7 oraciones). Elige la primera, que dice qué es el
libro —un manual de historia universal de la guerra, organizado cronológicamente—, y la tercera,
que dice cómo está organizado, con sus tres ejes. Qué es y cómo está hecho: es lo que uno quiere
de un resumen de dos oraciones.

**Pierde información: *Geno y su cocina*** (4 oraciones). Elige la primera y la última, que son
las dos más genéricas: «es un clásico de la gastronomía navarra» y «su historia y su calidad lo
convierten en un clásico imprescindible». Y **deja afuera la única oración con datos concretos**:
1.298 recetas clasificadas, consejos para servir la mesa y 12 menús recomendados. Esa oración
tiene la centralidad más baja de las cuatro, **0,14**, contra 0,38 y 0,25 de las elegidas.

Ahí está la debilidad estructural del método, y conviene decirla con el nombre correcto: **la
centralidad premia la redundancia.** Una oración es central cuando repite el vocabulario del
resto, y la oración más informativa es a menudo la que menos se repite, justamente porque aporta
algo que no está en otra parte. Las dos frases sobre «clásico» comparten vocabulario entre sí y
se sostienen mutuamente; la de las recetas no comparte con nadie y queda última.

**Un tercer caso, de otro tipo:** *Ciencia: lo que hay que saber* tiene **dos oraciones** en
total. No hay nada que resumir, y forzar un resumen solo podría perder información, así que la
función devuelve el texto completo. Con sinopsis de 222 a 4581 caracteres, esta situación no es
una rareza.

In [89]:
# --- 8.3 El perfil del lector con spaCy ---
# Frases sustantivas (noun_chunks) y entidades de las sinopsis de entrenamiento,
# separadas por valoracion. Se agrupa por el LEMA DEL NUCLEO de cada frase, como
# pide el enunciado: 'la novela', 'las novelas' y 'una novela' son lo mismo.
#
# Se cuenta EN CUANTOS LIBROS aparece cada lema y no cuantas veces en total: un
# solo libro que repite nueve veces 'geografia' inventaria un interes que no esta.
def perfil_del_grupo(textos):
    """Lemas de nucleo y entidades, contados por libro en que aparecen."""
    lemas, entidades = Counter(), Counter()
    for doc in nlp.pipe(textos, batch_size=16):
        vistos = set()
        for frase in doc.noun_chunks:
            nucleo = frase.root.lemma_.lower()
            if len(nucleo) > 2 and not frase.root.is_stop:
                vistos.add(nucleo)
        lemas.update(vistos)
        entidades.update({(e.text.strip(), e.label_) for e in doc.ents})
    return lemas, entidades


valor_ent = pd.to_numeric(libros_ent["interes"])
textos_pos = libros_ent[valor_ent == 1]["synopsis"].tolist()
textos_neg = libros_ent[valor_ent == 0]["synopsis"].tolist()
lemas_pos, ents_pos = perfil_del_grupo(textos_pos)
lemas_neg, ents_neg = perfil_del_grupo(textos_neg)

for nombre, lemas, ents, total in [
    ("LE INTERESAN", lemas_pos, ents_pos, len(textos_pos)),
    ("NO LE INTERESAN", lemas_neg, ents_neg, len(textos_neg)),
]:
    print(f"=== {nombre} ({total} libros) ===")
    print("  frases sustantivas: " + ", ".join(
        f"{lema} ({c}/{total})" for lema, c in lemas.most_common(10)))
    print("  entidades: " + ", ".join(
        f"{txt} [{lab}] ({c})" for (txt, lab), c in ents.most_common(10)))
    print("")

=== LE INTERESAN (23 libros) ===
  frases sustantivas: historia (12/23), libro (12/23), obra (7/23), año (7/23), vida (4/23), arte (4/23), caso (4/23), tema (4/23), siglo (4/23), país (4/23)
  entidades: Hitler [PER] (2), Voltaire [PER] (2), El resultado [MISC] (2), España [LOC] (2), Alemania [LOC] (2), Polonia [LOC] (2), Francia [LOC] (2), De un modo u otro [MISC] (1), Atraco a mano armada [MISC] (1), Daria Galateria [PER] (1)

=== NO LE INTERESAN (137 libros) ===
  frases sustantivas: vida (49/137), libro (35/137), mundo (34/137), historia (30/137), año (26/137), obra (26/137), autor (20/137), hombre (20/137), novela (18/137), relato (18/137)
  entidades: Sin embargo [MISC] (6), España [LOC] (6), Este libro [MISC] (5), Europa [LOC] (5), Francia [LOC] (4), Barcelona [LOC] (2), Mark Twain [PER] (2), México [LOC] (2), Ahora [MISC] (2), Perón [PER] (2)



In [90]:
# --- 8.4 Lo que de verdad distingue a un grupo del otro ---
# Las dos listas de arriba comparten 'libro', 'historia', 'obra' y 'anio': son el
# vocabulario de cualquier sinopsis, no el gusto del lector. Y no son comparables
# tal cual, porque un grupo tiene 23 libros y el otro 137.
#
# Esto no lo pide el enunciado: resta las PROPORCIONES para que la lista describa
# al lector y no al genero 'sinopsis de contratapa'.
PRESENCIA_MINIMA = 4   # en menos de 4 libros de 23, cualquier diferencia es ruido

n_pos, n_neg = len(textos_pos), len(textos_neg)
comparacion = []
for lema, cuenta in lemas_pos.items():
    if cuenta >= PRESENCIA_MINIMA:
        tasa_pos = cuenta / n_pos
        tasa_neg = lemas_neg.get(lema, 0) / n_neg
        comparacion.append({"lema": lema, "en_pos": f"{cuenta}/{n_pos}",
                            "en_neg": f"{lemas_neg.get(lema, 0)}/{n_neg}",
                            "tasa_pos": round(tasa_pos, 3),
                            "tasa_neg": round(tasa_neg, 3),
                            "diferencia": round(tasa_pos - tasa_neg, 3)})
comparacion = pd.DataFrame(comparacion).sort_values("diferencia", ascending=False)
print("lemas sobrerrepresentados en los libros que le interesan:")
print(comparacion.head(10).to_string(index=False))
print("")
print("y los que estan sobrerrepresentados en los que NO le interesan:")
print(comparacion.tail(3).to_string(index=False))

lemas sobrerrepresentados en los libros que le interesan:
      lema en_pos en_neg  tasa_pos  tasa_neg  diferencia
  historia  12/23 30/137     0.522     0.219       0.303
     libro  12/23 35/137     0.522     0.255       0.266
      caso   4/23  5/137     0.174     0.036       0.137
      país   4/23  5/137     0.174     0.036       0.137
   ciencia   4/23  7/137     0.174     0.051       0.123
    guerra   4/23  7/137     0.174     0.051       0.123
      arte   4/23  7/137     0.174     0.051       0.123
       año   7/23 26/137     0.304     0.190       0.115
      obra   7/23 26/137     0.304     0.190       0.115
literatura   4/23  9/137     0.174     0.066       0.108

y los que estan sobrerrepresentados en los que NO le interesan:
  lema en_pos en_neg  tasa_pos  tasa_neg  diferencia
 siglo   4/23 17/137     0.174     0.124       0.050
relato   4/23 18/137     0.174     0.131       0.043
  vida   4/23 49/137     0.174     0.358      -0.184


### El perfil del lector, y por qué la lista cruda no lo describe

Las dos listas que pide el enunciado salen arriba. El problema es que **comparten seis de sus
diez lemas**: *libro*, *historia*, *obra*, *año*, *hombre* y *autor* están en los dos grupos. Eso
no es el gusto del lector, es el vocabulario de cualquier contratapa.

Hay dos razones, y las dos se arreglan midiendo distinto:

1. **Contar apariciones totales deja que un solo libro invente un interés.** Un libro que repite
   nueve veces «geografía» pone ese lema en el top 10 del grupo entero. Por eso se cuenta **en
   cuántos libros** aparece cada lema, no cuántas veces en total.
2. **Los grupos tienen 23 y 137 libros**, así que las cuentas no son comparables tal cual. Hay que
   comparar **proporciones**.

Corregido así, el perfil empieza a decir algo: **historia** aparece en el 52 % de los libros que
le interesaron contra el 22 % de los que no, y siguen **país**, **caso**, **guerra**, **ciencia**,
**arte** y **literatura**, todos alrededor del 17 % contra el 4-7 %. Del otro lado, **vida** está
en el 36 % de los que no le interesaron contra el 17 % de los que sí.

Eso coincide con lo que veníamos inferiendo de los títulos —divulgación científica, historia,
arte, ensayo— pero ahora **medido desde el texto de las sinopsis**, que es lo que el modelo lee.
Con una advertencia honesta: la mayoría de esos lemas aparecen en **4 de 23 libros**, así que la
diferencia se apoya en cuatro libros contra cinco. Es el mismo techo de datos de todo el TP.

**Las entidades vienen ruidosas**, y vale señalarlo: spaCy etiqueta «Sin embargo», «Este libro» y
«El resultado» como entidades, y «Marina» como organización. Es la misma debilidad que medimos en
el **2.4**, donde el NER daba precisión 0,07 contra un piso de azar de 0,042. No contradice esa
conclusión: la confirma desde otro lado.

### El comentario del lector

**Esta parte la completa Julián**, y es parte del entregable: el enunciado pide que la persona que
dio las valoraciones revise las recomendaciones y escriba cuáles le interesan y cuáles no, y que
diga si la lista de frases del perfil describe lo que le gusta.

Dos cosas conviene registrar acá, en sus palabras:

- **Sobre las cinco recomendaciones:** cuáles le interesan y cuáles no, y si hay alguna que le
  llame la atención por un motivo que el modelo no podría haber sabido.
- **Sobre el perfil:** si *historia, guerra, ciencia, arte, literatura* describe su criterio, o si
  falta algo que él tenía claro al valorar y que no aparece en la lista.

> *(a completar)*

Esta revisión no es un adorno: es la **única evaluación que tienen las recomendaciones del
catálogo**, porque esos 8184 libros no tienen etiqueta. Todo lo que medimos en los pasos 6 y 7 fue
sobre los 40 de prueba. Acá el juicio humano es la medición.

## Cierre del TP

| paso | qué dejó |
|---|---|
| 1 | 8591 de 8747 libros; 33 grupos de sinopsis repetida, 14 de ellos volúmenes legítimos |
| 2 | `sospechosos.csv` (407 libros con motivo) y `catalogo_limpio.csv` (8184) |
| 3 | 200 libros valorados: 29 positivos, 171 negativos, sin abstenciones |
| 4 | 160 de entrenamiento (23 positivos) y 40 de prueba (6) |
| 5 | dos clasificadores: matrices 160×6845 y 160×512 |
| 6 | al umbral 0,5 los dos fracasan; como ordenadores valen el doble del azar |
| 7 | precision@5 validada: embeddings 0,252, TF-IDF 0,196, perfil 0,160 (azar 0,145) |
| 8 | cinco recomendaciones del catálogo, con resumen, y el perfil del lector |

**La conclusión honesta del TP, en una línea:** el recomendador funciona, pero débilmente — mejor
que el azar de forma consistente y medida, y lejos de algo en lo que uno confiaría. Y sabemos por
qué: **29 libros positivos repartidos en cuatro o cinco temas** no alcanzan para aprender un
criterio, y eso no se arregla con una representación mejor. Los dos modelos empatan como
ordenadores justamente porque el límite no está en cómo se lee el texto, sino en cuánto hay para
leer.

**La lección metodológica fue la misma tres veces.** En el 2.5 con las plantillas, en el 6.4 con
el AUC y en el 7.2 con precision@5, una sola partición dijo lo contrario que la medición repetida,
y siempre exagerando. Las tres veces el número bueno apareció al repetir. Si hay algo que este TP
enseñó, es que **un número sin su variabilidad no es un resultado**.